# Loan Default Prediction, Regularization as a Lens

we predict whether an applicant will default on a loan using the home credit default risk dataset. the analysis follows three research questions, the effect of regularisation on each model (RQ1), how feature importance compares across models (RQ2), and fairness across demographic subgroups (RQ3). we compare a logistic regression baseline, an elastic net, a random forest, an XGBoost model, and a custom MLP, all using the same train and val splits.

the notebook downloads its own data and creates folders for intermediate artifacts. expensive steps are checkpointed so re-running is fast.

---

## **0. Setup**

### 0.1 Imports

we import everything up front so the rest of the notebook stays focused on analysis. the libraries cover preprocessing, four model families, and the keras stack we need for the MLP.

In [ ]:
import os
import time
import warnings
import joblib

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)


In [ ]:
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from scipy.stats import uniform, randint

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"numpy {np.__version__}  |  pandas {pd.__version__}  |  tensorflow {tf.__version__}")


### 0.2 Output directories

we create three folders next to the notebook, one for the downloaded dataset, one for intermediate checkpoints, and one for trained models and figures. all paths are relative to the working directory so it works on any machine.

In [ ]:
# === output directories ===
# creates folders next to the notebook for everything we save
# works on any machine, no hardcoded paths

import os

DATA_DIR = os.path.join(os.getcwd(), 'data')
CHECKPOINT_DIR = os.path.join(os.getcwd(), 'checkpoints')
OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')

for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"ready: {d}")


### 0.3 Random seeds

we fix the seed at 42 across numpy, tensorflow, and every sklearn random_state we use, so the results are reproducible.

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"SEED = {SEED}")


### 0.4 Shared helpers

we define the helpers once here and reuse them in every model section. there are two flavours of evaluation, the sklearn one calls predict_proba and the keras one calls predict on the tensor model. the keras evaluator uses a 0.3 threshold instead of 0.5 becuase the MLP rarely outputs probabilities above 0.5 on this imbalanced dataset, so a lower cut-off is closer to how a bank would actually act on the score.

In [ ]:
# counter for auto-naming saved plots

_plot_counter = [0]

def save_plot(name=None):
    _plot_counter[0] += 1
    n = _plot_counter[0]
    fname = f"{n:03d}_{name or 'plot'}.png"
    plt.gcf().savefig(os.path.join(OUTPUT_DIR, fname), dpi=150, bbox_inches='tight')


In [ ]:
# prints roc-auc, pr-auc, classification report for an sklearn model

def evaluate_model(model, X, y, split_name="Val", threshold=0.5):

    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    roc_auc = roc_auc_score(y, y_prob)
    pr_auc  = average_precision_score(y, y_prob)

    print(f"\n{'='*55}")
    print(f"  {split_name} threshold = {threshold}")
    print(f"{'='*55}")
    print(f"  ROC-AUC : {roc_auc:.4f}")
    print(f"  PR-AUC  : {pr_auc:.4f}")
    print(f"\n{classification_report(y, y_pred, target_names=['No Default (0)', 'Default (1)'])}")

    return {"split": split_name, "roc_auc": roc_auc, "pr_auc": pr_auc}


In [ ]:
# 3-panel plot, confusion matrix, roc curve, pr curve, for sklearn models

def plot_evaluation(model, X_val, y_val, X_test, y_test, title_prefix=""):

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{title_prefix} Evaluation Plots", fontsize=14, fontweight='bold')

    # confusion matrix on val
    y_prob_val = model.predict_proba(X_val)[:, 1]
    y_pred_val = (y_prob_val >= 0.5).astype(int)
    cm = confusion_matrix(y_val, y_pred_val)
    ConfusionMatrixDisplay(cm, display_labels=["No Default", "Default"]).plot(
        ax=axes[0], colorbar=False, cmap="Blues"
    )
    axes[0].set_title("Confusion Matrix (Val, threshold=0.5)")

    # roc curve for val and test
    for X, y, label in [(X_val, y_val, "Val"), (X_test, y_test, "Test")]:
        y_prob = model.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y, y_prob)
        auc = roc_auc_score(y, y_prob)
        axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc:.4f})")
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve")
    axes[1].legend()

    # pr curve for val and test
    for X, y, label in [(X_val, y_val, "Val"), (X_test, y_test, "Test")]:
        y_prob = model.predict_proba(X)[:, 1]
        prec, rec, _ = precision_recall_curve(y, y_prob)
        ap = average_precision_score(y, y_prob)
        axes[2].plot(rec, prec, label=f"{label} (AP={ap:.4f})")
    baseline_rate = y_val.mean()
    axes[2].axhline(baseline_rate, color='k', linestyle='--', linewidth=0.8,
                    label=f"Random baseline ({baseline_rate:.3f})")
    axes[2].set_xlabel("Recall")
    axes[2].set_ylabel("Precision")
    axes[2].set_title("Precision-Recall Curve")
    axes[2].legend()

    plt.tight_layout()


In [ ]:
# same as evaluate_model but for keras, uses .predict not .predict_proba
# default threshold 0.3 because MLP outputs are low on imbalanced data

def evaluate_keras_model(model, X, y, split_name="Val", threshold=0.3):

    y_prob = model.predict(X, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    roc_auc = roc_auc_score(y, y_prob)
    pr_auc  = average_precision_score(y, y_prob)

    print(f"\n{'='*55}")
    print(f"  {split_name} threshold = {threshold}")
    print(f"{'='*55}")
    print(f"  ROC-AUC : {roc_auc:.4f}")
    print(f"  PR-AUC  : {pr_auc:.4f}")
    print(f"\n{classification_report(y, y_pred, target_names=['No Default (0)', 'Default (1)'])}")

    return {"split": split_name, "roc_auc": roc_auc, "pr_auc": pr_auc}


In [ ]:
# plots loss and auc training curves from keras history

def plot_training_curves(history, title="Training curves"):

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].plot(history.history['loss'],     label='Train loss')
    axes[0].plot(history.history['val_loss'], label='Val loss')
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Binary cross-entropy")
    axes[0].set_title("Loss")
    axes[0].legend()

    auc_key = [k for k in history.history if 'auc' in k and 'val' not in k][0]
    val_auc_key = f'val_{auc_key}'

    axes[1].plot(history.history[auc_key],     label='Train AUC')
    axes[1].plot(history.history[val_auc_key], label='Val AUC')
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("ROC-AUC")
    axes[1].set_title("AUC")
    axes[1].legend()

    plt.tight_layout()


In [ ]:
# 3-panel evaluation plot for keras models

def plot_evaluation_keras(model, X_val, y_val, X_test, y_test, title_prefix="", threshold=0.3):

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{title_prefix} Evaluation Plots", fontsize=14, fontweight='bold')

    y_prob_val = model.predict(X_val, verbose=0).ravel()
    y_pred_val = (y_prob_val >= threshold).astype(int)
    cm = confusion_matrix(y_val, y_pred_val)
    ConfusionMatrixDisplay(cm, display_labels=["No Default", "Default"]).plot(
        ax=axes[0], colorbar=False, cmap="Blues"
    )
    axes[0].set_title(f"Confusion Matrix (Val, threshold={threshold})")

    for X, y, label in [(X_val, y_val, "Val"), (X_test, y_test, "Test")]:
        y_prob = model.predict(X, verbose=0).ravel()
        fpr, tpr, _ = roc_curve(y, y_prob)
        auc = roc_auc_score(y, y_prob)
        axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc:.4f})")
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve")
    axes[1].legend()

    for X, y, label in [(X_val, y_val, "Val"), (X_test, y_test, "Test")]:
        y_prob = model.predict(X, verbose=0).ravel()
        prec, rec, _ = precision_recall_curve(y, y_prob)
        ap = average_precision_score(y, y_prob)
        axes[2].plot(rec, prec, label=f"{label} (AP={ap:.4f})")
    baseline_rate = y_val.mean()
    axes[2].axhline(baseline_rate, color='k', linestyle='--', linewidth=0.8,
                    label=f"Random baseline ({baseline_rate:.3f})")
    axes[2].set_xlabel("Recall")
    axes[2].set_ylabel("Precision")
    axes[2].set_title("Precision-Recall Curve")
    axes[2].legend()

    plt.tight_layout()


In [ ]:
print("helpers ready")


## **1. Data Loading**

we host the dataset on google drive and pull it down with gdown. no kaggle login or API key needed, the professor just runs the cell.

In [ ]:
# === data loading ===
# dataset: Home Credit Default Risk
# hosted on Google Drive (public link, no login needed)
# original source: https://www.kaggle.com/datasets/diogoamaro0/loan-deafult-prediction

import os
import gdown
import pandas as pd

DATA_DIR = os.path.join(os.getcwd(), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

FILE_ID = '1gQqkj4FCSZP1QEHCn5GwMReDX7uLh2EZ'
DATA_PATH = os.path.join(DATA_DIR, 'application_train.csv')

if not os.path.exists(DATA_PATH):
    url = f'https://drive.google.com/uc?id={FILE_ID}'
    gdown.download(url, DATA_PATH, quiet=False)
    print("download complete")
else:
    print("data already downloaded")

df = pd.read_csv(DATA_PATH)
print(f"loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())


## **2. Exploratory Data Analysis**

we describe and quantify only in this section. no imputation, no feature engineering. those decisions live in section 3 with the preprocessing pipeline.

In [ ]:
# checking the shape

print("Shape:", df.shape)


We work only with `application_train.csv`. The Kaggle competition's `application_test.csv` has no `TARGET` column since the labels were never released after the competition closed, so we cannot score any model on it. Our train/validation/test split is done inside `application_train.csv` instead.

In [ ]:
# checking the df head

df.head(10)

In [ ]:
# checking basic info

df.info()

In [ ]:
# we reprint so it is more readable

df.dtypes.value_counts()

### Target Variable

we check how often default happens in the training data and how imbalanced the classes are.

The target variable `TARGET` is binary and flags whether an applicant had
payment difficulties.

- `1` = the client had late payment of more than X days on at least one of
  the first Y installments of the loan (the "bad" outcome we want to predict)
- `0` = all other cases, including loans paid on time and minor delays

In [ ]:
# cointing how many 0s and 1s we have in the target column
counts = df["TARGET"].value_counts().sort_index()

# we take the result and examine that as a percentage
pct = (counts / len(df)) * 100

# put them side by side in a dataframe
target_summary = pd.DataFrame({"count": counts, "pct": pct.round(2)})

# we determine the default ratio
ratio = counts[0] / counts[1]


print(f"\n1 defaulter per {ratio:.1f} non-defaulters")
target_summary

In [ ]:
# overall default rate, used as a reference line in bivariate plots
default_rate = df["TARGET"].mean()

In [ ]:
# we plotting the results above

plt.figure(figsize=(5, 4))
plt.bar(["No default (0)", "Default (1)"], counts.values, color=["steelblue", "tomato"])

# add count and percentage labels on top of each bar
for i, (cnt, p) in enumerate(zip(counts.values, pct.values)):
    plt.text(i, cnt, f"{cnt:,}\n({p:.1f}%)", ha="center", va="bottom")

# extend the y-axis because it was hitting the label
plt.ylim(0, counts.max() * 1.15)

plt.ylabel("Count")
plt.title("Target distribution")
plt.tight_layout()
save_plot(); plt.show()

As we can see the dataset is heavily imbalanced, this is not strange for default data. Needs to get addressed.

### Splitting columns by type

we separate numeric and categorical columns so we can apply the right plots and summaries to each.

In [ ]:
# we split column names into two lists by data type

numeric_cols = df.select_dtypes(include="number").columns.tolist()
object_cols = df.select_dtypes(include="object").columns.tolist()

# quick sanity check on the counts

print(f"numeric columns: {len(numeric_cols)}")
print(f"object columns: {len(object_cols)}")
print(f"total: {len(numeric_cols) + len(object_cols)} (should be {df.shape[1]})")

In [ ]:
# id and target are not features, exclude them first

id_and_target = ["SK_ID_CURR", "TARGET"]

# a binary column has exactly 2 unique values but is stored as int so we take them out

feature_cols = [c for c in numeric_cols if c not in id_and_target]
binary_cols = [c for c in feature_cols if df[c].nunique() == 2]

# whatever is left after removing binaries is continuous/discrete

continuous_cols = [c for c in feature_cols if c not in binary_cols]

# sanity check

print(f"binary flags: {len(binary_cols)}")
print(f"continuous / discrete: {len(continuous_cols)}")
print(f"id + target: {len(id_and_target)}")
print(f"total: {len(binary_cols) + len(continuous_cols) + len(id_and_target)} (should be {len(numeric_cols)})")

In [ ]:
# we count unique values for each column in the continuous group
nunique_cont = df[continuous_cols].nunique(dropna=True)

# discrete counts = few distinct integer levels (like CNT_CHILDREN, HOUR_APPR_PROCESS_START)
discrete_cols = [c for c in continuous_cols if nunique_cont[c] <= 20]

# true continuous = amounts, ratios, days, everything with a wide numeric range

true_continuous_cols = [c for c in continuous_cols if nunique_cont[c] > 20]

# sanity check

print(f"true continuous (>20 unique): {len(true_continuous_cols)}")
print(f"discrete count  (3-20)      : {len(discrete_cols)}")
print(f"total                       : {len(true_continuous_cols) + len(discrete_cols)} "
      f"(should be {len(continuous_cols)})")

In [ ]:
# we count how many unique values each categorical column has
cat_cardinality = (
    df[object_cols].nunique().sort_values(ascending=False).to_frame("n_unique")
)
display(cat_cardinality)

`ORGANIZATION_TYPE` stands out with 58 categories, far more than any other column. One-hot encoding it as-is would add 57 sparse columns, so we will group rare categories during preprocessing. `OCCUPATION_TYPE` (18) and `NAME_INCOME_TYPE` (8) sit in a middle range and need similar attention. The rest have 7 or fewer levels and can be one-hot encoded directly.

### Univariate Numeric

we look at each numeric column on its own to spot shape, scale and weird values.

### 4.1 All continuous numeric columns

histograms and boxplots across every continuous numeric column.

In [ ]:
# quick info

summary = df[true_continuous_cols].describe().T

summary

In [ ]:
# histograms for every continuous numeric column.
# heavily right-skewed columns (skew > 3) that are strictly positive get a log-scale x-axis, which makes the shape easier to see.

cols = sorted(true_continuous_cols)

# figure out how many rows of 5 plots we need
ncols = 5
nrows = (len(cols) + ncols - 1) // ncols

# build the grid and flatten so we can loop with a single index
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 2.2))
axes = axes.flatten()

# one histogram per column
for i, col in enumerate(cols):
    s = df[col].dropna()
    ax = axes[i]

    # plot the histogram (same call regardless of scale)
    ax.hist(s, bins=50, color="steelblue", edgecolor="white")

    # switch to log-x only if strictly positive and heavily right-skewed
    if s.min() > 0 and s.skew() > 3:
        ax.set_xscale("log")

    ax.set_title(col, fontsize=7)
    ax.tick_params(axis="both", labelsize=6)

# hide any leftover empty subplots in the last row
for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f"Histograms of all {len(cols)} continuous numeric columns "
    f"(log-x where skew > 3 and min > 0)",
    y=1.002, fontsize=11,
)
plt.tight_layout()
save_plot(); plt.show()

A few things stand out from the distributions.

- `DAYS_EMPLOYED` has a weird spike near 365,000. That is the sentinel 365243 for unemployed people, not a real value. We will turn it into NaN and add a flag column.

- `AMT_INCOME_TOTAL` is destroyed by outliers, the histogram is basically one block going up to 10⁸. We will winsorize at the 99th percentile before scaling.

- All the `DAYS_*` columns are negative because they count backwards from the application date. Not a bug, but we will flip them or convert to years.

- `EXT_SOURCE_1/2/3` look clean and well spread in [0, 1]. These are usually the strongest predictors of default.

- The `_AVG / _MEDI / _MODE` triplets are basically duplicates of each other. We will revisit this in the missing-values audit in section 9, where we decide what to do with the whole building-features block.

- `YEARS_BEGINEXPLUATATION_*` is saturated at 1, almost no variation. Probably useless, candidate for dropping.

- `OWN_CAR_AGE` has a small spike at 65 that might be another sentinel. Check with `value_counts()` before scaling.

In [ ]:
# boxplots for every continuous numeric column.
# same log-scale rule as the histograms: skew > 3 and min > 0 -> log-y axis.

cols = sorted(true_continuous_cols)

# figure out how many rows of 5 plots we need
ncols = 5
nrows = (len(cols) + ncols - 1) // ncols

# build the grid and flatten so we can loop with a single index
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 2.2))
axes = axes.flatten()

# one boxplot per column
for i, col in enumerate(cols):
    s = df[col].dropna()
    ax = axes[i]

    # draw the vertical boxplot
    ax.boxplot(s)

    # switch to log-y only if strictly positive and heavily right-skewed
    if s.min() > 0 and s.skew() > 3:
        ax.set_yscale("log")

    ax.set_title(col, fontsize=7)
    ax.tick_params(axis="both", labelsize=6)

# hide any leftover empty subplots in the last row
for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f"Boxplots of all {len(cols)} continuous numeric columns "
    f"(log-y where skew > 3 and min > 0)",
    y=1.002, fontsize=11,
)
plt.tight_layout()
save_plot(); plt.show()

The boxplots confirm what the histograms already showed, just sharper.

- `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_GOODS_PRICE` and `AMT_ANNUITY` all have huge upper tails of outliers. Winsorizing at the 99th percentile in Phase 2 should fix this.

- `DAYS_EMPLOYED` is the cleanest case, the box is squashed near 0 and one isolated point sits at 365,000. That single point is the sentinel hiding a NaN.

- `OBS_30_CNT_SOCIAL_CIRCLE` and `OBS_60_CNT_SOCIAL_CIRCLE` show one extreme outlier around 350 while the rest of the data sits near 0. Probably another sentinel or a data error, we will check it with `value_counts()`.

- `OWN_CAR_AGE` confirms the spike around 65 we saw in the histograms.

- The building features (`APARTMENTS_*`, `BASEMENTAREA_*`, `LIVINGAREA_*`, `COMMONAREA_*`, `NONLIVING*_*`) are heavily zero-inflated with dense outlier clouds above. Same multicollinear `_AVG/_MEDI/_MODE` block we want to reduce.

- `EXT_SOURCE_1/2/3` have wide, well-centered boxes with no outliers. They look like the cleanest features in the dataset.

- `HOUR_APPR_PROCESS_START` is tight around midday with a few low-hour outliers, nothing strange.

### 4.2 Full sweep, all discrete numeric columns

Columns with 3..20 unique integer values plotted as bar counts.

In [ ]:
cols = sorted(discrete_cols)
ncols = 5
nrows = (len(cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 2.2))
axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes

for i, col in enumerate(cols):
    counts = df[col].value_counts().sort_index()
    ax = axes[i]
    ax.bar(counts.index.astype(str), counts.values,
           color="darkorange", edgecolor="white")
    ax.set_title(col, fontsize=7)
    ax.tick_params(axis="both", labelsize=6)
    # rotate x labels if there are many bars
    if len(counts) > 6:
        ax.tick_params(axis="x", rotation=45)

for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f"Bar counts -- all {len(cols)} discrete numeric columns (3..20 unique values)",
    y=1.002, fontsize=11,
)
plt.tight_layout()
save_plot(); plt.show()

The discrete columns mostly behave as expected.

- The four `AMT_REQ_CREDIT_BUREAU_*` columns (day, hour, qrt, week) are almost entirely 0, meaning the vast majority of applicants had no credit bureau enquiries in those windows. Very low variance, probably weak predictors on their own but cheap to keep.

- `CNT_CHILDREN` and `CNT_FAM_MEMBERS` are right-skewed with most applicants having 0 to 2 children and 1 to 3 family members. A handful of extreme cases (10+ children) are visible, worth checking if they are real or data entry errors.

- `DEF_30_CNT_SOCIAL_CIRCLE` and `DEF_60_CNT_SOCIAL_CIRCLE` are heavily concentrated at 0, with the same pattern as the `OBS_*` versions. These might still carry signal because "any default in the social circle" could matter even if rare.

- `REGION_RATING_CLIENT` and `REGION_RATING_CLIENT_W_CITY` are clean 1/2/3 ordinals with most applicants in rating 2. We can treat them as ordinal directly without one-hot encoding.



### 4.3 All binary flag columns

All 0/1 columns ranked by % = 1. FLAG_MOBIL sits near 100%; some FLAG_DOCUMENT_* columns sit below 1%.

In [ ]:
pct_ones = (
    df[binary_cols]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, max(6, len(pct_ones) * 0.22)))

# horizontal is a deliberate exception to the vertical house style --
# with 50+ binary columns, vertical labels overlap. same reasoning we
# used in the missingness overview.
y_positions = range(len(pct_ones))
ax.barh(y_positions, pct_ones.values, color="seagreen", edgecolor="white")
ax.set_yticks(list(y_positions))
ax.set_yticklabels(pct_ones.index, fontsize=7)
ax.invert_yaxis()  # highest % at top
ax.set_xlabel("% of rows where flag = 1")
ax.set_title(f"Binary flag columns -- all {len(pct_ones)}, ranked by % = 1")
ax.set_xlim(0, 100)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
save_plot(); plt.show()

The 32 binary flags split into three clear groups.

- Useless flags at 100%, `FLAG_MOBIL` and `FLAG_CONT_MOBILE` are 1 for everyone. Zero variance, they carry no information and we can drop them.

- Near-zero flags, most `FLAG_DOCUMENT_*` columns (4, 7, 9, 10, 11, 12, 13, 14, 15, 17, 19, 20, 21) are 1 for less than 1% of applicants. Very low variance, but they could still carry weak signal in tree models. Candidate for grouping (for example "submitted any rare document") or dropping.

- Useful flags with real variance, `FLAG_EMP_PHONE` (~82%), `FLAG_DOCUMENT_3` (~71%), `FLAG_PHONE` (~28%), `REG_CITY_NOT_WORK_CITY` (~22%), `FLAG_WORK_PHONE` (~20%), `LIVE_CITY_NOT_WORK_CITY` (~18%) all have meaningful spread and likely carry signal. We will keep these as-is.

- The address mismatch flags (`REG_CITY_NOT_WORK_CITY`, `LIVE_CITY_NOT_WORK_CITY`, `REG_REGION_NOT_*`) are interesting because they capture mobility/instability, which is often correlated with credit risk.

### 4.4 Skewness ranking

continuous columns ranked by absolute skew, binary columns excluded.

In [ ]:
# skewness is analytically meaningless on binary columns: a 99/1 split
# produces a huge number that tells you nothing about a tail. drop anything
# with <= 2 unique values before ranking.
nunique = df[numeric_cols].nunique(dropna=True)
non_binary_cols = nunique[nunique > 2].index.tolist()

skew_values = df[non_binary_cols].skew()
skew_table = (
    pd.DataFrame({
        "feature": skew_values.index,
        "skewness": skew_values.values.round(3)
    })
    .assign(abs_skew=lambda x: x["skewness"].abs())
    .sort_values("abs_skew", ascending=False)
    .drop(columns="abs_skew")
    .reset_index(drop=True)
)
print(f"Ranking skewness on {len(non_binary_cols)} non-binary numeric columns "
      f"(excluded {len(numeric_cols) - len(non_binary_cols)} binary columns).")
display(skew_table.head(20))

Top 20 features by absolute skewness, sorted from worst to least bad.

- `AMT_INCOME_TOTAL` is by far the most skewed (391.6), driven by the same extreme outliers we saw in the boxplots. Confirms that winsorization is needed before scaling.

- The `AMT_REQ_CREDIT_BUREAU_*` columns (qrt, day, hour, week, mon) are highly skewed because they are almost all 0, as we saw in the discrete bar charts. Skewness here is just a side effect of the zero-inflation, not a separate problem.

- The `NONLIVINGAPARTMENTS_*` and `NONLIVINGAREA_*` triplets show extreme positive skew (15+), and `YEARS_BEGINEXPLUATATION_*` show extreme negative skew (-15) because of the saturation at 1.0. Same multicollinear building-features block we already plan to reduce.

- `OBS_30/60_CNT_SOCIAL_CIRCLE` skewness (~12) comes from the single ~350 outlier we spotted earlier. Once we cap or remove it the skewness should drop a lot.

- `COMMONAREA_*` triplet (skew ~5.5) is more zero-inflation in the same building block.


### Univariate Categorical

we look at each categorical column on its own to spot dominant levels and rare categories.

In [ ]:
# bar counts for every categorical column.
# ORGANIZATION_TYPE has 58 levels, too many to plot readably, so for any
# column with more than 15 categories we show the top 15 plus an "other" bar.

cols = sorted(object_cols)

# figure out how many rows of 4 plots we need (categoricals need more width per plot than histograms)
ncols = 4
nrows = (len(cols) + ncols - 1) // ncols

# build the grid and flatten so we can loop with a single index
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3))
axes = axes.flatten()

# one bar chart per column
for i, col in enumerate(cols):
    vc = df[col].value_counts(dropna=False)
    ax = axes[i]

    # for high-cardinality columns, keep the top 15 and lump the rest into "other"
    if len(vc) > 15:
        top = vc.iloc[:15]
        other = pd.Series({"other": vc.iloc[15:].sum()})
        vc = pd.concat([top, other])

    ax.bar(range(len(vc)), vc.values, color="steelblue", edgecolor="white")
    ax.set_xticks(range(len(vc)))
    ax.set_xticklabels(vc.index, rotation=45, ha="right", fontsize=6)
    ax.set_title(col, fontsize=8)
    ax.tick_params(axis="y", labelsize=6)

# hide any leftover empty subplots in the last row
for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f"Value counts for all {len(cols)} categorical columns "
    f"(top 15 + 'other' when levels > 15)",
    y=1.002, fontsize=11,
)
plt.tight_layout()
save_plot(); plt.show()

The categorical columns confirm a few things and surface a couple of new ones.

- `CODE_GENDER` has the documented `XNA` category with very few rows (4 cases). We will drop or merge these into the majority class in Phase 2.

- `NAME_EDUCATION_TYPE` has the natural ordering noted in the data dictionary (Lower secondary < Secondary < Incomplete higher < Higher education < Academic degree). We will encode it as ordinal instead of one-hot.

- `ORGANIZATION_TYPE` has the 58-category explosion we expected. Top categories are Business Entity Type 3, XNA, Self-employed. The long tail of small categories will need grouping into "Other" before one-hot encoding, otherwise we get 58 extra columns of mostly zeros.

- `OCCUPATION_TYPE`, `FONDKAPREMONT_MODE`, `HOUSETYPE_MODE`, `WALLSMATERIAL_MODE`, `EMERGENCYSTATE_MODE` and `NAME_TYPE_SUITE` all have `nan` as a top category. Missingness here is not random and may itself carry signal, so we will impute with a literal "Missing" category instead of dropping or mode-filling.

- `NAME_INCOME_TYPE` has very rare categories (Student, Businessman, Maternity leave, Unemployed) with only a handful of rows each. Group into "Other" before encoding.

- `NAME_FAMILY_STATUS` has a tiny "Unknown" category that we should also fold into the majority or drop.

- `WEEKDAY_APPR_PROCESS_START` shows the expected weekday pattern (more applications Mon-Fri, less on weekends). Probably weak predictor but cheap to keep.

- `NAME_CONTRACT_TYPE` is mostly Cash loans (~90%) vs Revolving loans. Useful binary feature.

### Bivariate

we compare each feature against `TARGET` to see which ones look predictive.

For categorical features we plot the default rate per category; for continuous features we compare the distribution for `TARGET=0` vs `TARGET=1`. This gives a first ranking of which features look predictive and a first signal of subgroup differences in default rate.

### 6.1 Default rate by categorical subgroup

default rate per category for six categoricals tied to demographics and loan type.

For each of these six categoricals we compute how many applicants fall in each category (`n`) and what share defaulted (`default_rate`). We pick these six because they are the ones most relevant for the report, three demographics tied to fairness (`CODE_GENDER`, `NAME_EDUCATION_TYPE`, `NAME_FAMILY_STATUS`) and three loan / lifestyle descriptors (`NAME_CONTRACT_TYPE`, `NAME_INCOME_TYPE`, `NAME_HOUSING_TYPE`).

In [ ]:
# six categoricals worth looking at: three demographics (RQ3) + three loan/lifestyle
subgroup_cols = [
    "CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE",
]

In [ ]:
# default rate per category, one subplot per feature.
# red dashed line shows the overall default rate as a reference.

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(subgroup_cols):
    # share of defaulters within each category, sorted highest first
    dr = df.groupby(col)["TARGET"].mean().sort_values(ascending=False)

    ax = axes[i]
    ax.bar(range(len(dr)), dr.values, color="steelblue", edgecolor="white")
    ax.set_xticks(range(len(dr)))
    ax.set_xticklabels(dr.index, rotation=30, ha="right", fontsize=8)

    # reference line: dataset-wide default rate
    ax.axhline(default_rate, color="tomato", linestyle="--", linewidth=1.2,
               label=f"overall {default_rate:.2%}")

    ax.set_ylabel("default rate")
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Default rate by category", y=1.01)
plt.tight_layout()
save_plot(); plt.show()

This is the first chart that directly shows which categories carry signal for default. Overall default rate is 8.07%, anything far from that line is a real differentiator.

- `CODE_GENDER`, men default at ~10.1%, women at ~7.0%. Real gap of roughly 3 percentage points. Important for RQ3 fairness analysis, this is exactly the kind of disparity we will need to discuss.

- `NAME_EDUCATION_TYPE`, clean monotonic pattern. Lower secondary defaults at ~11%, Academic degree at <2%. Confirms the natural ordering and strongly justifies ordinal encoding.

- `NAME_FAMILY_STATUS`, Civil marriage and Single default the most (~10%), Widow the least (~6%). Decent spread but smaller than education.

- `NAME_CONTRACT_TYPE`, Cash loans default at ~8%, Revolving loans at ~5.5%. Useful binary signal.

- `NAME_INCOME_TYPE`, massive gap. Maternity leave (~40%) and Unemployed (~36%) default at huge rates, but these are tiny categories (handful of rows each), so the estimates are noisy. Working/Commercial associate sit near average. Pensioners and Students default the least. This is a clear case where rare categories carry strong signal but small samples, exactly why we need to think carefully before grouping them into "Other".

- `NAME_HOUSING_TYPE`, Rented apartment and With parents default at ~12%, Office apartment at ~6.5%. Housing instability tracking with credit risk.

### 6.2 Default rate by age bracket

default rate per age decade; age is the second protected attribute flagged in the plan.

`DAYS_BIRTH` is stored as a negative number of days relative to the application date. We convert to years (`age = -DAYS_BIRTH / 365.25`) and bin by decade so we can treat age as a categorical subgroup, the same way we handled the six columns in 6.1.

In [ ]:
# DAYS_BIRTH is stored as negative days (e.g. -10000 for ~27 years old).
# multiply by -1 and divide by 365.25 (accounts for leap years) to get age in years.
age_years = -df["DAYS_BIRTH"] / 365.25

# bin applicants into decade brackets so we can treat age as a categorical subgroup.
# pd.cut is right-inclusive by default, so someone aged exactly 30.0 goes into the 20-30 bin.
age_bins = pd.cut(
    age_years,
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-30", "30-40", "40-50", "50-60", "60-70"],
)

# quick sanity check that the conversion looks right (mean around 44, min ~20, max ~70)
print("age summary (years):")
age_years.describe().round(1)

In [ ]:
# default rate per age bracket, computed inline from age_bins
age_dr = df.groupby(age_bins, observed=True)["TARGET"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(age_dr)), age_dr.values, color="steelblue", edgecolor="white")
ax.set_xticks(range(len(age_dr)))
ax.set_xticklabels(age_dr.index.tolist(), rotation=0)

# reference line: dataset-wide default rate
ax.axhline(default_rate, color="tomato", linestyle="--", linewidth=1.2,
           label=f"overall {default_rate:.2%}")

ax.set_ylabel("default rate")
ax.set_title("Default rate by age band")
ax.legend()
plt.tight_layout()
save_plot(); plt.show()

Default rate by age band shows a clean monotonic pattern. Younger applicants (20-30) default at ~11.4%, older applicants (60-70) at ~4.9%. The rate drops steadily with age across all five bands.

This is a strong, interpretable signal and matches what we would expect from credit risk literature, younger borrowers tend to have shorter credit histories, less stable income, and higher default rates.

Important for RQ3 fairness, age is another protected attribute under the EU AI Act, so we will need to slice model performance by these same age bands in Phase 6 and check whether the model treats younger applicants disproportionately. The fact that the underlying default rate genuinely differs by age makes this analysis more nuanced, real risk differences vs unfair model behavior are not the same thing.

### 6.3 Default rate by income band

default rate per fixed-width income band on a round-number grid.

We split `AMT_INCOME_TOTAL` into fixed-width bands rather than quantiles. Edges follow the round-number grid 0-100k, 100-125k, ..., 350-500k, 500k+. Fixed bands show the *shape* of the risk curve at interpretable income levels (e.g. "what happens around 200k?"), which quantile binning would flatten since each quartile would just hold 25% by construction. The top band is left open-ended (500k+) to absorb the extreme-income outliers flagged in the data dictionary without creating sparse singleton bins in the tail.

In [ ]:
# fixed-width income bands on a round-number grid.
# top band is open-ended so extreme incomes (e.g. the 117M outlier) don't create 1-applicant bins.
income_edges = [0, 100_000, 125_000, 150_000, 175_000, 200_000,
                250_000, 350_000, 500_000, float("inf")]
income_labels = ["0-100k", "100-125k", "125-150k", "150-175k",
                 "175-200k", "200-250k", "250-350k", "350-500k",
                 "500k+"]

# right=False makes bins left-inclusive, so 100k goes into "100-125k" not "0-100k"
income_band = pd.cut(df["AMT_INCOME_TOTAL"], bins=income_edges,
                     labels=income_labels, right=False, include_lowest=True)

# count applicants and compute default rate per band
income_dr = (
    df.groupby(income_band, observed=True)["TARGET"]
    .agg(n="count", default_rate="mean")
    .round(4)
)
display(income_dr)

In [ ]:
# bar chart of default rate per income band, with sample size above each bar
plt.figure(figsize=(9, 4))
plt.bar(range(len(income_dr)), income_dr["default_rate"], color="steelblue")

# use the band labels as x-tick labels
plt.xticks(range(len(income_dr)), income_dr.index, rotation=30, ha="right")

# reference line: dataset-wide default rate
plt.axhline(default_rate, color="tomato", linestyle="--",
            label=f"overall {default_rate:.2%}")

# annotate each bar with n so the reader sees which bands have few applicants
for i, (rate, n) in enumerate(zip(income_dr["default_rate"], income_dr["n"])):
    plt.text(i, rate, f"n={n:,}", ha="center", va="bottom", fontsize=7)

plt.ylabel("default rate")
plt.title("Default rate by income band")
plt.legend()
plt.tight_layout()
save_plot(); plt.show()

Default rate by income band is flatter than expected. The bottom five bands (0-200k) all sit around 8-9%, basically on the overall average. The drop only starts above 200k, reaching ~5.4% in the 500k+ band.

Income alone is not a strong separator in the low-to-mid range, which is interesting. It suggests that for most applicants other features (education, age, external scores) matter more than raw income for predicting default.

The sample sizes (n= annotations) are healthy across all bands, so the pattern is reliable and not driven by noise in small buckets.

In [ ]:
# 5000-row sample used in the pairplots for readability
sample5k = df.sample(5000, random_state=SEED)

### 6.4 Pairwise scatters across feature groups

Building on the correlation ranking computed in section 7, we plot two pairwise grids coloured by `TARGET`.

The two groups are motivated by findings from the correlation matrix.

- **Top predictors group** (`EXT_SOURCE_1/2/3`, age), the three external credit scores sit at the top of the correlation-with-target ranking (|r| ≈ 0.15-0.18, the largest in the matrix), and the age gradient in 6.2 made age a natural fourth. We plot them pairwise to see where defaulters concentrate in the 2D projections.
- **Money features group** (`AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`), none of these top the correlation ranking, but they are the four amount columns that feed the engineered ratio features in preprocessing (`CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `CREDIT_TERM`, `credit_to_goods`). We plot them pairwise to see the structural relationships that motivate the ratios.

Both grids use a 5000-row subsample for readability.

In [ ]:
# top-predictors pairplot: the 3 external scores + age
# age_years derived from DAYS_BIRTH for readability
top_pred = sample5k[["TARGET", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].copy()
top_pred["age_years"] = -sample5k["DAYS_BIRTH"] / 365.25

sns.pairplot(
    top_pred.dropna(),
    hue="TARGET",
    palette={0: "steelblue", 1: "tomato"},
    plot_kws={"alpha": 0.4, "s": 12},
    diag_kind="kde",
    height=2.2,
)
plt.suptitle("Top predictors: EXT_SOURCE_1/2/3 and age", y=1.02)
save_plot(); plt.show()

- The `EXT_SOURCE_*` off-diagonals show visible class separation.
  defaulters concentrate in the low-score region across every pair of
  external scores. This is the clearest single piece of evidence that
  these three features will dominate feature importance in the models.
- The `age_years` pairs tilt slightly, with defaulters skewing toward
  younger applicants. Combined with low external scores, the
  "young + low score" corner is the highest-risk region.
- Diagonal densities show the class imbalance clearly, the `TARGET=1`
  curve is always smaller (~8% of the sample), and for the three
  external scores it is visibly shifted left (toward lower values).

In [ ]:
# money-features pairplot: the four amount columns that feed the engineered ratios
money = sample5k[["TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT",
                  "AMT_ANNUITY", "AMT_GOODS_PRICE"]]

sns.pairplot(
    money.dropna(),
    hue="TARGET",
    palette={0: "steelblue", 1: "tomato"},
    plot_kws={"alpha": 0.4, "s": 12},
    diag_kind="kde",
    height=2.2,
)
plt.suptitle("Money features: income, credit, annuity, goods price", y=1.02)
save_plot(); plt.show()

- `AMT_CREDIT` vs `AMT_ANNUITY` and `AMT_CREDIT` vs `AMT_GOODS_PRICE`
  trace tight diagonals, confirming the near-linear structure between
  these pairs. The diagonal is the expected relationship, and the
  deviation from it is exactly what the engineered ratios
  (`CREDIT_TERM`, `credit_to_goods`) capture.
- `AMT_INCOME_TOTAL` against any of the other three shows a wide
  noisy cloud with extreme-income outliers in the tail and no visible
  class separation. Raw income alone does not predict default.
- Defaulters and non-defaulters are intermixed throughout every
  off-diagonal pair in this group, which is why the engineered
  ratios add signal that the raw columns don't carry on their own.

### Correlation Heatmap

Pearson correlation on the top-20 most TARGET-correlated numeric features.

In [ ]:
# we rank numeric features by strength of their linear association with TARGET
# we exclude the ID column and TARGET itself
excluded = {"SK_ID_CURR", "TARGET"}
feature_cols = [c for c in numeric_cols if c not in excluded]

# correlation of each feature with TARGET, sorted by absolute value (strongest first)
corr_with_target = df[feature_cols].corrwith(df["TARGET"]).abs().sort_values(ascending=False)
top20 = corr_with_target.head(20).index.tolist()

# correlation matrix among top 20 features + TARGET
# keeping TARGET in lets us read off each feature's sign vs the target
corr_matrix = df[top20 + ["TARGET"]].corr()

# show the ranking so readers see what made the cut
# print("Top 20 features by |correlation with TARGET|:")
# print(corr_with_target.head(20).round(4).to_string())

In [ ]:
# correlation heatmap of the top 20 features and TARGET.
# upper triangle is masked because the matrix is symmetric (no need to show both halves).
plt.figure(figsize=(14, 12))

# mask hides the upper triangle so only the lower half and diagonal are drawn
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,     # write the numeric value in each cell
    fmt=".2f",      # two decimals
    cmap="coolwarm", center=0,   # blue for negative, red for positive, centered at 0
    linewidths=0.5,
    annot_kws={"size": 7},
)

plt.title("Pearson correlation: top 20 features + TARGET")
plt.tight_layout()
save_plot(); plt.show()

Correlation with TARGET is weak across the board. The strongest is `EXT_SOURCE_3` at -0.18, followed by `EXT_SOURCE_2` and `EXT_SOURCE_1` around -0.16. Everything else sits below 0.1 in absolute value. This is normal for tabular credit data, no single feature is a strong predictor on its own, the signal comes from combinations. It also justifies using non-linear models (MLP, Random Forest) alongside linear ones.

The three `EXT_SOURCE_*` features are the clear winners again. They correlate negatively with TARGET (higher score, lower default) which is the expected direction.

`DAYS_BIRTH` correlates 0.08 with TARGET (younger = more default), matching the age band chart.

Some very strong inter-feature correlations to flag.

- `FLOORSMAX_AVG`, `FLOORSMAX_MEDI`, `FLOORSMAX_MODE` correlate at 0.99-1.00 with each other. Perfect duplicates, same story as the other `_AVG/_MEDI/_MODE` block. Keeping one is enough.
- `REGION_RATING_CLIENT` and `REGION_RATING_CLIENT_W_CITY` correlate at 0.95. Near-duplicates, we can drop one.
- `DAYS_EMPLOYED` and `FLAG_EMP_PHONE` correlate at -1.00. This is suspicious and almost certainly an artifact of the 365243 sentinel, unemployed applicants have the sentinel in `DAYS_EMPLOYED` and 0 in `FLAG_EMP_PHONE`. Once we clean the sentinel, this correlation should drop.
- `DAYS_BIRTH` and `DAYS_EMPLOYED` correlate -0.62, and `DAYS_BIRTH` and `FLAG_EMP_PHONE` at 0.62. Makes sense, older applicants tend to have longer employment history.
- `REGION_POPULATION_RELATIVE` correlates -0.53 with both region ratings. Denser regions tend to have better ratings.


In [ ]:
# we find all feature pairs with |correlation| > threshold, sorted by strength.
# 0.5 flags both redundant columns (|r| > 0.9) and structural correlations (0.5-0.9).

threshold = 0.5

# mask the lower triangle and diagonal so each pair is counted once
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# flatten the upper triangle into a (feature_1, feature_2) -> correlation series
pairs_series = upper.stack()

# filter by threshold, sort by absolute correlation, format as a dataframe
high_corr_pairs = (
    pairs_series[pairs_series.abs() > threshold]
    .sort_values(key=abs, ascending=False)
    .round(3)
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
)

if len(high_corr_pairs) > 0:
    display(high_corr_pairs)
else:
    print(f"no pairs with |r| > {threshold} found among the top 20 features by |corr with TARGET|.")

### Outlier Scan

we scan the financial fields for extreme values using IQR flags and boxplots.

### 8.1 Overall distributions and IQR flags

histograms and IQR-based outlier flags on the financial fields.

In [ ]:
# iqr outlier detection on continuous numeric columns only. we skip binary flags,
# bounded [0,1] columns, discrete counts, and already-handled sentinels because
# iqr fences on those produce meaningless "outliers"

# pick continuous columns with real right tails, same family as the monetary ones
iqr_cols = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "OWN_CAR_AGE",
    "OBS_30_CNT_SOCIAL_CIRCLE", "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR",
]
iqr_cols = [c for c in iqr_cols if c in df.columns]

# for days_employed we mask the sentinel 365243 first, otherwise it blows up the iqr
df_iqr = df[iqr_cols].copy()

rows = []
for col in iqr_cols:
    s = df_iqr[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = int(((s < lower) | (s > upper)).sum())
    pct_out = round(n_out / len(s) * 100, 2)
    rows.append({
        "column": col,
        "n_outliers": n_out,
        "pct_outliers": pct_out,
        "lower_fence": round(lower, 2),
        "upper_fence": round(upper, 2),
        "max_value": s.max(),
        "min_value": s.min(),
    })

iqr_summary = pd.DataFrame(rows).sort_values("pct_outliers", ascending=False).reset_index(drop=True)
display(iqr_summary)

The IQR analysis splits the columns into three groups.

- **Heavy-tailed distributions (>5% flagged as outliers)**, `AMT_REQ_CREDIT_BUREAU_QRT` (19%), `AMT_REQ_CREDIT_BUREAU_MON` (16%), `DEF_30_CNT_SOCIAL_CIRCLE` (11%), `DEF_60_CNT_SOCIAL_CIRCLE` (8.4%), `OBS_30/60_CNT_SOCIAL_CIRCLE` (~6.5%). These are zero-inflated counts where IQR is basically flagging "anything above 0" as an outlier. Not real outliers, just the shape of the data. We will leave them as is.

- **Real right-tail outliers (1-5%)**, `AMT_GOODS_PRICE` (4.8%), `OWN_CAR_AGE` (4.7%), `AMT_INCOME_TOTAL` (4.6%), `AMT_ANNUITY` (2.4%), `AMT_CREDIT` (2.1%). These are the monetary and continuous columns that need winsorization at the 99th percentile.

- **Extreme single-point issues**, look at the max values. `AMT_INCOME_TOTAL` max is 117,000,000 against an upper fence of 337,500 (confirms the documented 117M outlier). `AMT_CREDIT` and `AMT_GOODS_PRICE` both reach 4,050,000. `OBS_30/60_CNT_SOCIAL_CIRCLE` hit 348 and 344 (the extreme points we spotted in the boxplots earlier). `CNT_CHILDREN` max is 19, worth sanity-checking whether that is real.

- **Low/no outlier columns**, `DAYS_REGISTRATION`, `DAYS_LAST_PHONE_CHANGE`, `DAYS_ID_PUBLISH` are clean, negative values as expected.

In [ ]:
fin_cols = [c for c in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
            if c in df.columns]

fig, axes = plt.subplots(1, len(fin_cols), figsize=(16, 5))
for ax, col in zip(axes, fin_cols):
    ax.boxplot(
        df[col].dropna(), vert=True, patch_artist=True,
        boxprops=dict(facecolor="steelblue", alpha=0.7),
        flierprops=dict(marker=".", markersize=2, alpha=0.3)
    )
    ax.set_title(col, fontsize=9)

plt.suptitle("Overall boxplots -- key financial columns")
plt.tight_layout()
save_plot(); plt.show()


- `AMT_INCOME_TOTAL` is the worst case, the box is completely flattened at the bottom because of the 117M outlier stretching the y-axis to 1.2×10⁸. Without capping, any standardization on this column would be dominated by a handful of extreme values.

- `AMT_CREDIT` and `AMT_GOODS_PRICE` look similar to each other, with medians around 500k, IQR up to ~1.5M, and a long tail of outliers reaching 4M.

- `AMT_ANNUITY` follows the same pattern on a smaller scale, median around 25k, outliers above 150k.


### 8.2 Boxplots split by TARGET

Same financial fields, split by defaulter / non-defaulter. Read medians, not whiskers.

In [ ]:
fig, axes = plt.subplots(1, len(fin_cols), figsize=(16, 5))
for ax, col in zip(axes, fin_cols):
    sns.boxplot(
        data=df, x="TARGET", y=col, ax=ax,
        hue="TARGET", palette={0: "steelblue", 1: "tomato"},
        order=[0, 1], legend=False
    )
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("TARGET  (0=no default, 1=default)")

plt.suptitle("Boxplots by TARGET -- key financial columns")
plt.tight_layout()
save_plot(); plt.show()

Same four monetary columns, now split by TARGET. The boxes for defaulters (1) and non-defaulters (0) look almost identical across all four features.

- `AMT_INCOME_TOTAL` is unreadable because of the 117M outlier. The 117M case is a non-defaulter (TARGET=0), which is a small data-cleaning detail worth noting but changes nothing about the conclusion.

- `AMT_CREDIT`, defaulters have a slightly lower median and tighter IQR, but the overlap is massive.

- `AMT_ANNUITY`, essentially identical distributions between the two classes.

- `AMT_GOODS_PRICE`, same story, near-identical boxes.

### Missing Values Audit

we audit which columns are missing and group them by the mechanism that caused the missingness.

The plan asks for a bar chart of % missing per column. We go further, 67 of the 122 columns have at least one missing value, and they do not fail independently. Grouping them by cause reduces a 67-column problem to six mechanisms of different sizes and strengths, which fundamentally changes the preprocessing decision in Phase 2.

We placed this section at the end of the EDA, not at the start as the plan's task ordering suggests, because our grouped-by-cause analysis is the most substantial part of this notebook and reads better as a culmination than as a housekeeping step.

In [ ]:
# we check for nulls (as percentage)

(df.isna().sum() / len(df) * 100).round(2).sort_values(ascending=False).head(20)

In [ ]:
# how many columns have nulls

null_cols = df.isna().sum().gt(0).sum()

print(f'Null columns: {null_cols}')

In [ ]:
# total nulls in the dataset

total_nulls = df.isna().sum().sum()

total_cells = df.size

print(f"total nulls: {total_nulls:,} ({total_nulls / total_cells * 100:.2f}%)")


67 out of 122 columns contain nulls, and the total share of missing cells in the dataset is ~24%. Looking at the list, the null pattern is not random, the same column families show identical missing rates (the three `COMMONAREA_*` variants all at 69.87%, the three `NONLIVINGAPARTMENTS_*` at 69.43%, and so on). This tells us the nulls share an upstream cause rather than being 67 independent problems.

To deal with them in a meaningful way, we group the 67 columns by the reason they are missing, based on the data dictionary.

- **G1 building block**, apartment and building features, missing when no appraisal data exists for the building.
- **G2 external scores**, the three `EXT_SOURCE_*` columns, missing when the applicant has no record at that bureau.
- **G3 optional personal**, `OCCUPATION_TYPE` and `NAME_TYPE_SUITE`, missing for pensioners/unemployed or applicants who came alone.
- **G4 bureau enquiries**, `AMT_REQ_CREDIT_BUREAU_*`, missing when the bureau query was not run.
- **G5 social circle**, `OBS_*` and `DEF_*` counts, missing when social-circle data was not collected.
- **G6 isolated small gaps**, a handful of columns with very low missingness (`OWN_CAR_AGE`, `AMT_ANNUITY`, etc.) that do not fit the patterns above.

In [ ]:
# we define the six missingness groups once, used by every cell below.
# grouping follows the data dictionary's explanations, not an automatic clustering.

g1 = [
    "APARTMENTS_AVG", "APARTMENTS_MODE", "APARTMENTS_MEDI",
    "BASEMENTAREA_AVG", "BASEMENTAREA_MODE", "BASEMENTAREA_MEDI",
    "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BEGINEXPLUATATION_MODE", "YEARS_BEGINEXPLUATATION_MEDI",
    "YEARS_BUILD_AVG", "YEARS_BUILD_MODE", "YEARS_BUILD_MEDI",
    "COMMONAREA_AVG", "COMMONAREA_MODE", "COMMONAREA_MEDI",
    "ELEVATORS_AVG", "ELEVATORS_MODE", "ELEVATORS_MEDI",
    "ENTRANCES_AVG", "ENTRANCES_MODE", "ENTRANCES_MEDI",
    "FLOORSMAX_AVG", "FLOORSMAX_MODE", "FLOORSMAX_MEDI",
    "FLOORSMIN_AVG", "FLOORSMIN_MODE", "FLOORSMIN_MEDI",
    "LANDAREA_AVG", "LANDAREA_MODE", "LANDAREA_MEDI",
    "LIVINGAPARTMENTS_AVG", "LIVINGAPARTMENTS_MODE", "LIVINGAPARTMENTS_MEDI",
    "LIVINGAREA_AVG", "LIVINGAREA_MODE", "LIVINGAREA_MEDI",
    "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAREA_AVG", "NONLIVINGAREA_MODE", "NONLIVINGAREA_MEDI",
    "FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "TOTALAREA_MODE",
    "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE",
]

g2 = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

g3 = ["OCCUPATION_TYPE", "NAME_TYPE_SUITE"]

g4 = [
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

g5 = [
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
]

g6 = [
    "OWN_CAR_AGE",
    "CNT_FAM_MEMBERS",
    "DAYS_LAST_PHONE_CHANGE",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
]

MISSINGNESS_GROUPS = {
    "G1 building block":       g1,
    "G2 external scores":      g2,
    "G3 optional personal":    g3,
    "G4 bureau enquiries":     g4,
    "G5 social circle":        g5,
    "G6 isolated small gaps":  g6,
}

# keep all_missing as a list so downstream cells can iterate it in a fixed order
all_missing = df.columns[df.isnull().any()].tolist()
grouped = set().union(*MISSINGNESS_GROUPS.values())

missing_from_groups = set(all_missing) - grouped
extra_in_groups = grouped - set(df.columns)

print(f"columns with any missing: {len(all_missing)}")
print(f"columns covered by groups: {len(set(all_missing) & grouped)}")
if missing_from_groups:
    print(f"columns with nulls NOT in any group: {missing_from_groups}")
if extra_in_groups:
    print(f"columns in groups but not in df: {extra_in_groups}")

In [ ]:
# horizontal bars are a deliberate exception to the vertical-bar house style:
# 67 categories stacked vertically would be unreadable.
# color maps each bar to its missingness group so the clustering is visible at a glance.

group_colors = {
    "G1 building block":       "#1f77b4",
    "G2 external scores":      "#ff7f0e",
    "G3 optional personal":    "#2ca02c",
    "G4 bureau enquiries":     "#d62728",
    "G5 social circle":        "#9467bd",
    "G6 isolated small gaps":  "#8c564b",
}

# reverse-lookup: column name -> group name
col_to_group = {c: g for g, cols in MISSINGNESS_GROUPS.items() for c in cols}

# one row per missing column with its percent missing and its group
miss_df = (
    pd.DataFrame({
        "column": all_missing,
        "pct_missing": [df[c].isnull().mean() * 100 for c in all_missing],
        "group": [col_to_group[c] for c in all_missing],
    })
    .sort_values("pct_missing", ascending=True)
    .reset_index(drop=True)
)

plt.figure(figsize=(8, 14))
colors = [group_colors[g] for g in miss_df["group"]]
plt.barh(range(len(miss_df)), miss_df["pct_missing"], color=colors)
plt.yticks(range(len(miss_df)), miss_df["column"], fontsize=7)
plt.axvline(50, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
plt.xlabel("percent missing (50% reference line, not a drop threshold)")
plt.title(f"Missing values per column, grouped by cause ({len(miss_df)} columns)")

# one legend entry per group
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, label=g) for g, c in group_colors.items()]
plt.legend(handles=handles, loc="lower right", fontsize=8)

plt.tight_layout()
save_plot(); plt.show()


The chart makes the grouping visible at a glance. The big blue block at the top (G1) is the building-info cascade, all clustered between 47% and 70% missing. The orange bars (G2, external scores) show that `EXT_SOURCE_1` is missing in 56% of cases but `EXT_SOURCE_3` in only 20% and `EXT_SOURCE_2` in less than 1%. The other groups sit well below the 50% reference line.

The 50% line is drawn as a reference, not as an automatic drop threshold. The actual treatment per group in Phase 2 is.

- **G1 building block**, drop the whole block. Imputing 60% of a column with the median means inventing most of the data, and the correlation with TARGET is essentially zero. Top Kaggle solutions on this dataset did not rely on these features either. Regularization cannot recover information that was never there.
- **G2 external scores**, keep. These are the three strongest predictors of default in the whole dataset, so dropping them is not an option even at 56% missingness. Impute with the median and add a binary missingness indicator so the fact of being missing is preserved as its own feature.
- **G3 optional personal**, keep. Missingness here is semantically meaningful (no occupation = pensioner/unemployed, no companion = came alone). Impute with a literal "Missing" category instead of collapsing into the mode.
- **G4 bureau enquiries**, keep. Low missingness (~13%) and potentially useful signal. Plain median imputation, no flag.
- **G5 social circle**, keep. Negligible missingness. Plain median imputation.
- **G6 isolated small gaps**, mostly drop `OWN_CAR_AGE` (too high and too redundant with `FLAG_OWN_CAR`), plain impute the rest.

Net effect, we go from 122 columns to roughly 74, before one-hot encoding expands the categoricals. All imputers will be fit on the training split only and applied to validation and test through the shared Pipeline, to avoid leakage.

### 2.10 Findings to treatments

each EDA finding above maps to a candidate treatment, and the chosen treatment lives in section 3 not here. this is the bridge between the descriptive work in this section and the implementation in the next.

| finding | candidate treatments | decided in section 3 |
|---|---|---|
| 6 rows have XNA or Unknown in CODE_GENDER and NAME_FAMILY_STATUS | drop rows, impute mode, keep with sentinel | drop the 6 rows |
| DAYS_EMPLOYED has a 365243 sentinel for unemployed applicants | replace with NaN plus a binary flag, drop column, keep sentinel | NaN plus DAYS_EMPLOYED_ANOM flag |
| DAYS_BIRTH stored as negative days | flip sign, convert to years | derive age_years equal to negative DAYS_BIRTH divided by 365.25 |
| 49 cols in the G1 building block plus rare doc flags plus zero-variance flags plus IDs | drop all, keep with imputation, pca-collapse | drop 66 cols total |
| extreme single-point outliers in CNT_CHILDREN, CNT_FAM_MEMBERS, OBS_30/60_CNT_SOCIAL_CIRCLE, AMT_REQ_CREDIT_BUREAU_QRT | hard-cap, winsorize, keep | hard-cap to plausible max |
| OCCUPATION_TYPE and NAME_TYPE_SUITE have semantic missing | impute Missing as a category, impute mode, drop | impute the literal Missing |
| ORGANIZATION_TYPE (58), NAME_INCOME_TYPE (8), OCCUPATION_TYPE (19) high cardinality | rare-group at threshold, target-encode, keep all | group below 1 percent to Other, protect Maternity leave and Unemployed |
| AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE heavy right tails | winsorize, log-transform, keep raw | winsorize at 99th pct then log1p then scale |
| EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3 are the strongest predictors but have missingness | impute median plus missingness flag, impute median only, drop | impute median plus add MISSING flag |
| NAME_EDUCATION_TYPE has a natural order | ordinal encode, one-hot, target-encode | ordinal encode |
| target is 8.07 percent defaults, heavily imbalanced | SMOTE, class_weight balanced, scale_pos_weight | both, compared per model |

## **3. Preprocessing and Feature Engineering**

two branches. the standard pipeline produces the 113-feature dataset used by most models. the PCA branch projects to 49 components retaining 95 percent variance, used by the LR-PCA and MLP-PCA experiments.

each branch is checkpointed. the first run computes everything and saves to checkpoints, subsequent runs detect the checkpoints and load instantly.

### 3.1 Standard pipeline

In [ ]:
# we reload the raw data for preprocessing

df = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))
print(f"raw shape: {df.shape}")


In [ ]:
# we drop ambiguous rows

before = len(df)

df = df[df['CODE_GENDER'] != 'XNA']
df = df[df['NAME_FAMILY_STATUS'] != 'Unknown']
df = df.reset_index(drop=True)

print(f"dropped {before - len(df)} ambiguous rows, new shape: {df.shape}")


In [ ]:
# 365243 is a fake value for unemployed applicants

df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"unemployed flagged: {df['DAYS_EMPLOYED_ANOM'].sum():,}")


In [ ]:
# we convert age to years from negative days

df['age_years'] = -df['DAYS_BIRTH'] / 365.25

print(f"age range: {df['age_years'].min():.1f} to {df['age_years'].max():.1f}")


In [ ]:
# we flip the sign on the remaining DAYS columns so positive means longer ago

for col in ['DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']:
    df[col] = -df[col]

df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].apply(lambda x: -x if pd.notna(x) and x < 0 else x)

print("DAYS columns flipped to positive")


In [ ]:
# we engineer four ratio features

df['CREDIT_INCOME_RATIO']  = df['AMT_CREDIT']  / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_TERM']          = df['AMT_CREDIT']  / df['AMT_ANNUITY']
df['CREDIT_GOODS_RATIO']   = df['AMT_CREDIT']  / df['AMT_GOODS_PRICE']

for col in ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'CREDIT_GOODS_RATIO']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

print(f"shape after feature engineering: {df.shape}")


In [ ]:
# we drop the high-missing G1 building block, rare doc flags, zero-variance flags, IDs

COLS_TO_DROP = [
    'SK_ID_CURR', 'DAYS_BIRTH',
    'FLAG_MOBIL', 'FLAG_CONT_MOBILE',
    'REGION_RATING_CLIENT_W_CITY', 'OWN_CAR_AGE',
    'APARTMENTS_AVG','APARTMENTS_MEDI','APARTMENTS_MODE',
    'BASEMENTAREA_AVG','BASEMENTAREA_MEDI','BASEMENTAREA_MODE',
    'YEARS_BEGINEXPLUATATION_AVG','YEARS_BEGINEXPLUATATION_MEDI','YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BUILD_AVG','YEARS_BUILD_MEDI','YEARS_BUILD_MODE',
    'COMMONAREA_AVG','COMMONAREA_MEDI','COMMONAREA_MODE',
    'ELEVATORS_AVG','ELEVATORS_MEDI','ELEVATORS_MODE',
    'ENTRANCES_AVG','ENTRANCES_MEDI','ENTRANCES_MODE',
    'FLOORSMAX_AVG','FLOORSMAX_MEDI','FLOORSMAX_MODE',
    'FLOORSMIN_AVG','FLOORSMIN_MEDI','FLOORSMIN_MODE',
    'LANDAREA_AVG','LANDAREA_MEDI','LANDAREA_MODE',
    'LIVINGAPARTMENTS_AVG','LIVINGAPARTMENTS_MEDI','LIVINGAPARTMENTS_MODE',
    'LIVINGAREA_AVG','LIVINGAREA_MEDI','LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_AVG','NONLIVINGAPARTMENTS_MEDI','NONLIVINGAPARTMENTS_MODE',
    'NONLIVINGAREA_AVG','NONLIVINGAREA_MEDI','NONLIVINGAREA_MODE',
    'FONDKAPREMONT_MODE','HOUSETYPE_MODE','TOTALAREA_MODE','WALLSMATERIAL_MODE','EMERGENCYSTATE_MODE',
    'FLAG_DOCUMENT_2','FLAG_DOCUMENT_4','FLAG_DOCUMENT_7','FLAG_DOCUMENT_9','FLAG_DOCUMENT_10',
    'FLAG_DOCUMENT_12','FLAG_DOCUMENT_13','FLAG_DOCUMENT_14','FLAG_DOCUMENT_15',
    'FLAG_DOCUMENT_17','FLAG_DOCUMENT_19','FLAG_DOCUMENT_20','FLAG_DOCUMENT_21',
]

existing_drops = [c for c in COLS_TO_DROP if c in df.columns]
df.drop(columns=existing_drops, inplace=True)

print(f"dropped {len(existing_drops)} columns, shape: {df.shape}")


In [ ]:
# we cap a few extreme outliers identified in the EDA

HARD_CAPS = {
    'CNT_CHILDREN':              10,
    'CNT_FAM_MEMBERS':           10,
    'OBS_30_CNT_SOCIAL_CIRCLE':  30,
    'OBS_60_CNT_SOCIAL_CIRCLE':  30,
    'AMT_REQ_CREDIT_BUREAU_QRT': 20,
}

for col, cap in HARD_CAPS.items():
    if col in df.columns:
        n_capped = (df[col] > cap).sum()
        df[col] = df[col].clip(upper=cap)
        print(f"  {col}: capped {n_capped} values at {cap}")


In [ ]:
# stratified train val test split, 70 15 15

X = df.drop(columns=['TARGET'])
y = df['TARGET']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print(f"train: {X_train.shape}  | default rate: {y_train.mean():.4f}")
print(f"val:   {X_val.shape}   | default rate: {y_val.mean():.4f}")
print(f"test:  {X_test.shape}  | default rate: {y_test.mean():.4f}")


In [ ]:
# we group columns by how they should be encoded

ordinal_cols = ['NAME_EDUCATION_TYPE']
EDUCATION_ORDER = [
    ['Lower secondary', 'Secondary / secondary special',
     'Incomplete higher', 'Higher education', 'Academic degree']
]

onehot_cols = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE',
    'OCCUPATION_TYPE',
]
onehot_cols = [c for c in onehot_cols if c in X_train.columns]

monetary_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']
monetary_cols = [c for c in monetary_cols if c in X_train.columns]

ext_source_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
ext_source_cols = [c for c in ext_source_cols if c in X_train.columns]

print(f"ordinal: {len(ordinal_cols)}, onehot: {len(onehot_cols)}, monetary: {len(monetary_cols)}, ext: {len(ext_source_cols)}")


In [ ]:
# we add missingness flags for the EXT_SOURCE columns becuase missing is informative

for split_X in [X_train, X_val, X_test]:
    for col in ext_source_cols:
        split_X[f'{col}_MISSING'] = split_X[col].isna().astype(int)

ext_missing_flags = [f'{c}_MISSING' for c in ext_source_cols]
print(f"missingness flags added: {ext_missing_flags}")


In [ ]:
# everything else numeric goes to a default impute and scale pipeline

all_cat = set(ordinal_cols + onehot_cols)
all_special_numeric = set(monetary_cols + ext_source_cols + ext_missing_flags + ['DAYS_EMPLOYED_ANOM'])

numeric_rest = [
    c for c in X_train.columns
    if c not in all_cat and c not in all_special_numeric
    and X_train[c].dtype in [np.float64, np.int64, np.float32, np.int32]
]

print(f"remaining numeric: {len(numeric_rest)}")


In [ ]:
# fill NaNs in semantic categoricals with the literal Missing

for col in ['OCCUPATION_TYPE', 'NAME_TYPE_SUITE']:
    for split_X in [X_train, X_val, X_test]:
        if col in split_X.columns:
            split_X[col] = split_X[col].fillna('Missing')

print("categorical NaNs filled with 'Missing'")


In [ ]:
# we group rare categories below 1 percent into Other, protecting strong default signals

RARE_THRESHOLD = 0.01
HIGH_CARD_COLS = ['ORGANIZATION_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE']
PROTECTED_CATS = {'NAME_INCOME_TYPE': ['Maternity leave', 'Unemployed']}

for col in HIGH_CARD_COLS:
    if col not in X_train.columns:
        continue
    freq = X_train[col].value_counts(normalize=True)
    protected = PROTECTED_CATS.get(col, [])
    rare_cats = freq[(freq < RARE_THRESHOLD) & (~freq.index.isin(protected))].index.tolist()

    for split_X in [X_train, X_val, X_test]:
        split_X[col] = split_X[col].apply(lambda v: 'Other' if v in rare_cats else v)

    print(f"  {col}: {len(freq)} to {X_train[col].nunique()} categories")


In [ ]:
# we compute winsorization caps from training only to avoid leakage

winsor_caps = {col: X_train[col].quantile(0.99) for col in monetary_cols}

for col, cap in winsor_caps.items():
    print(f"  {col}: {cap:,.0f}")


In [ ]:
# helper transforms used inside the monetary pipeline

def winsorize_monetary(X, caps=winsor_caps, cols=monetary_cols):
    X = X.copy()
    for i, col in enumerate(cols):
        X[:, i] = np.clip(X[:, i], a_min=None, a_max=caps[col])
    return X

def log1p_transform(X):
    return np.log1p(np.abs(X))

print("transform helpers defined")


In [ ]:
# we build one Pipeline per column group

monetary_pipeline = Pipeline([
    ('impute',    SimpleImputer(strategy='median')),
    ('winsorize', FunctionTransformer(winsorize_monetary, validate=False, feature_names_out='one-to-one')),
    ('log1p',     FunctionTransformer(log1p_transform, validate=False, feature_names_out='one-to-one')),
    ('scale',     StandardScaler()),
])

ext_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

ordinal_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OrdinalEncoder(categories=EDUCATION_ORDER,
                              handle_unknown='use_encoded_value', unknown_value=-1)),
])

onehot_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])

print("pipelines defined")


In [ ]:
# we assemble the ColumnTransformer and fit on training only

passthrough_cols = ext_missing_flags + ['DAYS_EMPLOYED_ANOM']
passthrough_cols = [c for c in passthrough_cols if c in X_train.columns]

preprocessor = ColumnTransformer(
    transformers=[
        ('monetary', monetary_pipeline,  monetary_cols),
        ('ext_src',  ext_pipeline,       ext_source_cols),
        ('ordinal',  ordinal_pipeline,   ordinal_cols),
        ('onehot',   onehot_pipeline,    onehot_cols),
        ('numeric',  numeric_pipeline,   numeric_rest),
        ('passthru', 'passthrough',      passthrough_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

feature_names_raw = preprocessor.get_feature_names_out()

print(f"X_train_proc: {X_train_proc.shape}")
print(f"features: {len(feature_names_raw)}")


In [ ]:
# sanity, no NaN or Inf survived

train_df_proc = pd.DataFrame(X_train_proc, columns=feature_names_raw)

print(f"remaining NaN: {train_df_proc.isna().sum().sum()}")
print(f"remaining Inf: {np.isinf(X_train_proc).sum()}")


In [ ]:
# we eyeball the post-scaling ranges, mean should be near 0 and std near 1

print(train_df_proc.describe().loc[['mean', 'std', 'min', 'max']].round(3).T.head(20))


In [ ]:
# we plot post-preprocessing distributions for a few key features

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

check_features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_EMPLOYED', 'age_years', 'CREDIT_INCOME_RATIO'
]

for i, feat in enumerate(check_features):
    if feat in feature_names_raw:
        idx = list(feature_names_raw).index(feat)
        axes[i].hist(X_train_proc[:, idx], bins=50, color='steelblue', edgecolor='none')
        axes[i].set_title(feat, fontsize=8)
    else:
        axes[i].set_visible(False)

plt.suptitle("Preprocessed feature distributions (training set)")
plt.tight_layout()
save_plot('preproc_distributions')
plt.show()


In [ ]:
# we apply SMOTE on the training split only

smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_smote_arr, y_train_smote = smote.fit_resample(X_train_proc, y_train)

print(f"SMOTE: {X_train_proc.shape[0]:,} to {X_train_smote_arr.shape[0]:,} rows")


In [ ]:
# we wrap matrices as DataFrames so model sections can index by feature name

feature_names = list(feature_names_raw)

X_train       = pd.DataFrame(X_train_proc,      columns=feature_names)
X_val         = pd.DataFrame(X_val_proc,        columns=feature_names)
X_test        = pd.DataFrame(X_test_proc,       columns=feature_names)
X_train_smote = pd.DataFrame(X_train_smote_arr, columns=feature_names)

print(f"X_train: {X_train.shape}, X_train_smote: {X_train_smote.shape}")


In [ ]:
# we save all preprocessing artifacts so re-runs skip this whole block

for key, val in [
    ('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
    ('X_train_smote', X_train_smote),
    ('y_train', y_train), ('y_val', y_val), ('y_test', y_test),
    ('y_train_smote', pd.Series(y_train_smote, name='TARGET')),
    ('preprocessor', preprocessor),
    ('feature_names_raw', feature_names_raw),
]:
    joblib.dump(val, os.path.join(CHECKPOINT_DIR, f"{key}.pkl"))

print("preprocessing checkpoints saved")


### 3.2 PCA dimensionality reduction

In [ ]:
PCA_VARIANCE_THRESHOLD = 0.95

print(f"PCA target: {PCA_VARIANCE_THRESHOLD*100:.0f}% variance")


In [ ]:
# we fit a full PCA to inspect the variance spectrum

pca_full = PCA(random_state=SEED)
pca_full.fit(X_train.values)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_total = X_train.shape[1]

print(f"first PC explains {pca_full.explained_variance_ratio_[0]*100:.1f}%")
print(f"cumulative variance for 10 PCs: {cumvar[9]*100:.1f}%")


In [ ]:
# we pick the smallest k that hits the variance threshold

n_components_selected = int(np.searchsorted(cumvar, PCA_VARIANCE_THRESHOLD)) + 1

print(f"selected {n_components_selected} components for {PCA_VARIANCE_THRESHOLD*100:.0f}% variance")
print(f"reduction: {n_total} to {n_components_selected} ({100*(1 - n_components_selected/n_total):.1f}% removed)")


In [ ]:
# we fit the final PCA and project all splits

pca = PCA(n_components=n_components_selected, random_state=SEED)
X_train_pca_arr = pca.fit_transform(X_train.values)
X_val_pca_arr   = pca.transform(X_val.values)
X_test_pca_arr  = pca.transform(X_test.values)

pca_feature_names = [f"PC{i+1}" for i in range(n_components_selected)]

print(f"X_train_pca shape: {X_train_pca_arr.shape}")


In [ ]:
# we apply SMOTE in the PCA space

smote_pca = SMOTE(random_state=SEED, k_neighbors=5)
X_train_smote_pca_arr, y_train_smote_pca = smote_pca.fit_resample(X_train_pca_arr, y_train)

print(f"SMOTE PCA: {X_train_pca_arr.shape[0]:,} to {X_train_smote_pca_arr.shape[0]:,} rows")


In [ ]:
# wrap as DataFrames so the rest of the notebook can index by PC name

X_train_pca       = pd.DataFrame(X_train_pca_arr,       columns=pca_feature_names)
X_val_pca         = pd.DataFrame(X_val_pca_arr,         columns=pca_feature_names)
X_test_pca        = pd.DataFrame(X_test_pca_arr,        columns=pca_feature_names)
X_train_smote_pca = pd.DataFrame(X_train_smote_pca_arr, columns=pca_feature_names)
y_train_smote_pca = pd.Series(y_train_smote_pca, name='TARGET')

print(f"X_train_pca DataFrame: {X_train_pca.shape}")


In [ ]:
# we save the PCA artifacts

for key, val in [('X_train_pca', X_train_pca), ('X_val_pca', X_val_pca),
                 ('X_test_pca', X_test_pca), ('X_train_smote_pca', X_train_smote_pca),
                 ('y_train_smote_pca', y_train_smote_pca),
                 ('pca', pca), ('pca_feature_names', pca_feature_names)]:
    joblib.dump(val, os.path.join(CHECKPOINT_DIR, f"pca_{key}.pkl"))

print("PCA checkpoints saved")


### 3.2.1 PCA diagnostics, scree plot

we look at the variance spectrum so we know whether 95 percent is the right cut-off or if we should retain more components.

In [ ]:
# scree plot + cumulative variance curve (always regenerated, fast)
pca_full_for_plot = PCA(random_state=SEED).fit(X_train.values)
cumvar = np.cumsum(pca_full_for_plot.explained_variance_ratio_)
n_total = X_train.shape[1]
n_components_95 = int(np.searchsorted(cumvar, 0.95)) + 1
n_components_99 = int(np.searchsorted(cumvar, 0.99)) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PCA Variance Analysis", fontsize=14, fontweight='bold')
n_show = min(60, n_total)
axes[0].bar(range(1, n_show + 1),
            pca_full_for_plot.explained_variance_ratio_[:n_show] * 100,
            color='steelblue', edgecolor='none')
axes[0].set_xlabel("Principal Component"); axes[0].set_ylabel("Individual Explained Variance (%)")
axes[0].set_title(f"Scree Plot (first {n_show} components)")

axes[1].plot(range(1, n_total + 1), cumvar * 100, color='steelblue', linewidth=1.5)
axes[1].axhline(95, color='tomato',     linestyle='--', linewidth=1.2, label='95% threshold')
axes[1].axhline(99, color='darkorange', linestyle='--', linewidth=1.2, label='99% threshold')
axes[1].axvline(n_components_95, color='tomato',     linestyle=':',  linewidth=1)
axes[1].axvline(n_components_99, color='darkorange', linestyle=':',  linewidth=1)
axes[1].set_xlabel("Number of Components"); axes[1].set_ylabel("Cumulative Explained Variance (%)")
axes[1].set_title("Cumulative Explained Variance"); axes[1].legend()
axes[1].set_ylim(0, 102)
plt.tight_layout()
save_plot('pca_scree'); plt.show()


### 3.2.2 PCA diagnostics, 2D scatter PC1 vs PC2

we plot PC1 against PC2 coloured by target class. tabular credit data rarely separates cleanly in 2D so we expect overlap, but if there is no overlap that would be a red flag for leakage.

In [ ]:
# 2D scatter PC1 vs PC2 by target class
rng = np.random.default_rng(SEED)
sample_n = min(6000, len(X_train_pca))
idx = rng.choice(len(X_train_pca), size=sample_n, replace=False)

fig, ax = plt.subplots(figsize=(8, 6))
for label, color, marker, zorder in [(0, 'steelblue', '.', 1), (1, 'tomato', 'x', 2)]:
    mask = y_train.values[idx] == label
    ax.scatter(X_train_pca.values[idx][mask, 0], X_train_pca.values[idx][mask, 1],
               c=color, marker=marker, alpha=0.35, s=12, zorder=zorder,
               label=f"{'No Default (0)' if label == 0 else 'Default (1)'}")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("Training Set: PC1 vs PC2 by Target Class (sample)")
ax.legend()
plt.tight_layout()
save_plot('pca_scatter_2d'); plt.show()


### 3.2.3 PCA diagnostics, top loadings per component

each principal component is a linear combination of the original features. plotting the largest absolute loadings tells us what each PC actually represents.

In [ ]:
# top contributing original features per PC
N_TOP = 10
N_PCS_TO_SHOW = 4

fig, axes = plt.subplots(1, N_PCS_TO_SHOW, figsize=(5 * N_PCS_TO_SHOW, 5))
for pc_idx in range(N_PCS_TO_SHOW):
    loadings = pd.Series(np.abs(pca.components_[pc_idx]),
                         index=feature_names_raw).nlargest(N_TOP)
    axes[pc_idx].barh(loadings.index[::-1], loadings.values[::-1], color='steelblue')
    axes[pc_idx].set_title(f"PC{pc_idx+1}  ({pca.explained_variance_ratio_[pc_idx]*100:.1f}% var)")
    axes[pc_idx].set_xlabel("|Loading|")
    axes[pc_idx].tick_params(axis='y', labelsize=8)

plt.suptitle(f"Top {N_TOP} Feature Loadings per Principal Component", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
save_plot('pca_loadings'); plt.show()


## **4. Modeling, Baseline (Logistic Regression)**

L2-regularised logistic regression. we compare the original training set against SMOTE-balanced training. fitted models are saved to outputs and second runs reuse them.

### 4.1 Standard features

### Train Baseline Model, Original Training Set

Logistic Regression with `class_weight='balanced'` to compensate for the ~8% default rate.
`lbfgs` solver with L2 regularisation (default C=1.0). `max_iter=1000` to ensure convergence.

> **Note.** We use `class_weight='balanced'` rather than SMOTE here. Section 5 trains on
> the SMOTE set so we can directly compare both strategies on the same val/test splits.

In [ ]:
lr_base = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    solver='lbfgs',
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1,
)

print("Training Logistic Regression on original (imbalanced) training set...")
lr_base.fit(X_train, y_train)
print(f"  Converged in {lr_base.n_iter_[0]} iterations.")
# print(lr_base.coef_.shape)  # was checking shape during debugging

print("done")


In [ ]:
# saving

joblib.dump(lr_base, os.path.join(OUTPUT_DIR, 'lr_base.pkl'))
print("saved")


### 4.1 Evaluate on Validation Set

In [ ]:
results_base_val  = evaluate_model(lr_base, X_val,  y_val,  split_name="Val  (original train)")
results_base_test = evaluate_model(lr_base, X_test, y_test, split_name="Test (original train)")

plot_evaluation(lr_base, X_val, y_val, X_test, y_test,
                title_prefix="LR Baseline — class_weight='balanced'")

### Train Baseline Model, SMOTE Training Set

Same hyperparameters, trained on the SMOTE-balanced set. SMOTE creates synthetic minority
samples during training, val/test remain the original distribution so evaluation is fair.

In [ ]:
lr_smote = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight=None,     # class imbalance handled by SMOTE, not by weighting
    solver='lbfgs',
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1,
)

print("Training Logistic Regression on SMOTE-resampled training set...")
lr_smote.fit(X_train_smote, y_train_smote)
print(f"  Converged in {lr_smote.n_iter_[0]} iterations.")

results_smote_val  = evaluate_model(lr_smote, X_val,  y_val,  split_name="Val  (SMOTE train)")
results_smote_test = evaluate_model(lr_smote, X_test, y_test, split_name="Test (SMOTE train)")

plot_evaluation(lr_smote, X_val, y_val, X_test, y_test,
                title_prefix="LR Baseline — SMOTE")

print("done")


In [ ]:
# saving

joblib.dump(lr_smote, os.path.join(OUTPUT_DIR, 'lr_smote.pkl'))
print("saved")


we keep both fits even though the SMOTE one usually wins on recall, becuase comparing the precision tradeoff is the whole point of section 5 in the elastic net story.

### Model Comparison Summary

Side-by-side ROC-AUC and PR-AUC for both training strategies across val and test splits.

In [ ]:
summary = pd.DataFrame([
    results_base_val,
    results_base_test,
    results_smote_val,
    results_smote_test,
])
summary["training_set"] = ["Original", "Original", "SMOTE", "SMOTE"]
summary = summary[["training_set", "split", "roc_auc", "pr_auc"]]
summary.columns = ["Training Set", "Split", "ROC-AUC", "PR-AUC"]

print("=== Baseline Logistic Regression — Results Summary ===\n")
print(summary.to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Baseline LR: ROC-AUC & PR-AUC Comparison", fontsize=13, fontweight='bold')

for ax, metric in zip(axes, ["ROC-AUC", "PR-AUC"]):
    bar_data = summary.pivot(index="Split", columns="Training Set", values=metric)
    bar_data.plot(kind='bar', ax=ax, rot=0, edgecolor='black')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3, fontsize=9)
    ax.legend(title="Training Set")

plt.tight_layout()
save_plot(); plt.show()

### Threshold Analysis (Best Model)

At 0.5 the model is very conservative for the minority class. We sweep thresholds and
plot F1, Precision, and Recall vs threshold to find the operating point that maximises
F1 for the positive (default) class on the validation set.

In [ ]:
# we pick the model with the higher val ROC-AUC as the "best" baseline
best_model = lr_base if results_base_val["roc_auc"] >= results_smote_val["roc_auc"] else lr_smote
best_label = "class_weight='balanced'" if best_model is lr_base else "SMOTE"
print(f"Best baseline model: {best_label}")

y_prob_val = best_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 200)
metrics = []
for t in thresholds:
    y_pred_t = (y_prob_val >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_val == 1)).sum()
    fp = ((y_pred_t == 1) & (y_val == 0)).sum()
    fn = ((y_pred_t == 0) & (y_val == 1)).sum()
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    metrics.append({"threshold": t, "precision": prec, "recall": rec, "f1": f1})

metrics_df = pd.DataFrame(metrics)
best_t_row = metrics_df.loc[metrics_df["f1"].idxmax()]
best_threshold = best_t_row["threshold"]

print(f"\nBest F1 threshold on val set : {best_threshold:.3f}")
print(f"  Precision : {best_t_row['precision']:.4f}")
print(f"  Recall    : {best_t_row['recall']:.4f}")
print(f"  F1        : {best_t_row['f1']:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(metrics_df["threshold"], metrics_df["precision"], label="Precision", linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["recall"],    label="Recall",    linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["f1"],        label="F1 (positive class)", linewidth=2)
ax.axvline(best_threshold, color='red', linestyle='--', linewidth=1.2,
           label=f"Best F1 threshold = {best_threshold:.3f}")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis — {best_label} (Val Set)")
ax.legend()
plt.tight_layout()
save_plot(); plt.show()

# Evaluate best model at the optimal threshold on test set
print("\n--- Test set evaluation at optimal threshold ---")
_ = evaluate_model(best_model, X_test, y_test,
                   split_name=f"Test (threshold={best_threshold:.3f})",
                   threshold=best_threshold)

### Feature Importance, Top Coefficients

Logistic Regression coefficients directly reflect feature influence on the log-odds of default.
We plot the **top 20 positive** (increase default risk) and **top 20 negative** (decrease default risk) features.

> All features were standardised in preprocessing so coefficient magnitudes are comparable.

In [ ]:
coef_df = pd.DataFrame({
    "feature"    : feature_names,
    "coefficient": best_model.coef_[0],
}).sort_values("coefficient", ascending=False)

top_pos = coef_df.head(20)
top_neg = coef_df.tail(20).sort_values("coefficient")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(f"LR Baseline ({best_label}) — Top 20 Feature Coefficients",
             fontsize=13, fontweight='bold')

# Positive coefficients, increase default risk
axes[0].barh(top_pos["feature"], top_pos["coefficient"],
             color="salmon", edgecolor="black", linewidth=0.6)
axes[0].set_title("Top 20: Increase Default Risk", fontsize=11)
axes[0].set_xlabel("Coefficient (log-odds)")
axes[0].invert_yaxis()

# Negative coefficients, decrease default risk
axes[1].barh(top_neg["feature"], top_neg["coefficient"],
             color="steelblue", edgecolor="black", linewidth=0.6)
axes[1].set_title("Top 20: Decrease Default Risk", fontsize=11)
axes[1].set_xlabel("Coefficient (log-odds)")
axes[1].invert_yaxis()

plt.tight_layout()
save_plot(); plt.show()

print("\nFull coefficient table (sorted by magnitude):")
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df.sort_values("abs_coef", ascending=False).drop(columns="abs_coef").head(30)

### Summary & Takeaways

| Model | Training Set | Val ROC-AUC | Test ROC-AUC | Val PR-AUC | Test PR-AUC |
|---|---|---|---|---|---|
| LR Baseline | Original (class_weight=balanced) | *see results above* | ... | ... | ... |
| LR Baseline | SMOTE | *see results above* | ... | ... | ... |

**What to expect from this baseline.**
- Logistic Regression with balanced weights typically reaches **ROC-AUC ≈ 0.73, 0.75** on the Home Credit dataset.
- This establishes the floor, any model in the project must beat this AUC to justify its complexity.
- EXT_SOURCE_1/2/3 dominate the coefficient magnitudes, consistent with EDA findings.

**Next steps.**
- `04_random_forest.ipynb`, tree ensemble, no scaling needed, can capture nonlinear interactions
- `05_mlp.ipynb`, neural network baseline, same scaled features
- `06_elastic_net.ipynb`, regularised LR, automatic feature selection via L1+L2

### 4.2 PCA features

In [ ]:
# alias PCA matrices to the names the original notebook used
X_train, X_train_smote, X_val, X_test = X_train_pca, X_train_smote_pca, X_val_pca, X_test_pca
y_train_smote = y_train_smote_pca
feature_names = pca_feature_names
print(f"now using PCA features: {X_train.shape[1]} components")


### Train Model, Original Training Set (PCA)

Logistic Regression with `class_weight='balanced'`. Because PCA produces orthogonal features,
multicollinearity is eliminated, L2 regularisation with `C=1.0` is a fair starting point
(we can tune if needed). `lbfgs` converges faster on the smaller PCA feature set.

In [ ]:
lr_base = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    solver='lbfgs',
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1,
)

print("Training LR (class_weight='balanced') on PCA-reduced training set...")
lr_base.fit(X_train, y_train)
print(f"  Converged in {lr_base.n_iter_[0]} iterations.")
print(f"  Number of features (PCs): {X_train.shape[1]}")

print("done")


In [ ]:
# saving

joblib.dump(lr_base, os.path.join(OUTPUT_DIR, 'lr_base_pca.pkl'))
print("saved")


In [ ]:
results_base_val  = evaluate_model(lr_base, X_val,  y_val,  split_name="Val  (original train, PCA)")
results_base_test = evaluate_model(lr_base, X_test, y_test, split_name="Test (original train, PCA)")

plot_evaluation(lr_base, X_val, y_val, X_test, y_test,
                title_prefix="LR + PCA — class_weight='balanced'")

### Train Model, SMOTE Training Set (PCA)

SMOTE was applied in PCA space in notebook 02b, so synthetic samples are already in the reduced
orthogonal representation. No `class_weight` needed here since SMOTE already balanced the classes.

In [ ]:
lr_smote = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight=None,
    solver='lbfgs',
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1,
)

print("Training LR (SMOTE) on PCA-reduced training set...")
lr_smote.fit(X_train_smote, y_train_smote)
print(f"  Converged in {lr_smote.n_iter_[0]} iterations.")

results_smote_val  = evaluate_model(lr_smote, X_val,  y_val,  split_name="Val  (SMOTE train, PCA)")
results_smote_test = evaluate_model(lr_smote, X_test, y_test, split_name="Test (SMOTE train, PCA)")

plot_evaluation(lr_smote, X_val, y_val, X_test, y_test,
                title_prefix="LR + PCA — SMOTE")

print("done")


In [ ]:
# saving

joblib.dump(lr_smote, os.path.join(OUTPUT_DIR, 'lr_smote_pca.pkl'))
print("saved")


### Model Comparison Summary

In [ ]:
summary = pd.DataFrame([
    results_base_val,
    results_base_test,
    results_smote_val,
    results_smote_test,
])
summary["training_set"] = ["Original", "Original", "SMOTE", "SMOTE"]
summary = summary[["training_set", "split", "roc_auc", "pr_auc"]]
summary.columns = ["Training Set", "Split", "ROC-AUC", "PR-AUC"]

print("=== LR + PCA — Results Summary ===\n")
print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("LR + PCA: ROC-AUC & PR-AUC Comparison", fontsize=13, fontweight='bold')

for ax, metric in zip(axes, ["ROC-AUC", "PR-AUC"]):
    bar_data = summary.pivot(index="Split", columns="Training Set", values=metric)
    bar_data.plot(kind='bar', ax=ax, rot=0, edgecolor='black')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3, fontsize=9)
    ax.legend(title="Training Set")

plt.tight_layout()
save_plot(); plt.show()

### Threshold Analysis (Best Model)

In [ ]:
best_model = lr_base if results_base_val["roc_auc"] >= results_smote_val["roc_auc"] else lr_smote
best_label = "class_weight='balanced'" if best_model is lr_base else "SMOTE"
print(f"Best model (by val ROC-AUC): {best_label}")

y_prob_val = best_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 200)
metrics = []
for t in thresholds:
    y_pred_t = (y_prob_val >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_val == 1)).sum()
    fp = ((y_pred_t == 1) & (y_val == 0)).sum()
    fn = ((y_pred_t == 0) & (y_val == 1)).sum()
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    metrics.append({"threshold": t, "precision": prec, "recall": rec, "f1": f1})

metrics_df = pd.DataFrame(metrics)
best_t_row = metrics_df.loc[metrics_df["f1"].idxmax()]
best_threshold = best_t_row["threshold"]

print(f"\nBest F1 threshold on val set : {best_threshold:.3f}")
print(f"  Precision : {best_t_row['precision']:.4f}")
print(f"  Recall    : {best_t_row['recall']:.4f}")
print(f"  F1        : {best_t_row['f1']:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(metrics_df["threshold"], metrics_df["precision"], label="Precision", linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["recall"],    label="Recall",    linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["f1"],        label="F1 (positive class)", linewidth=2)
ax.axvline(best_threshold, color='red', linestyle='--', linewidth=1.2,
           label=f"Best F1 threshold = {best_threshold:.3f}")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis — LR + PCA ({best_label}, Val Set)")
ax.legend()
plt.tight_layout()
save_plot(); plt.show()

print("\n--- Test set evaluation at optimal threshold ---")
_ = evaluate_model(best_model, X_test, y_test,
                   split_name=f"Test (threshold={best_threshold:.3f})",
                   threshold=best_threshold)

### Feature Importance, Two Views

Since LR coefficients are now in **PC space**, we use two complementary views.

1. **PC-space coefficients**, which principal components the model weighted most, and in which direction.
2. **Back-projected original-feature importance**, weight each original feature by how much it contributes to the high-coefficient PCs. Formula, $w_j = \sum_k |\beta_k| \cdot |v_{kj}|$ where $\beta_k$ is the LR coefficient for PC $k$ and $v_{kj}$ is the loading of original feature $j$ on PC $k$.

In [ ]:
# --- 8.1 PC-space coefficients ---
coef_pc = pd.DataFrame({
    "PC"         : pca_feature_names,
    "coefficient": best_model.coef_[0],
    "variance_%" : pca.explained_variance_ratio_ * 100,
}).sort_values("coefficient", ascending=False)

n_pcs = len(pca_feature_names)
n_show = min(30, n_pcs)
top_half = n_show // 2
top_pos = coef_pc.head(top_half)
top_neg = coef_pc.tail(top_half).sort_values("coefficient")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"LR + PCA ({best_label}) — PC Coefficients",
             fontsize=13, fontweight='bold')

axes[0].barh(top_pos["PC"], top_pos["coefficient"],
             color="salmon", edgecolor="black", linewidth=0.5)
axes[0].set_title(f"Top {top_half}: Increase Default Risk (PC space)", fontsize=11)
axes[0].set_xlabel("Coefficient (log-odds)")
axes[0].invert_yaxis()

axes[1].barh(top_neg["PC"], top_neg["coefficient"],
             color="steelblue", edgecolor="black", linewidth=0.5)
axes[1].set_title(f"Top {top_half}: Decrease Default Risk (PC space)", fontsize=11)
axes[1].set_xlabel("Coefficient (log-odds)")
axes[1].invert_yaxis()

plt.tight_layout()
save_plot(); plt.show()

print("Top 10 PCs by absolute coefficient:")
coef_pc["abs_coef"] = coef_pc["coefficient"].abs()
print(coef_pc.sort_values("abs_coef", ascending=False)
             .drop(columns="abs_coef")
             .head(10)
             .to_string(index=False))

In [ ]:
# --- 8.2 Back-projection to original feature space ---
# Approximate original-feature importance:
#   importance_j = sum_k ( |beta_k| * |v_kj| )
# This sums the absolute LR weight of each PC, scaled by how much each
# original feature loads onto that PC.

abs_coefs   = np.abs(best_model.coef_[0])           # shape (n_pcs,)
abs_loadings = np.abs(pca.components_)              # shape (n_pcs, n_original_features)

# Weighted sum: (n_pcs,) @ (n_pcs, n_features) to (n_features,)
backprojected = abs_coefs @ abs_loadings

importance_df = pd.DataFrame({
    "feature"   : raw_feature_names,
    "importance": backprojected,
}).sort_values("importance", ascending=False)

N_TOP = 25
top_features = importance_df.head(N_TOP)

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top_features["feature"][::-1], top_features["importance"][::-1],
        color="steelblue", edgecolor="black", linewidth=0.5)
ax.set_xlabel("Back-projected importance  (Σ |β_k| · |v_kj|)", fontsize=10)
ax.set_title(f"Top {N_TOP} Original Features — Back-projected from PCA+LR ({best_label})",
             fontsize=11)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
save_plot(); plt.show()

print(f"\nTop {N_TOP} original features by back-projected importance:")
print(importance_df.head(N_TOP).to_string(index=False))

back-projecting the PC coefficients into original-feature space is the only reason notebook 5 existed as a separate notebook. the rest is the same training and threshold work as 4.1.

### Compare PCA Model vs Standard LR Baseline

Load the standard (non-PCA) baseline metrics saved by `lr_baseline_model.ipynb` and put
both side by side. This is the key comparison, did PCA hurt, help, or have no effect on AUC?

In [ ]:
try:
    baseline_summary = pd.read_csv(os.path.join(OUTPUT_DIR, 'lr_baseline_summary.csv'))
    baseline_summary.insert(0, "Model", "LR (no PCA)")

    pca_summary = summary.copy()
    pca_summary.insert(0, "Model", "LR + PCA")
    pca_summary.columns = baseline_summary.columns

    comparison = pd.concat([baseline_summary, pca_summary], ignore_index=True)
    print("=== LR Baseline vs LR + PCA ===\n")
    print(comparison.to_string(index=False))

    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("LR Baseline vs LR + PCA", fontsize=13, fontweight='bold')

    for ax, metric in zip(axes, ["ROC-AUC", "PR-AUC"]):
        pivot = comparison.pivot_table(
            index="Split", columns="Model", values=metric, aggfunc="mean"
        )
        pivot.plot(kind='bar', ax=ax, rot=15, edgecolor='black')
        ax.set_title(metric)
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
        for container in ax.containers:
            ax.bar_label(container, fmt="%.4f", padding=3, fontsize=8)
        ax.legend(title="Model")

    plt.tight_layout()
    save_plot(); plt.show()

except FileNotFoundError:
    print("Note: lr_baseline_summary.csv not found — run lr_baseline_model.ipynb first.")
    print("\nPCA model summary:")
    print(summary.to_string(index=False))

### Summary & Takeaways

| Model | Training Set | Val ROC-AUC | Test ROC-AUC | Val PR-AUC | Test PR-AUC |
|---|---|---|---|---|---|
| LR + PCA | Original (class_weight=balanced) | *see results above* | ... | ... | ... |
| LR + PCA | SMOTE | *see results above* | ... | ... | ... |

**Interpreting the PCA vs no-PCA comparison.**
- **AUC ≈ same or slightly lower.** PCA is lossless at 95% variance but the dropped 5% may contain
  discriminative signal. A small AUC dip (e.g. 0.005) is expected and acceptable.
- **AUC notably lower.** The discarded components contained structured signal, try 99% threshold.
- **AUC higher.** PCA regularised the model by removing noisy/collinear dimensions, this is the
  ideal outcome for Logistic Regression on correlated tabular data.

**Coefficient interpretation in PC space.**
- PC coefficients are not directly interpretable in original feature terms.
- Use the back-projected importance (Section 8.2) to identify which original features drive predictions.
- EXT_SOURCE_1/2/3 should still dominate, consistent with the non-PCA baseline.

**Next steps.**
- `04_random_forest.ipynb`, PCA less critical for trees but can still be compared
- `05_mlp.ipynb`, PCA often most beneficial here (reduces input width, speeds up training)
- `06_elastic_net.ipynb`, PCA + Elastic Net is interesting because EN already does feature selection

In [ ]:
# restore standard 113-feature dataset for sections 5..8
X_train          = joblib.load(os.path.join(CHECKPOINT_DIR, 'X_train.pkl'))
X_val            = joblib.load(os.path.join(CHECKPOINT_DIR, 'X_val.pkl'))
X_test           = joblib.load(os.path.join(CHECKPOINT_DIR, 'X_test.pkl'))
X_train_smote    = joblib.load(os.path.join(CHECKPOINT_DIR, 'X_train_smote.pkl'))
y_train_smote    = joblib.load(os.path.join(CHECKPOINT_DIR, 'y_train_smote.pkl'))
feature_names    = list(feature_names_raw)
print(f"restored standard features: {X_train.shape[1]}")


In [ ]:
# we save LR baseline test metrics for the cross-model comparison

lr_test_metrics = results_base_test.copy()
print(lr_test_metrics)


## **5. Modeling, Elastic Net Logistic Regression**

L1 plus L2 mix tuned via 5-fold gridsearch. the cross-model comparison against the LR baseline lives in section 10.

### Default Elastic Net (Out-of-the-Box)

Reference point before tuning, equal L1/L2 mix (`l1_ratio=0.5`), default `C=1.0`,
`class_weight='balanced'` to handle the ~8% default rate (mirrors the LR baseline).

**Solver.** `saga`, the only sklearn solver that supports the elasticnet penalty for
LogisticRegression. It also handles large datasets well thanks to its stochastic average
gradient acceleration.

In [ ]:
enet_default = LogisticRegression(
    penalty='elasticnet',
    l1_ratio=0.5,           # equal L1/L2 mix
    C=1.0,                  # default regularisation strength
    solver='saga',
    class_weight='balanced',
    max_iter=2000,
    random_state=SEED,
    n_jobs=-1,
)

t0 = time.time()
enet_default.fit(X_train, y_train)
default_train_time = time.time() - t0
print(f"Default Elastic Net trained in {default_train_time:.1f} s")
print(f"  Converged in {enet_default.n_iter_[0]} iterations.")

# Quick look at sparsity even at the default config
n_zero = (enet_default.coef_[0] == 0).sum()
print(f"  Zero coefficients: {n_zero} / {len(feature_names)} ({100*n_zero/len(feature_names):.1f}%)")

print("done")


In [ ]:
# saving

joblib.dump(enet_default, os.path.join(OUTPUT_DIR, 'enet_default.pkl'))
print("saved")


In [ ]:
results_default_val  = evaluate_model(enet_default, X_val,  y_val,  split_name="Val  (default ENet)")
results_default_test = evaluate_model(enet_default, X_test, y_test, split_name="Test (default ENet)")

plot_evaluation(enet_default, X_val, y_val, X_test, y_test,
                title_prefix="Default Elastic Net (l1_ratio=0.5, C=1.0)")

### Hyperparameter Tuning, Grid Search with 5-Fold CV

We sweep two hyperparameters that together control the regularisation.

| Parameter | Range | What it controls |
|---|---|---|
| `C` | 0.01 to 10 (log scale) | **Inverse** regularisation strength. Small `C` = strong penalty = more shrinkage; large `C` = weak penalty = closer to unregularised LR |
| `l1_ratio` | 0.1, 0.3, 0.5, 0.7, 0.9 | Mix between L2 (0) and L1 (1). Higher values = more sparsity |

- **Grid size.** 4 × 5 = 20 combinations × 5 folds = **100 fits**.
- **CV folds.** 5, stratified on `TARGET` to preserve the class ratio.
- **Scoring.** ROC-AUC (consistent across all four models in this project).
- We use **grid search** rather than random search because the grid is small enough to
  enumerate fully, and a complete grid lets us plot the regularisation landscape
  (`C` × `l1_ratio` heatmap) which is informative for the report's RQ1.

> warning, This cell is the slowest in the notebook (~360 min on Colab CPU). Set
> `max_iter=500` during search for speed, then refit the winner with `max_iter=2000`.
> If you re-run, skip this cell and load the saved best estimator instead.

In [ ]:
# --- Hyperparameter grid ---
param_grid = {
    'C'         : [0.01, 0.1, 1.0, 10.0],
    'l1_ratio'  : [0.1, 0.3, 0.5, 0.7, 0.9],
}

# Base model, class_weight handles imbalance, saga is the only elasticnet solver
enet_base = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=500,             # capped during search; final fit uses 2000
    random_state=SEED,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid_search = GridSearchCV(
    estimator=enet_base,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

t0 = time.time()
grid_search.fit(X_train, y_train)
search_time = time.time() - t0
print(f"\n=== Grid search complete in {search_time/60:.1f} minutes ===")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")
print(f"Best params:")
for k, v in grid_search.best_params_.items():
    print(f"  {k:10s}: {v}")

print("done")


In [ ]:
# saving

joblib.dump(grid_search, os.path.join(OUTPUT_DIR, 'enet_grid_search.pkl'))
print("saved")


In [ ]:
# --- Inspect all configurations ---
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_summary = cv_results[[
    'mean_test_score', 'std_test_score', 'mean_train_score',
    'param_C', 'param_l1_ratio', 'mean_fit_time'
]].sort_values('mean_test_score', ascending=False).reset_index(drop=True)

cv_summary.columns = [
    'CV ROC-AUC', 'Std', 'Train ROC-AUC',
    'C', 'l1_ratio', 'Fit time (s)'
]
print("All 20 configurations (sorted by CV ROC-AUC):")
cv_summary.round(4)

### 5.1 Visualise the Regularisation Landscape

Two complementary plots.
1. **Heatmap** of CV ROC-AUC across the full `C` × `l1_ratio` grid, shows where the sweet spot lives.
2. **Train vs CV gap** per config, the overfitting diagnostic for RQ1.

In [ ]:
# --- Heatmap of CV ROC-AUC across the grid ---
heatmap_data = cv_results.pivot_table(
    index='param_l1_ratio', columns='param_C', values='mean_test_score'
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: CV ROC-AUC heatmap
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='YlGnBu',
            cbar_kws={'label': 'CV ROC-AUC'}, ax=axes[0])
axes[0].set_title("CV ROC-AUC across (C, l1_ratio) grid", fontsize=12)
axes[0].set_xlabel("C (inverse regularisation strength)")
axes[0].set_ylabel("l1_ratio")
axes[0].invert_yaxis()

# Panel 2: Train vs CV gap (overfitting check)
x_pos = np.arange(len(cv_summary))
axes[1].errorbar(x_pos, cv_summary['CV ROC-AUC'], yerr=cv_summary['Std'],
                 fmt='o', capsize=4, color='steelblue', label='CV (val) ROC-AUC')
axes[1].plot(x_pos, cv_summary['Train ROC-AUC'], 's', color='salmon',
             label='Train ROC-AUC', alpha=0.7)
axes[1].set_xlabel("Configuration rank (by CV ROC-AUC)")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_title("Train vs CV ROC-AUC across configs")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
save_plot(); plt.show()

# Quick numeric summary of the gap
gap = cv_summary['Train ROC-AUC'] - cv_summary['CV ROC-AUC']
print(f"Mean train-val gap across configs : {gap.mean():.4f}")
print(f"Best config train-val gap         : {gap.iloc[0]:.4f}")

### Takeaway, regularisation effect (feeds RQ1)

The CV-variance plot and heatmap show something striking, **all 20 configurations reach essentially the same CV ROC-AUC** (range 0.7482, 0.7485, Δ = 0.0003). The differences are well within the CV standard deviation (~0.0035). Three things follow.

- **The train, val gap is 0.0023 across all configs**, an order of magnitude smaller than the Random Forest's gap of 0.088. Linear models on this dataset have very low variance to begin with, so adding more regularisation simply slides along an already-flat generalisation curve.
- **The (C, l1_ratio) heatmap is flat.** `C` matters a tiny bit (smallest `C=0.01` is 0.0002 worse than `C=0.1`), `l1_ratio` matters not at all for AUC. This is the diagnostic signature of a well-conditioned problem with a small number of dominant features (the `EXT_SOURCE_*` block), once those features are in, the model's accuracy is locked in regardless of how the long tail is regularised.
- **GridSearchCV picked C=0.1, l1_ratio=0.5** essentially by tiebreaker. This config converged in 927 iterations, while the default `C=1.0` config hit the 2000-iteration cap without converging, so stronger regularisation also made optimisation easier.

For RQ1, regularisation here is a *parsimony* lever, not a *generalisation* lever, and that's the more interesting story. We see this play out concretely in the sparsity analysis below.

the heatmap shows thatt the AUC is mostly flat across the grid, the regularisation strength matters far more than the L1 vs L2 mix at this scale.

### Retrain Best Configuration & Evaluate

`GridSearchCV` already refits on the full training set (`refit=True` by default), but
we re-train explicitly with `max_iter=2000` to ensure full convergence and to time
the final fit separately for the complexity-analysis table.

In [ ]:
# Refit the best config with the full max_iter budget for clean convergence
enet_best_params = grid_search.best_params_

enet_best = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    C=enet_best_params['C'],
    l1_ratio=enet_best_params['l1_ratio'],
    max_iter=2000,
    random_state=SEED,
    n_jobs=-1,
)

t0 = time.time()
enet_best.fit(X_train, y_train)
best_train_time = time.time() - t0
print(f"Best Elastic Net retrained in {best_train_time:.1f} s")
print(f"  Converged in {enet_best.n_iter_[0]} iterations.")
print(f"  Best params: {enet_best_params}")

print("done")


In [ ]:
# saving

joblib.dump(enet_best, os.path.join(OUTPUT_DIR, 'enet_best.pkl'))
print("saved")


In [ ]:
results_best_val  = evaluate_model(enet_best, X_val,  y_val,  split_name="Val  (tuned ENet)")
results_best_test = evaluate_model(enet_best, X_test, y_test, split_name="Test (tuned ENet)")

plot_evaluation(enet_best, X_val, y_val, X_test, y_test,
                title_prefix="Tuned Elastic Net")

### SMOTE Comparison

Same best hyperparameters, trained on the SMOTE-resampled set. Mirrors the LR baseline
and RF notebooks so the final consolidated comparison can show both imbalance-handling
strategies on identical splits.

In [ ]:
enet_smote = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight=None,        # SMOTE replaces class_weight as the imbalance handler
    C=enet_best_params['C'],
    l1_ratio=enet_best_params['l1_ratio'],
    max_iter=2000,
    random_state=SEED,
    n_jobs=-1,
)

t0 = time.time()
enet_smote.fit(X_train_smote, y_train_smote)
print(f'fitted on {len(X_train_smote)} samples')  # sanity check, len here, shape[0] elsewhere is a habit
smote_train_time = time.time() - t0
print(f"Elastic Net (SMOTE) trained in {smote_train_time:.1f} s")
print(f"  Converged in {enet_smote.n_iter_[0]} iterations.")

results_smote_val  = evaluate_model(enet_smote, X_val,  y_val,  split_name="Val  (SMOTE ENet)")
results_smote_test = evaluate_model(enet_smote, X_test, y_test, split_name="Test (SMOTE ENet)")

plot_evaluation(enet_smote, X_val, y_val, X_test, y_test,
                title_prefix="Elastic Net — SMOTE")

print("done")


In [ ]:
# saving

joblib.dump(enet_smote, os.path.join(OUTPUT_DIR, 'enet_smote.pkl'))
print("saved")


In [ ]:
# --- Pick the better of the two for downstream coefficient analysis ---
summary = pd.DataFrame([
    results_default_val,  results_default_test,
    results_best_val,     results_best_test,
    results_smote_val,    results_smote_test,
])
summary["model"] = [
    "Default ENet",         "Default ENet",
    "Tuned ENet (balanced)", "Tuned ENet (balanced)",
    "Tuned ENet (SMOTE)",    "Tuned ENet (SMOTE)",
]
summary = summary[["model", "split", "roc_auc", "pr_auc"]]
summary.columns = ["Model", "Split", "ROC-AUC", "PR-AUC"]

print("=== Elastic Net — Results Summary ===\n")
print(summary.to_string(index=False))

# Pick the best by val ROC-AUC for coefficient analysis
best_model_name = (
    summary[summary["Split"].str.startswith("Val")]
    .sort_values("ROC-AUC", ascending=False)
    .iloc[0]["Model"]
)
best_model = {
    "Default ENet":          enet_default,
    "Tuned ENet (balanced)": enet_best,
    "Tuned ENet (SMOTE)":    enet_smote,
}[best_model_name]
print(f"\n>>> Best model selected: {best_model_name}")

In [ ]:
# --- Bar-chart comparison (same style as the LR baseline / RF notebooks) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Elastic Net: ROC-AUC & PR-AUC Comparison", fontsize=13, fontweight='bold')

for ax, metric in zip(axes, ["ROC-AUC", "PR-AUC"]):
    bar_data = summary.pivot(index="Split", columns="Model", values=metric)
    bar_data.plot(kind='bar', ax=ax, rot=0, edgecolor='black')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3, fontsize=8)
    ax.legend(title="Model", fontsize=8)

plt.tight_layout()
save_plot(); plt.show()

### Threshold Analysis

Sweep classification thresholds on the val set to find the operating point that maximises
F1 for the positive (default) class, same procedure as the LR baseline and RF.

In [ ]:
y_prob_val = best_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 200)
metrics = []
for t in thresholds:
    y_pred_t = (y_prob_val >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_val == 1)).sum()
    fp = ((y_pred_t == 1) & (y_val == 0)).sum()
    fn = ((y_pred_t == 0) & (y_val == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    metrics.append({"threshold": t, "precision": prec, "recall": rec, "f1": f1})

metrics_df = pd.DataFrame(metrics)
best_t_row = metrics_df.loc[metrics_df["f1"].idxmax()]
best_threshold = best_t_row["threshold"]

print(f"Best F1 threshold on val set : {best_threshold:.3f}")
print(f"  Precision : {best_t_row['precision']:.4f}")
print(f"  Recall    : {best_t_row['recall']:.4f}")
print(f"  F1        : {best_t_row['f1']:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(metrics_df["threshold"], metrics_df["precision"], label="Precision", linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["recall"],    label="Recall",    linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["f1"],        label="F1 (positive class)", linewidth=2)
ax.axvline(best_threshold, color='red', linestyle='--', linewidth=1.2,
           label=f"Best F1 threshold = {best_threshold:.3f}")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis — {best_model_name} (Val Set)")
ax.legend()
plt.tight_layout()
save_plot(); plt.show()

### Coefficient Analysis, Top 10 + Sparsity

Two things we care about for Elastic Net specifically.
1. **Top 10 positive / top 10 negative coefficients**, features that most
   increase / decrease default risk, surviving regularisation.
2. **Sparsity count**, how many coefficients did L1 shrink to *exactly* zero?
   This is the headline regularisation effect of Elastic Net.

> All features were standardised in preprocessing, so coefficient magnitudes
> are directly comparable.

In [ ]:
# --- Build coefficient table ---
enet_coef_df = pd.DataFrame({
    "feature"    : feature_names,
    "coefficient": best_model.coef_[0],
})
enet_coef_df["abs_coef"] = enet_coef_df["coefficient"].abs()

# --- Sparsity count (L1 effect) ---
n_zero = (enet_coef_df["coefficient"] == 0).sum()
n_total = len(enet_coef_df)
n_nonzero = n_total - n_zero
sparsity_pct = 100 * n_zero / n_total

print(f"=== Sparsity (L1 effect) ===")
print(f"Total features        : {n_total}")
print(f"Zero coefficients     : {n_zero} ({sparsity_pct:.1f}%)")
print(f"Non-zero coefficients : {n_nonzero} ({100-sparsity_pct:.1f}%)")
print(f"\nBest l1_ratio used    : {enet_best_params['l1_ratio']}")
print(f"Best C used           : {enet_best_params['C']}")

In [ ]:
# --- Top 10 positive (increase default risk) and top 10 negative (decrease default risk) ---
sorted_coefs = enet_coef_df.sort_values("coefficient", ascending=False)
top10_pos = sorted_coefs.head(10)
top10_neg = sorted_coefs.tail(10).sort_values("coefficient")

print("=== Top 10 features INCREASING default risk ===")
print(top10_pos[["feature", "coefficient"]].to_string(index=False))
print("\n=== Top 10 features DECREASING default risk ===")
print(top10_neg[["feature", "coefficient"]].to_string(index=False))

# --- Bar chart: top 10 positive vs top 10 negative ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(f"Tuned Elastic Net (l1_ratio={enet_best_params['l1_ratio']}, "
             f"C={enet_best_params['C']}) — Top 10 Coefficients",
             fontsize=13, fontweight='bold')

# Positive
axes[0].barh(top10_pos["feature"], top10_pos["coefficient"],
             color="salmon", edgecolor="black", linewidth=0.6)
axes[0].set_title("Top 10: Increase Default Risk", fontsize=11)
axes[0].set_xlabel("Coefficient (log-odds)")
axes[0].invert_yaxis()

# Negative
axes[1].barh(top10_neg["feature"], top10_neg["coefficient"],
             color="steelblue", edgecolor="black", linewidth=0.6)
axes[1].set_title("Top 10: Decrease Default Risk", fontsize=11)
axes[1].set_xlabel("Coefficient (log-odds)")
axes[1].invert_yaxis()

plt.tight_layout()
save_plot(); plt.show()

### 9.1 Sparsity Across the Full Grid

How does sparsity change with `l1_ratio` and `C`? We refit each grid point on the full
training set and count zero coefficients, this is the canonical regularisation-vs-sparsity
plot for the report's RQ1 section.

> We use `max_iter=300` here purely to keep runtime down; the *zero count* (sparsity)
> stabilises quickly even before full convergence.

In [ ]:
# --- Sparsity heatmap: refit each grid point on full train, count zeros ---
sparsity_grid = []
print("Computing sparsity for each (C, l1_ratio) combination...")
for C in param_grid['C']:
    for l1r in param_grid['l1_ratio']:
        m = LogisticRegression(
            penalty='elasticnet', solver='saga', class_weight='balanced',
            C=C, l1_ratio=l1r, max_iter=300, random_state=SEED, n_jobs=-1,
        )
        m.fit(X_train, y_train)
        n_z = (m.coef_[0] == 0).sum()
        sparsity_grid.append({
            'C': C, 'l1_ratio': l1r,
            'n_zero': n_z,
            'pct_zero': 100 * n_z / len(feature_names),
        })

sparsity_df = pd.DataFrame(sparsity_grid)
sparsity_pivot = sparsity_df.pivot(index='l1_ratio', columns='C', values='pct_zero')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(sparsity_pivot, annot=True, fmt='.1f', cmap='Reds',
            cbar_kws={'label': '% coefficients = 0'}, ax=ax)
ax.set_title("Coefficient Sparsity (% zero) across (C, l1_ratio) grid",
             fontsize=12, fontweight='bold')
ax.set_xlabel("C (inverse regularisation strength)")
ax.set_ylabel("l1_ratio")
ax.invert_yaxis()
plt.tight_layout()
save_plot(); plt.show()

print("\nSparsity table:")
print(sparsity_pivot.round(1))

### Takeaway, sparsity story (feeds RQ1)

The sparsity heatmap is the mirror image of the AUC heatmap from Section 5, completely opposite shape, same grid, two very different stories.

- **Sparsity grows monotonically with `l1_ratio` and inversely with `C`.** At the strong-regularisation corner (C=0.01, l1_ratio=0.9) the model zeros out **34.5%** of the 113 coefficients. At the weak-regularisation corner (C=10, any `l1_ratio`) it zeros out **0%**, identical to unregularised LR.
- **The chosen "best" config (C=0.1, l1_ratio=0.5) sits in the middle of this gradient, 7/113 (6.2%) zeros.** Pushing to the strong-regularisation corner would zero out ~39 features at a CV ROC-AUC cost of only 0.0003, basically free.
- **Geometric intuition (for the oral exam).** the L1 penalty's diamond-shaped constraint region has corners that lie on the coordinate axes, so the optimum often sits at a corner where one or more coefficients are exactly zero. The L2 penalty has a circular region, the optimum is rarely on an axis, so no exact zeros. `l1_ratio` slides between these two regimes.

**For the report.** Elastic Net gives us *automatic feature selection*, at the strong-regularisation corner we can drop roughly one-third of the features at no measurable accuracy cost. This is a direct, quantitative regularisation-vs-complexity trade-off for RQ1, and a nice contrast with the Random Forest, where regularisation actively reduced an existing 0.09 train, val gap.

### Summary & Takeaways

### Headline numbers

| Item | Value |
|---|---|
| Best configuration | `C=0.1, l1_ratio=0.5` (essentially a tiebreaker, see below) |
| Best CV ROC-AUC (5-fold) | **0.7485** |
| Tuned ENet, Val ROC-AUC / Test ROC-AUC | **0.7496 / 0.7523** |
| Tuned ENet, Val PR-AUC / Test PR-AUC | 0.2257 / 0.2350 |
| Default ENet, Test ROC-AUC | 0.7522 (essentially identical to tuned) |
| SMOTE ENet, Test ROC-AUC | 0.7454 (worse than balanced) |
| Optimal F1 threshold (Val) | 0.670 (P=0.236, R=0.396, F1=0.296) |
| Coefficients zeroed by L1 | 7 / 113 (6.2%) |
| Default ENet training time | 1,579 s (~26 min, hit 2000-iter cap, did not converge) |
| Tuned ENet training time | 764 s (~13 min, converged in 927 iter) |
| Grid search time (20 configs × 5 folds) | **412 min (~6.9 h)** |

### Key findings

**1. The (C, l1_ratio) grid is essentially flat for AUC (RQ1).**
All 20 configurations reach CV ROC-AUC ∈ [0.7482, 0.7485]. The CV-fold standard deviation (~0.0035) is more than 10× the spread across configs (0.0003). Linear models on this dataset are already at their generalisation ceiling regardless of regularisation strength, the train, val gap is just 0.0023 across the board. **Regularisation here is a parsimony lever, not a generalisation lever.**

**2. Sparsity ranges from 0% to 34.5% across the same grid.**
The same grid that produced flat AUC produces a 5× spread in % of zero coefficients. At the strong-regularisation corner (C=0.01, l1_ratio=0.9), Elastic Net zeros out 39 of 113 features at zero AUC cost (Δ = 0.0003 from the best config). The chosen best config (C=0.1, l1_ratio=0.5) sits in the middle at 7 zeros (6.2%). This is the central RQ1 finding, **on this dataset, you can get a one-third smaller model for free.**

**3. Elastic Net agrees with LR baseline on which features matter (RQ2).**
Spearman ρ = 0.917 between Elastic Net and unregularised LR coefficients. Top-10 overlap = 8/10. The two linear models extract the same signal; Elastic Net just shrinks the long tail and pushes a handful of marginal features below the cut. Notable re-rankings, `NAME_INCOME_TYPE_Pensioner` (LR rank 7 to ENet rank 0), `CODE_GENDER_M` (12 to 8), `NAME_INCOME_TYPE_Unemployed` (13 to 5).

**4. Comparison with Random Forest (Phase 5 setup).**
The RF reached Test ROC-AUC ≈ 0.7518 with Spearman ρ = 0.13, 0.15 vs LR. Elastic Net reaches Test ROC-AUC = 0.7523 with Spearman ρ = 0.92 vs LR. **All three classical models are statistically tied on AUC, but the linear-vs-tree feature-importance gap is dramatic, while the regularised-vs-unregularised linear gap is negligible.** This will be the backbone of the model-comparison narrative in Phase 5.

**5. SMOTE underperformed `class_weight='balanced'` (consistent with the RF).**
SMOTE Test ROC-AUC = 0.7454 vs balanced = 0.7523, a 0.007 drop. Same direction and similar magnitude as the RF saw (0.752 to 0.718). Synthetic minority samples generated by k-NN interpolation appear to add noise rather than signal in this high-dimensional one-hot-encoded space, worth flagging in the Discussion section.

**6. Stronger regularisation made optimisation easier (small bonus finding).**
The default config (C=1.0) hit the `max_iter=2000` cap without converging. The chosen best config (C=0.1) converged in 927 iterations. Stronger L1+L2 penalty smooths the loss surface and accelerates `saga` convergence, a connection between regularisation theory and numerical optimisation worth mentioning at the oral.

### Implications for next phases

- **For the MLP.** the bar to clear is Test ROC-AUC ≈ 0.752. If the MLP can't beat well-regularised linear models, that's itself a finding (NN data-hunger on tabular data dominated by a small number of strong features, here the `EXT_SOURCE_*` block).
- **For the fairness analysis (Phase 6).** Elastic Net's parsimonious 106-active-feature model is the strongest candidate for the "EU AI Act high-risk" explainability discussion. With ρ = 0.92 vs unregularised LR, explanations transfer cleanly between the two.
- **For the complexity table (Phase 5).** Elastic Net training (~764 s) is ~one to two orders of magnitude slower than LR baseline (lbfgs solver, a few seconds), the cost of `saga` supporting the elasticnet penalty. The 6.9-hour grid-search cost is a real consideration for retraining cadence in production but a one-off engineering cost.

### Limitations

- **Grid was probably too tight in the regularisation direction.** All four `C` values landed on the flat plateau. To find where AUC actually starts to drop we would need `C` ∈ {1e-4, 1e-3} as well. Future-work item.
- **Default config did not converge** at `max_iter=2000` (saga solver is sensitive to large `C`). All conclusions above are based on the converged tuned config; this only affected the unused default reference point.

### Files saved to `MODEL_DIR`
`enet_best.pkl`, `enet_test_preds.csv`, `enet_summary.csv`, `enet_coefficients.csv`, `enet_sparsity_grid.csv`, `enet_cv_results.csv`, `enet_best_info.json`.

In [ ]:
# we save Elastic Net test metrics for the cross-model comparison

enet_test_metrics = results_best_test.copy()
print(enet_test_metrics)


## **6. Modeling, Random Forest**

ensemble of 500 decision trees with bootstrap sampling. RandomizedSearchCV explores 15 configs. the cross-model comparison against the LR baseline lives in section 10.

### Default Random Forest (Out-of-the-Box)

We start with a default `RandomForestClassifier` to establish a reference point before tuning.
We add `class_weight='balanced'` to compensate for the ~8% default rate (mirrors the LR baseline
strategy so the comparison is apples-to-apples).

> Default = `n_estimators=100`, no `max_depth` cap (trees grown fully), `min_samples_split=2`.

In [ ]:
rf_default = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
)

t0 = time.time()
rf_default.fit(X_train, y_train)
default_train_time = time.time() - t0
print(f"Default RF trained in {default_train_time:.1f} s")
print(f"Number of trees: {rf_default.n_estimators}")
print(f"Mean tree depth: {np.mean([t.get_depth() for t in rf_default.estimators_]):.1f}")

print("done")


In [ ]:
# saving

joblib.dump(rf_default, os.path.join(OUTPUT_DIR, 'rf_default.pkl'))
print("saved")


In [ ]:
results_default_val  = evaluate_model(rf_default, X_val,  y_val,  split_name="Val  (default RF)")
results_default_test = evaluate_model(rf_default, X_test, y_test, split_name="Test (default RF)")

plot_evaluation(rf_default, X_val, y_val, X_test, y_test,
                title_prefix="Default Random Forest")

random forest defaults are sklearn's choice, not ours. the random search below explores depth and leaf size since those are the two knobs that change overfitting the most.

### Hyperparameter Tuning, Randomised Search with 5-Fold CV

We tune four hyperparameters with `RandomizedSearchCV`.

| Parameter | Range | What it controls |
|---|---|---|
| `n_estimators` | 100, 500 | Number of trees in the forest (more = better but slower) |
| `max_depth` | 5, 30 (or unlimited) | Per-tree depth (caps overfitting) |
| `max_leaf_nodes` | None / 50, 500 | Hard cap on leaves per tree (alternative complexity control) |
| `min_samples_split` | 2, 20 | Minimum samples needed to split a node (regularises) |

- **CV folds.** 5, stratified on `TARGET` to preserve class ratio.
- **Scoring.** ROC-AUC (consistent across all four models in this project).
- **`n_iter=15`.** 15 random configurations × 5 folds = 75 fits. Random search is much more
  efficient than grid search when only a few parameters strongly matter, see Bergstra & Bengio (2012).
- **`class_weight='balanced'`** kept fixed (handled at the model level, not in the search).

> warning, This cell takes the longest to run (~120 min on Colab CPU; faster with `n_jobs=-1`).
> If you re-run the notebook, skip this cell and load the saved best estimator instead.

In [ ]:
# random search over teh hyperparameter grid
# --- Hyperparameter grid for random sampling ---
param_dist = {
    'n_estimators'      : [100, 200, 300, 400, 500],
    'max_depth'         : [5, 10, 15, 20, 25, 30, None],
    'max_leaf_nodes'    : [None, 50, 100, 200, 500],
    'min_samples_split' : [2, 5, 10, 20],
}

# Base model, class_weight handles imbalance, n_jobs=-1 parallelises tree fitting
rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=15,                # 15 random configs
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,                # parallelise across folds/configs
    verbose=2,
    random_state=SEED,
    return_train_score=True,
)

t0 = time.time()
random_search.fit(X_train, y_train)
search_time = time.time() - t0
print(f"\n=== Random search complete in {search_time/60:.1f} minutes ===")
print(f"Best CV ROC-AUC: {random_search.best_score_:.4f}")
print(f"Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k:20s}: {v}")

print("done")


In [ ]:
# saving

joblib.dump(random_search, os.path.join(OUTPUT_DIR, 'rf_random_search.pkl'))
print("saved")


In [ ]:
# --- Inspect top configurations ---
cv_results = pd.DataFrame(random_search.cv_results_)
cv_summary = cv_results[[
    'mean_test_score', 'std_test_score', 'mean_train_score',
    'param_n_estimators', 'param_max_depth', 'param_max_leaf_nodes',
    'param_min_samples_split', 'mean_fit_time'
]].sort_values('mean_test_score', ascending=False).reset_index(drop=True)

cv_summary.columns = [
    'CV ROC-AUC', 'Std', 'Train ROC-AUC',
    'n_estimators', 'max_depth', 'max_leaf_nodes',
    'min_samples_split', 'Fit time (s)'
]
print("Top 10 configurations:")
cv_summary.head(10).round(4)

In [ ]:
# --- Visualise CV variance across configs (overfitting check) ---
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(cv_summary))
ax.errorbar(x_pos, cv_summary['CV ROC-AUC'], yerr=cv_summary['Std'],
            fmt='o', capsize=4, color='steelblue', label='CV (val) ROC-AUC')
ax.plot(x_pos, cv_summary['Train ROC-AUC'], 's', color='salmon',
        label='Train ROC-AUC', alpha=0.7)
ax.set_xlabel("Configuration rank (by CV ROC-AUC)")
ax.set_ylabel("ROC-AUC")
ax.set_title("Random Forest — CV ROC-AUC across 15 sampled configs")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_plot(); plt.show()

# Quick sanity check: train-val gap = overfitting indicator
gap = cv_summary['Train ROC-AUC'] - cv_summary['CV ROC-AUC']
print(f"Mean train-val gap across configs: {gap.mean():.4f}")
print(f"Best config train-val gap        : {gap.iloc[0]:.4f}")

### Takeaway, regularisation effect (feeds RQ1)

The CV-variance plot and gap statistics show.

- **Train ROC-AUC ≈ 0.836 vs CV ROC-AUC ≈ 0.748** for the best config to train, val gap of **0.088**.
- The unregularised default RF (no `max_depth` cap, `min_samples_split=2`) had Val ROC-AUC of only 0.731 despite higher capacity, classic overfitting symptom.
- The winning config combines **three regularisation mechanisms simultaneously**, `max_depth=30` (depth cap), `max_leaf_nodes=500` (leaf cap), `min_samples_split=20` (minimum samples per split). Configs that relied on only one (e.g. config rank 6, `max_depth=15` with `max_leaf_nodes=None`) showed train ROC-AUC up to 0.94 and overfit harder.

This is the RF analogue of the regularisation story we'll see again in the MLP (dropout + batch norm + early stopping). For the report's RQ1 section we can frame all three models as "more regularisation, smaller train, val gap, better generalisation."

### Retrain Best Configuration & Evaluate

`RandomizedSearchCV` already refits on the full training set with the best params (`refit=True`
by default), so `random_search.best_estimator_` is ready to use. We record the wall-clock
time of this final fit separately for the report's complexity-analysis section.

In [ ]:
# Refit the best config on the full training set (timed for the complexity table)
rf_best_params = random_search.best_params_

rf_best = RandomForestClassifier(
    **rf_best_params,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
)

t0 = time.time()
rf_best.fit(X_train, y_train)
best_train_time = time.time() - t0
print(f"Best RF retrained in {best_train_time:.1f} s")
print(f"Best params: {rf_best_params}")

print("done")


In [ ]:
# saving

joblib.dump(rf_best, os.path.join(OUTPUT_DIR, 'rf_best.pkl'))
print("saved")


In [ ]:
results_best_val  = evaluate_model(rf_best, X_val,  y_val,  split_name="Val  (tuned RF)")
results_best_test = evaluate_model(rf_best, X_test, y_test, split_name="Test (tuned RF)")

plot_evaluation(rf_best, X_val, y_val, X_test, y_test,
                title_prefix="Tuned Random Forest")

### SMOTE Comparison

Same best hyperparameters, trained on the SMOTE-resampled training set instead of
`class_weight='balanced'`. This mirrors the LR baseline so the final model-comparison
table can compare both imbalance-handling strategies on identical splits.

In [ ]:
rf_smote = RandomForestClassifier(
    **rf_best_params,
    class_weight=None,         # SMOTE replaces class_weight as the imbalance handler
    random_state=SEED,
    n_jobs=-1,
)

t0 = time.time()
rf_smote.fit(X_train_smote, y_train_smote)
smote_train_time = time.time() - t0
print(f"RF (SMOTE) trained in {smote_train_time:.1f} s")

results_smote_val  = evaluate_model(rf_smote, X_val,  y_val,  split_name="Val  (SMOTE RF)")
results_smote_test = evaluate_model(rf_smote, X_test, y_test, split_name="Test (SMOTE RF)")

plot_evaluation(rf_smote, X_val, y_val, X_test, y_test,
                title_prefix="Random Forest — SMOTE")

print("done")


In [ ]:
# saving

joblib.dump(rf_smote, os.path.join(OUTPUT_DIR, 'rf_smote.pkl'))
print("saved")


In [ ]:
# --- Pick the better of the two for downstream analysis ---
summary = pd.DataFrame([
    results_default_val,  results_default_test,
    results_best_val,     results_best_test,
    results_smote_val,    results_smote_test,
])
summary["model"] = [
    "Default RF", "Default RF",
    "Tuned RF (balanced)", "Tuned RF (balanced)",
    "Tuned RF (SMOTE)", "Tuned RF (SMOTE)",
]
summary = summary[["model", "split", "roc_auc", "pr_auc"]]
summary.columns = ["Model", "Split", "ROC-AUC", "PR-AUC"]

print("=== Random Forest — Results Summary ===\n")
print(summary.to_string(index=False))

# Pick the best by val ROC-AUC for feature-importance + threshold analysis
best_model_name = (
    summary[summary["Split"].str.startswith("Val")]
    .sort_values("ROC-AUC", ascending=False)
    .iloc[0]["Model"]
)
best_model = {
    "Default RF":          rf_default,
    "Tuned RF (balanced)": rf_best,
    "Tuned RF (SMOTE)":    rf_smote,
}[best_model_name]
print(f"\n>>> Best model selected: {best_model_name}")

the SMOTE training brings recall up but precision drops by the same amount, the AUC is unchanged. consistent with what we saw on the LR baseline.

In [ ]:
# --- Bar-chart comparison (same style as LR baseline) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Random Forest — ROC-AUC & PR-AUC Comparison", fontsize=13, fontweight='bold')

for ax, metric in zip(axes, ["ROC-AUC", "PR-AUC"]):
    bar_data = summary.pivot(index="Split", columns="Model", values=metric)
    bar_data.plot(kind='bar', ax=ax, rot=0, edgecolor='black')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3, fontsize=8)
    ax.legend(title="Model", fontsize=8)

plt.tight_layout()
save_plot(); plt.show()

### Threshold Analysis

Sweep classification thresholds on the val set to find the operating point that maximises
F1 for the positive (default) class. Same procedure as the LR baseline so the comparison
is consistent.

In [ ]:
y_prob_val = best_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 200)
metrics = []
for t in thresholds:
    y_pred_t = (y_prob_val >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_val == 1)).sum()
    fp = ((y_pred_t == 1) & (y_val == 0)).sum()
    fn = ((y_pred_t == 0) & (y_val == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    metrics.append({"threshold": t, "precision": prec, "recall": rec, "f1": f1})

metrics_df = pd.DataFrame(metrics)
best_t_row = metrics_df.loc[metrics_df["f1"].idxmax()]
best_threshold = best_t_row["threshold"]

print(f"Best F1 threshold on val set : {best_threshold:.3f}")
print(f"  Precision : {best_t_row['precision']:.4f}")
print(f"  Recall    : {best_t_row['recall']:.4f}")
print(f"  F1        : {best_t_row['f1']:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(metrics_df["threshold"], metrics_df["precision"], label="Precision", linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["recall"],    label="Recall",    linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["f1"],        label="F1 (positive class)", linewidth=2)
ax.axvline(best_threshold, color='red', linestyle='--', linewidth=1.2,
           label=f"Best F1 threshold = {best_threshold:.3f}")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis — {best_model_name} (Val Set)")
ax.legend()
plt.tight_layout()
save_plot(); plt.show()

### Feature Importance, Gini (Mean Decrease in Impurity)

Built into every tree-based model in scikit-learn. For each feature, sums the
weighted impurity decrease across all splits in all trees.

$$\text{Gini importance}(j) = \sum_{t \in \text{trees}} \sum_{n \in \text{nodes}(t)} \frac{N_n}{N} \cdot \Delta i(n) \cdot \mathbb{1}[\text{split}(n) = j]$$

**Caveat.** Gini importance is biased toward high-cardinality and continuous features. We
cross-check with permutation importance in the next section.

In [ ]:
gini_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_gini = gini_df.head(20)

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top20_gini["feature"], top20_gini["importance"],
        color="steelblue", edgecolor="black", linewidth=0.6)
ax.set_title(f"{best_model_name} — Top 20 Features by Gini Importance",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Mean decrease in impurity")
ax.invert_yaxis()
plt.tight_layout()
save_plot(); plt.show()

print("Top 20 features by Gini importance:")
top20_gini.round(4)

### Feature Importance, Permutation

Model-agnostic, less biased than Gini. For each feature $j$, randomly shuffle its values in
the validation set and measure the drop in ROC-AUC. Repeat several times and average.

**Performance note.** Permutation importance is slow on large datasets, 200K+ rows × 150
features × 5 repeats would take a long time. We subsample the validation set to **10,000 rows**
(stratified on `TARGET`) for tractability. This is documented in the report's Methodology section.

In [ ]:
# --- Stratified subsample of the val set for permutation importance ---
SUBSAMPLE = 10_000
rng = np.random.default_rng(SEED)

# Stratified subsample: keep class ratio identical to full val set
val_pos_idx = y_val[y_val == 1].index
val_neg_idx = y_val[y_val == 0].index
n_pos = int(SUBSAMPLE * y_val.mean())
n_neg = SUBSAMPLE - n_pos

sub_idx = np.concatenate([
    rng.choice(val_pos_idx, size=n_pos, replace=False),
    rng.choice(val_neg_idx, size=n_neg, replace=False),
])
X_val_sub = X_val.loc[sub_idx]
y_val_sub = y_val.loc[sub_idx]

print(f"Permutation subsample: {len(sub_idx):,} rows  |  default rate: {y_val_sub.mean():.4f}")

t0 = time.time()
perm_result = permutation_importance(
    best_model, X_val_sub, y_val_sub,
    n_repeats=5,
    scoring='roc_auc',
    random_state=SEED,
    n_jobs=-1,
)
perm_time = time.time() - t0
print(f"Permutation importance computed in {perm_time:.1f} s")

print("done")


In [ ]:
# saving

joblib.dump(perm_result, os.path.join(OUTPUT_DIR, 'rf_perm_importance.pkl'))
print("saved")


In [ ]:
perm_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": perm_result.importances_mean,
    "std"       : perm_result.importances_std,
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_perm = perm_df.head(20)

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top20_perm["feature"], top20_perm["importance"],
        xerr=top20_perm["std"], color="darkorange", edgecolor="black",
        linewidth=0.6, ecolor='black', capsize=3)
ax.set_title(f"{best_model_name} — Top 20 Features by Permutation Importance (Val)",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Mean drop in ROC-AUC when feature is shuffled")
ax.invert_yaxis()
plt.tight_layout()
save_plot(); plt.show()

print("Top 20 features by permutation importance:")
top20_perm.round(4)

### Summary & Takeaways

### Headline numbers

| Item | Value |
|---|---|
| Best configuration | `n_estimators=500, max_depth=30, max_leaf_nodes=500, min_samples_split=20` |
| Best CV ROC-AUC (5-fold) | **0.7483** |
| Tuned RF, Val ROC-AUC / Test ROC-AUC | **0.7487 / 0.7518** |
| Tuned RF, Val PR-AUC / Test PR-AUC | 0.2214 / 0.2287 |
| Default RF, Test ROC-AUC | 0.7338 |
| SMOTE RF, Test ROC-AUC | 0.7184 |
| Optimal F1 threshold (Val) | 0.552 (P=0.209, R=0.472, F1=0.290) |
| Default RF training time | 84 s |
| Tuned RF training time | 294 s |
| Random search time (15 configs × 5 folds) | 162.5 min |
| Permutation importance time (10k subsample) | 492 s |

### Key findings

**1. Tuning helped, SMOTE didn't.**
Random search lifted Test ROC-AUC by **+0.018** over the default RF (0.734 to 0.752).
SMOTE *underperformed* `class_weight='balanced'` by **0.033 ROC-AUC** on test, synthetic minority samples generated by k-NN interpolation appear to add noise rather than signal in this high-dimensional one-hot-encoded space. Worth flagging in the report's Discussion section.

**2. Regularisation reduces but doesn't eliminate overfitting (RQ1).**
Best config, train ROC-AUC 0.836 vs CV 0.748 to gap of 0.088. Tighter regularisation (`min_samples_split=20`, `max_leaf_nodes=500`, `max_depth=30`) closed the gap from ~0.20 (default config) to ~0.09, a clear regularisation-vs-generalisation story for RQ1.

**3. RF and LR see different features (RQ2).**
- Spearman rank correlation between LR coefficients and RF importance, **0.13, 0.15** (weak).
- Top-20 overlap, **only 7 features**.
- The two RF importance measures (Gini vs Permutation) agree much more closely with each other (ρ = 0.69, overlap = 14/20).
- **`EXT_SOURCE_*`** features and engineered ratios (`CREDIT_TERM`, `CREDIT_GOODS_RATIO`, `ANNUITY_INCOME_RATIO`) dominate the RF rankings.
- LR puts much more weight on **categoricals and missingness flags** (`NAME_TYPE_SUITE_Missing`, `EXT_SOURCE_2_MISSING`, `NAME_INCOME_TYPE_Pensioner`).

**4. RF performs at parity with LR, and that's the point.**
The LR baseline notebook reaches ROC-AUC ≈ 0.73, 0.75 on this dataset. The tuned RF reaches 0.7518 test. They are essentially tied. This is consistent with literature on tabular data showing that well-regularised linear models often match tree ensembles when the signal is largely captured by a small number of strong continuous features (here, the EXT_SOURCE block). The RF still earns its place in the model lineup because it complements LR's view of the data, see point 3.

### Implications for next phases

- **For the MLP (next notebook).** the bar to clear is ROC-AUC ≈ 0.75. If the MLP underperforms despite proper regularisation, that's itself a finding (NN's data-hunger on tabular data).
- **For the fairness analysis (Phase 6).** because LR and RF rank features differently, a fairness/explainability comparison across the two models will produce genuinely different conclusions about *why* the model decided what it decided. This strengthens the ethics section.
- **For the complexity table (Phase 5).** RF training is ~15× slower than LR but still tractable (5 min). Random search at 162 min is the real cost, for the production setting we'd cache the result.

### Files saved to `MODEL_DIR`
`rf_best.pkl`, `rf_test_preds.csv`, `rf_summary.csv`, `rf_gini_importance.csv`, `rf_permutation_importance.csv`, `rf_best_info.json`, `rf_cv_results.csv`.

In [ ]:
# we save Random Forest test metrics for the cross-model comparison

rf_test_metrics = results_best_test.copy()
print(rf_test_metrics)


## **7. Modeling, XGBoost**

gradient boosting with regularised trees. randomised search over 20 configs. cross-model comparisons live in sections 10 (importance) and 12 (final results).

### Default XGBoost (Out-of-the-Box)

We start with a default `XGBClassifier` to establish a reference point before tuning.
Two key differences from the sklearn models.

- **`scale_pos_weight`**, the XGBoost equivalent of `class_weight='balanced'`. Set to the
  ratio of negative to positive training samples (~11.5 for the ~8% default rate), so the model
  treats each default as roughly 11.5× more important than a non-default.
- **`eval_metric='auc'`**, tells XGBoost to log AUC during training (no effect on the final
  objective, which stays `binary:logistic`).
- **`tree_method='hist'`**, histogram-based split finding; faster and recommended for
  medium-to-large datasets on CPU.

> Default = 100 trees (`n_estimators=100`), `max_depth=6`, `learning_rate=0.3`.

In [ ]:
xgb_default = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=spw,          # handles class imbalance
    eval_metric='auc',
    tree_method='hist',            # faster on CPU
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

t0 = time.time()
xgb_default.fit(X_train, y_train)
default_train_time = time.time() - t0
print(f"Default XGBoost trained in {default_train_time:.1f} s")
print(f"Number of trees: {xgb_default.n_estimators}")
print(f"Max depth      : {xgb_default.max_depth}")
print(f"Learning rate  : {xgb_default.learning_rate}")

print("done")


In [ ]:
# saving

joblib.dump(xgb_default, os.path.join(OUTPUT_DIR, 'xgb_default.pkl'))
print("saved")


In [ ]:
results_default_val  = evaluate_model(xgb_default, X_val,  y_val,  split_name="Val  (default XGB)")
results_default_test = evaluate_model(xgb_default, X_test, y_test, split_name="Test (default XGB)")

plot_evaluation(xgb_default, X_val, y_val, X_test, y_test,
                title_prefix="Default XGBoost")

xgboost works out of the box on this data because scale_pos_weight handles the class imbalance directly. we still tune to see if we can push the AUC higher.

### Hyperparameter Tuning, Randomised Search with 5-Fold CV

We tune six hyperparameters with `RandomizedSearchCV`.

| Parameter | Range | What it controls |
|---|---|---|
| `n_estimators` | 100, 500 | Number of boosting rounds (trees) |
| `max_depth` | 3, 8 | Per-tree depth, shallower than RF is typical for boosting |
| `learning_rate` | 0.01, 0.3 | Shrinkage applied to each tree's contribution |
| `subsample` | 0.6, 1.0 | Row subsampling fraction per tree (stochastic boosting) |
| `colsample_bytree` | 0.5, 1.0 | Feature subsampling fraction per tree |
| `min_child_weight` | 1, 10 | Minimum sum of instance weights in a leaf (regularises) |

- **CV folds.** 5, stratified on `TARGET`.
- **Scoring.** ROC-AUC (consistent across all models).
- **`n_iter=20`.** 20 random configurations × 5 folds = 100 fits. More configurations than the
  RF search because XGBoost has more hyperparameters and trains faster per configuration.
- **`scale_pos_weight`** kept fixed (set from data, not a tunable hyperparameter here).

> warning, This cell may take 20, 40 minutes on Colab CPU depending on configuration.
> If re-running the notebook, skip this cell and load the saved best estimator instead.

In [ ]:
from scipy.stats import uniform, randint

param_dist = {
    'n_estimators'     : randint(100, 501),
    'max_depth'        : randint(3, 9),
    'learning_rate'    : uniform(0.01, 0.29),   # uniform on [0.01, 0.30]
    'subsample'        : uniform(0.6, 0.4),     # uniform on [0.6, 1.0]
    'colsample_bytree' : uniform(0.5, 0.5),     # uniform on [0.5, 1.0]
    'min_child_weight' : randint(1, 11),
}

xgb_base = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric='auc',
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=SEED,
    return_train_score=True,
)

t0 = time.time()
random_search.fit(X_train, y_train)
search_time = time.time() - t0
print(f"\n=== Random search complete in {search_time/60:.1f} minutes ===")
print(f"Best CV ROC-AUC: {random_search.best_score_:.4f}")
print(f"Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k:22s}: {v}")
n_configs = len(random_search.cv_results_["params"])  # we count just to confirm
print(f"explored {n_configs} configurations")

print("done")


In [ ]:
# saving

joblib.dump(random_search, os.path.join(OUTPUT_DIR, 'xgb_random_search.pkl'))
print("saved")


In [ ]:
# --- Inspect top configurations ---
cv_results = pd.DataFrame(random_search.cv_results_)
cv_summary = cv_results[[
    'mean_test_score', 'std_test_score', 'mean_train_score',
    'param_n_estimators', 'param_max_depth', 'param_learning_rate',
    'param_subsample', 'param_colsample_bytree', 'param_min_child_weight',
    'mean_fit_time'
]].sort_values('mean_test_score', ascending=False).reset_index(drop=True)

cv_summary.columns = [
    'CV ROC-AUC', 'Std', 'Train ROC-AUC',
    'n_estimators', 'max_depth', 'learning_rate',
    'subsample', 'colsample_bytree', 'min_child_weight',
    'Fit time (s)'
]
print("Top 10 configurations:")
cv_summary.head(10).round(4)

In [ ]:
# --- Visualise CV variance across configs (overfitting check) ---
fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(len(cv_summary))
ax.errorbar(x_pos, cv_summary['CV ROC-AUC'], yerr=cv_summary['Std'],
            fmt='o', capsize=4, color='steelblue', label='CV (val) ROC-AUC')
ax.plot(x_pos, cv_summary['Train ROC-AUC'], 's', color='salmon',
        label='Train ROC-AUC', alpha=0.7)
ax.set_xlabel("Configuration rank (by CV ROC-AUC)")
ax.set_ylabel("ROC-AUC")
ax.set_title("XGBoost \u2014 CV ROC-AUC across 20 sampled configs")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_plot(); plt.show()

gap = cv_summary['Train ROC-AUC'] - cv_summary['CV ROC-AUC']
print(f"Mean train-val gap across configs: {gap.mean():.4f}")
print(f"Best config train-val gap        : {gap.iloc[0]:.4f}")

### Takeaway, regularisation effect (feeds RQ1)

XGBoost's train, val gap is typically **smaller than Random Forest's** because.
- Boosting uses shallower trees (lower `max_depth`), each individual tree is a weak learner
  with far lower capacity than the deep RF trees.
- `learning_rate` acts as an additional shrinkage regulariser on every tree's contribution.
- `subsample` and `colsample_bytree` inject stochasticity similar to RF's bagging and feature
  subsampling, further reducing overfitting.

If the train, val gap here (< 0.05 is typical) is smaller than the RF gap (0.088), that is direct
evidence for the report's RQ1 section that gradient boosting is inherently better regularised
than bagging for this dataset.

### Retrain Best Configuration & Evaluate

`RandomizedSearchCV` already refits on the full training set with the best params (`refit=True`
by default). We time the final fit separately for the report's complexity table.

In [ ]:
xgb_best_params = random_search.best_params_

xgb_best = XGBClassifier(
    **xgb_best_params,
    scale_pos_weight=spw,
    eval_metric='auc',
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

t0 = time.time()
xgb_best.fit(X_train, y_train)
best_train_time = time.time() - t0
print(f"Best XGBoost retrained in {best_train_time:.1f} s")
print(f"Best params: {xgb_best_params}")

print("done")


In [ ]:
# saving

joblib.dump(xgb_best, os.path.join(OUTPUT_DIR, 'xgb_best.pkl'))
print("saved")


In [ ]:
results_best_val  = evaluate_model(xgb_best, X_val,  y_val,  split_name="Val  (tuned XGB)")
results_best_test = evaluate_model(xgb_best, X_test, y_test, split_name="Test (tuned XGB)")

plot_evaluation(xgb_best, X_val, y_val, X_test, y_test,
                title_prefix="Tuned XGBoost")

### SMOTE Comparison

Same best hyperparameters, trained on the SMOTE-resampled training set with `scale_pos_weight=1`
(since SMOTE already balances the classes). Mirrors the approach used in the LR and RF notebooks.

In [ ]:
xgb_smote = XGBClassifier(
    **xgb_best_params,
    scale_pos_weight=1,            # SMOTE already balanced the classes
    eval_metric='auc',
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

t0 = time.time()
xgb_smote.fit(X_train_smote, y_train_smote)
smote_train_time = time.time() - t0
print(f"XGBoost (SMOTE) trained in {smote_train_time:.1f} s")

results_smote_val  = evaluate_model(xgb_smote, X_val,  y_val,  split_name="Val  (SMOTE XGB)")
results_smote_test = evaluate_model(xgb_smote, X_test, y_test, split_name="Test (SMOTE XGB)")

plot_evaluation(xgb_smote, X_val, y_val, X_test, y_test,
                title_prefix="XGBoost \u2014 SMOTE")

print("done")


In [ ]:
# saving

joblib.dump(xgb_smote, os.path.join(OUTPUT_DIR, 'xgb_smote.pkl'))
print("saved")


In [ ]:
# --- Pick the better of the two for downstream analysis ---
summary = pd.DataFrame([
    results_default_val,  results_default_test,
    results_best_val,     results_best_test,
    results_smote_val,    results_smote_test,
])
summary["model"] = [
    "Default XGB", "Default XGB",
    "Tuned XGB (spw)", "Tuned XGB (spw)",
    "Tuned XGB (SMOTE)", "Tuned XGB (SMOTE)",
]
summary = summary[["model", "split", "roc_auc", "pr_auc"]]
summary.columns = ["Model", "Split", "ROC-AUC", "PR-AUC"]

print("=== XGBoost \u2014 Results Summary ===\n")
print(summary.to_string(index=False))

best_model_name = (
    summary[summary["Split"].str.startswith("Val")]
    .sort_values("ROC-AUC", ascending=False)
    .iloc[0]["Model"]
)
best_model = {
    "Default XGB":         xgb_default,
    "Tuned XGB (spw)":     xgb_best,
    "Tuned XGB (SMOTE)":   xgb_smote,
}[best_model_name]
print(f"\n>>> Best model selected: {best_model_name}")

xgboost is the one model where SMOTE actually loses on val ROC-AUC. the scale_pos_weight reweighting is doing the same job that SMOTE does, so we end up double-counting the minority class.

In [ ]:
# --- Bar-chart comparison ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("XGBoost \u2014 ROC-AUC & PR-AUC Comparison", fontsize=13, fontweight='bold')

for ax, metric in zip(axes, ["ROC-AUC", "PR-AUC"]):
    bar_data = summary.pivot(index="Split", columns="Model", values=metric)
    bar_data.plot(kind='bar', ax=ax, rot=0, edgecolor='black')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3, fontsize=8)
    ax.legend(title="Model", fontsize=8)

plt.tight_layout()
save_plot(); plt.show()

### Threshold Analysis

Sweep classification thresholds on the val set to find the operating point that maximises
F1 for the positive (default) class. Same procedure as all previous notebooks.

In [ ]:
y_prob_val = best_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 200)
metrics = []
for t in thresholds:
    y_pred_t = (y_prob_val >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_val == 1)).sum()
    fp = ((y_pred_t == 1) & (y_val == 0)).sum()
    fn = ((y_pred_t == 0) & (y_val == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    metrics.append({"threshold": t, "precision": prec, "recall": rec, "f1": f1})

metrics_df = pd.DataFrame(metrics)
best_t_row = metrics_df.loc[metrics_df["f1"].idxmax()]
best_threshold = best_t_row["threshold"]

print(f"Best F1 threshold on val set : {best_threshold:.3f}")
print(f"  Precision : {best_t_row['precision']:.4f}")
print(f"  Recall    : {best_t_row['recall']:.4f}")
print(f"  F1        : {best_t_row['f1']:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(metrics_df["threshold"], metrics_df["precision"], label="Precision", linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["recall"],    label="Recall",    linewidth=1.5)
ax.plot(metrics_df["threshold"], metrics_df["f1"],        label="F1 (positive class)", linewidth=2)
ax.axvline(best_threshold, color='red', linestyle='--', linewidth=1.2,
           label=f"Best F1 threshold = {best_threshold:.3f}")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis \u2014 {best_model_name} (Val Set)")
ax.legend()
plt.tight_layout()
save_plot(); plt.show()

### Feature Importance, Gain (Built-in)

XGBoost's built-in importance uses **gain** by default, the average improvement in the loss
function brought by a feature across all the splits it is used in. This is analogous to (but
not identical to) the Gini importance in Random Forest.

**Caveat.** Like Gini importance, gain-based importance is biased toward features used in many
splits. We cross-check with permutation importance in the next section.

In [ ]:
gain_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_gain = gain_df.head(20)

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top20_gain["feature"], top20_gain["importance"],
        color="steelblue", edgecolor="black", linewidth=0.6)
ax.set_title(f"{best_model_name} \u2014 Top 20 Features by Gain Importance",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Mean gain per split")
ax.invert_yaxis()
plt.tight_layout()
save_plot(); plt.show()

print("Top 20 features by gain importance:")
top20_gain.round(4)

### Feature Importance, Permutation

Model-agnostic permutation importance on a stratified 10,000-row subsample of the val set
(same procedure as the RF notebook so the comparison is apples-to-apples).

In [ ]:
SUBSAMPLE = 10_000
rng = np.random.default_rng(SEED)

val_pos_idx = y_val[y_val == 1].index
val_neg_idx = y_val[y_val == 0].index
n_pos = int(SUBSAMPLE * y_val.mean())
n_neg = SUBSAMPLE - n_pos

sub_idx = np.concatenate([
    rng.choice(val_pos_idx, size=n_pos, replace=False),
    rng.choice(val_neg_idx, size=n_neg, replace=False),
])
X_val_sub = X_val.loc[sub_idx]
y_val_sub = y_val.loc[sub_idx]

print(f"Permutation subsample: {len(sub_idx):,} rows  |  default rate: {y_val_sub.mean():.4f}")

t0 = time.time()
perm_result = permutation_importance(
    best_model, X_val_sub, y_val_sub,
    n_repeats=5,
    scoring='roc_auc',
    random_state=SEED,
    n_jobs=-1,
)
perm_time = time.time() - t0
print(f"Permutation importance computed in {perm_time:.1f} s")

print("done")


In [ ]:
# saving

joblib.dump(perm_result, os.path.join(OUTPUT_DIR, 'xgb_perm_importance.pkl'))
print("saved")


In [ ]:
perm_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": perm_result.importances_mean,
    "std"       : perm_result.importances_std,
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_perm = perm_df.head(20)

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top20_perm["feature"], top20_perm["importance"],
        xerr=top20_perm["std"], color="darkorange", edgecolor="black",
        linewidth=0.6, ecolor='black', capsize=3)
ax.set_title(f"{best_model_name} \u2014 Top 20 Features by Permutation Importance (Val)",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Mean drop in ROC-AUC when feature is shuffled")
ax.invert_yaxis()
plt.tight_layout()
save_plot(); plt.show()

print("Top 20 features by permutation importance:")
top20_perm.round(4)

### Summary & Takeaways

### Headline numbers

| Item | Value |
|---|---|
| Best CV ROC-AUC (5-fold, random search) | **0.7639** |
| Tuned XGB, Val ROC-AUC / Test ROC-AUC | **0.7667 / 0.7688** |
| Tuned XGB, Val PR-AUC / Test PR-AUC | **0.2530 / 0.2578** |
| Default XGB, Test ROC-AUC | 0.7540 |
| SMOTE XGB, Test ROC-AUC | 0.7564 |
| Optimal F1 threshold (Val) | 0.650 to Precision 0.24 / Recall 0.44 / F1 0.31 |
| Default XGB training time | 10.4 s |
| Tuned XGB training time | 30.4 s |
| Random search time (20 configs × 5 folds) | 30.0 min |

### Key findings

**1. Boosting beats Bagging (core lecture question, answered).**  
XGBoost (Test ROC-AUC = **0.7688**) outperforms Random Forest (**0.7518**) by **+0.017**,
confirming the lecture claim that gradient boosting tends to produce stronger models than
bagging on the same tabular dataset. The gain is also consistent on PR-AUC (+0.029),
where class imbalance makes the metric harder to game.

**2. `scale_pos_weight` beats SMOTE, again.**  
Tuned XGB with `scale_pos_weight` (0.7688) outperforms SMOTE (0.7564) by **+0.012**.
This is the fourth consecutive model in this project where native class-weight handling
outperforms oversampling. SMOTE also collapses recall to ~2% at threshold = 0.5 because
the model shifts its probability mass back toward the majority class after resampling.

**3. Regularisation (RQ1), best train, val gap in the project.**  
The best config's train, val gap is **0.061** (mean across configs, 0.108), comfortably
below RF's 0.088. `learning_rate = 0.039` and shallow trees (`max_depth = 5`,
`min_child_weight = 8`) together act as strong regularisers, directly supporting the
RQ1 narrative that boosting with shrinkage is better regularised than deep RF bagging.

**4. Feature rankings (RQ2), moderate agreement between Gain and Permutation.**  
Spearman ρ = **0.516** between Gain and Permutation importance; 14 of the top-20 features
overlap. The moderate correlation reflects a known limitation of Gain importance, it is
biased toward high-cardinality continuous features (like `EXT_SOURCE_*`), while
permutation importance is more conservative and model-agnostic.

### Cross-model performance summary

| Rank | Model | Test ROC-AUC | Test PR-AUC |
|---|---|---|---|
| 1 | **XGBoost (tuned)** | **0.7688** | **0.2578** |
| 2 | Elastic Net (tuned) | 0.7523 | 0.2350 |
| 3 | LR Baseline | 0.7521 | 0.2350 |
| 4 | Random Forest (tuned) | 0.7518 | 0.2287 |
| 5 | LR + PCA | 0.7481 | 0.2316 |
|, | Kaggle #1 (reference) | 0.8057 |, |

XGBoost is the clear winner across both metrics. The gap to Kaggle #1 (−0.037 ROC-AUC)
reflects the absence of feature engineering on the auxiliary tables (bureau, installments,
previous applications), which top-leaderboard solutions exploit heavily.

### How to improve further

| Strategy | Expected gain | Effort |
|---|---|---|
| **Feature engineering** from `bureau.csv`, `installments_payments.csv`, `previous_application.csv` | +0.02, 0.04 ROC-AUC (biggest lever) | High |
| **Bayesian hyperparameter search** (Optuna) instead of random search; add `reg_alpha`, `reg_lambda`, `gamma` | +0.003, 0.008 | Medium |
| **Early stopping** (`early_stopping_rounds=50`) on eval set instead of fixed `n_estimators` | Prevents over/under-fitting; slight gain | Low |
| **LightGBM** (same gradient boosting family, but leaf-wise growth + better handling of high-cardinality features) | Often +0.002, 0.005 with less tuning | Low |
| **Stacking / blending** XGBoost + LR + RF predictions as meta-features | +0.003, 0.010 | Medium |
| **Threshold calibration**, current optimal F1 threshold 0.65 shows recall can reach 0.44; tuning for business cost (FP vs FN tradeoff) changes the deployment decision | No ROC-AUC change | Low |

### Files saved to `MODEL_DIR`
`xgb_best.pkl`, `xgb_test_preds.csv`, `xgb_summary.csv`, `xgb_gain_importance.csv`,
`xgb_permutation_importance.csv`, `xgb_best_info.json`, `xgb_cv_results.csv`.

In [ ]:
# we save XGBoost test metrics for the cross-model comparison

xgb_test_metrics = results_best_test.copy()
print(xgb_test_metrics)


## **8. Modeling, MLP (Keras)**

custom multilayer perceptron with dropout, batch norm, and early stopping. we compare ReLU against Leaky ReLU, then run a hyperparameter sweep. the regularisation ablation that feeds RQ1 lives in section 9. the PCA variant in 8.2.

### 8.1 Standard features

In [ ]:
# we convert dataframes to float32 arrays sicne the keras default is float32 anyway (Keras + memory-efficient on GPU)
X_train_arr       = X_train.values.astype(np.float32)
X_train_smote_arr = X_train_smote.values.astype(np.float32)
X_val_arr         = X_val.values.astype(np.float32)
X_test_arr        = X_test.values.astype(np.float32)

y_train_arr       = y_train.values.astype(np.float32)
y_train_smote_arr = y_train_smote.values.astype(np.float32)
y_val_arr         = y_val.values.astype(np.float32)
y_test_arr        = y_test.values.astype(np.float32)

N_FEATURES = X_train_arr.shape[1]
print(f"input dimensionality: {N_FEATURES} features (float32)")


### Base MLP, ReLU + Dropout + BatchNorm + Early Stopping

We start with a clean three-layer MLP as our reference architecture.
- **Activation.** ReLU (Rectified Linear Unit), fast and avoids the vanishing gradient problem
  compared to sigmoid/tanh in hidden layers. Gradient is either 0 (negative input) or 1 (positive),
  so it does not saturate for large activations.
- **Dropout.** Applied after each hidden layer to randomly zero a fraction of activations during
  training, forcing the network to learn redundant representations. Equivalent to training an
  ensemble of thinned networks.
- **BatchNorm.** Normalises each mini-batch's activations to zero mean / unit variance before
  the activation function. Reduces internal covariate shift, allows higher learning rates, and
  acts as a mild regulariser.
- **Early Stopping.** Monitors `val_auc` and stops when it stops improving, restoring the
  best weights. This limits effective model complexity and is our primary defence against
  overfitting.
- **Optimizer.** Adam, adaptive learning rate per parameter, generally the best default
  for tabular data.

### 4.1 Build and train the base MLP

### Class imbalance fix, `class_weight` in `model.fit()`

The training set has ~8% defaults. Without correction, binary cross-entropy treats every sample equally, so the model quickly learns that predicting **always class 0** minimises loss (92% accuracy but 0% recall on defaulters).

`class_weight` re-scales the per-sample loss so that a mistake on a defaulter counts proportionally more than a mistake on a non-defaulter, equivalent to up-weighting the minority class without changing the data.

We compute the weights automatically from the training labels.

$$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_c}$$

This is identical to `sklearn`'s `class_weight='balanced'` formula.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train_arr)
weights = compute_class_weight('balanced', classes=classes, y=y_train_arr)
CLASS_WEIGHT = dict(zip(classes.astype(int), weights))

print(f"Class weights: {CLASS_WEIGHT}")
print(f"  Weight for class 0 (no default) : {CLASS_WEIGHT[0]:.4f}")
print(f"  Weight for class 1 (default)    : {CLASS_WEIGHT[1]:.4f}")
print(f"  Ratio (imbalance factor)        : {CLASS_WEIGHT[1]/CLASS_WEIGHT[0]:.1f}x")

In [ ]:
# builds MLP with relu, optional dropout and batch norm

def build_mlp_relu(n_features, hidden_units=(128, 64), dropout_rate=0.3,
                   use_batchnorm=True, learning_rate=0.001):
    keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = keras.Sequential(name="MLP_ReLU")
    model.add(layers.Input(shape=(n_features,)))

    for units in hidden_units:
        model.add(layers.Dense(units, use_bias=not use_batchnorm))  # bias redundant with BN
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.Dropout(dropout_rate, seed=SEED))

    #sigmoid output: produces a probability score for binary classification
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model


#instantiate and inspect
base_model = build_mlp_relu(N_FEATURES, hidden_units=(128, 64), dropout_rate=0.3,
                             use_batchnorm=True, learning_rate=0.001)
base_model.summary()

the gap between train and val loss is huge after the first few epochs. we need regularisation, that is what 4.3 onward addresses.

In [ ]:
# we set up callbacks

early_stop = EarlyStopping(
    monitor='val_auc',
    patience=10,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

#train


In [ ]:
# we train the model

print("Training base MLP (ReLU, dropout=0.3, BatchNorm, Adam lr=0.001)...")
t0 = time.time()

history_base = base_model.fit(
    X_train_arr, y_train_arr,
    validation_data=(X_val_arr, y_val_arr),
    epochs=100,
    batch_size=1024,
    callbacks=[early_stop],
    class_weight=CLASS_WEIGHT,   #re-weight minority class — prevents always predicting class 0
    verbose=1
)

train_time_base = time.time() - t0
print(f"\nTraining time: {train_time_base:.1f}s")
print(f"Stopped at epoch: {early_stop.stopped_epoch + 1 if early_stop.stopped_epoch > 0 else len(history_base.history['loss'])}")


### 4.2 Training curves

The key overfitting / underfitting diagnostic. A well-regularised model should show.
- Train loss decreasing steadily, val loss following closely and plateauing (not diverging).
- Train AUC and val AUC converging rather than separating.

In [ ]:
plot_training_curves(history_base, title="Base MLP (ReLU + Dropout 0.3 + BatchNorm + Adam)")

The key takeaway from these charts is that the model overfitted the training data. The distance between the Val and Train losses keep increasing. Moreover, on the Val data both the loss and AUC do not see many improvements along the epochs.

### 4.3 Evaluation, validation and test sets

In [ ]:
metrics_base_val  = evaluate_keras_model(base_model, X_val_arr,  y_val_arr,  split_name="Val")
metrics_base_test = evaluate_keras_model(base_model, X_test_arr, y_test_arr, split_name="Test")

These results are extremely poor. Our model can easily get a 98% precision on "No Default" because the data is extremely unbalanced, but when it comes to "Defaults" it is terrible (11%). On the other hand, our model captures 93% of all "Defaults", but it does not capture the "No Defaults" (32%), meaning it is predicting "Default" more times than it should. We tecnically want a high recall, since we do not want to miss "Default" cases, but this model is not good despite doing this.

In [ ]:
plot_evaluation_keras(base_model, X_val_arr, y_val_arr, X_test_arr, y_test_arr,
                      title_prefix="Base MLP (ReLU)")

### Leaky ReLU MLP, Avoiding Dead Neurons

**The dead neuron problem with ReLU.**  
ReLU outputs exactly 0 for any negative pre-activation. Once a neuron's output is 0, the
gradient flowing through it is also 0. If this happens consistently across many training steps,
the neuron's weights stop updating entirely, the neuron is "dead" and contributes nothing
to the network for the rest of training.

This is especially likely with.
- High learning rates that push weights into the negative region.
- Features that are already zero or negative after scaling.
- Large dropout rates combined with ReLU.

**Leaky ReLU** fixes this by allowing a small, non-zero gradient for negative inputs.

$$
\text{LeakyReLU}(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha x & \text{otherwise} \end{cases}
$$

where $\alpha$ is a small constant (typically 0.1 or 0.3). The negative slope ensures the
gradient is never exactly 0, so all neurons remain "alive" throughout training.

We use the **same architecture and hyperparameters** as the base MLP to isolate the effect
of the activation function.

In [ ]:
# builds MLP with leaky relu activation

def build_mlp_leaky_relu(n_features, hidden_units=(128, 64), dropout_rate=0.3,
                          use_batchnorm=True, learning_rate=0.001, alpha=0.1):
    keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = keras.Sequential(name="MLP_LeakyReLU")
    model.add(layers.Input(shape=(n_features,)))

    for units in hidden_units:
        model.add(layers.Dense(units, use_bias=not use_batchnorm))
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        #LeakyReLU is its own layer in Keras; alpha controls the negative slope
        model.add(layers.LeakyReLU(negative_slope=alpha))
        model.add(layers.Dropout(dropout_rate, seed=SEED))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model


leaky_model = build_mlp_leaky_relu(N_FEATURES, hidden_units=(128, 64), dropout_rate=0.3,
                                    use_batchnorm=True, learning_rate=0.001, alpha=0.1)
leaky_model.summary()

In [ ]:
# we set up callbacks

early_stop_leaky = EarlyStopping(
    monitor='val_auc', patience=10, mode='max',
    restore_best_weights=True, verbose=1
)


In [ ]:
# we train the model

print("Training Leaky ReLU MLP (alpha=0.1, dropout=0.3, BatchNorm, Adam lr=0.001)...")
t0 = time.time()

history_leaky = leaky_model.fit(
    X_train_arr, y_train_arr,
    validation_data=(X_val_arr, y_val_arr),
    epochs=100,
    batch_size=1024,
    callbacks=[early_stop_leaky],
    class_weight=CLASS_WEIGHT,   # same weights as base model for fair comparison
    verbose=1
)

train_time_leaky = time.time() - t0
print(f"\nTraining time: {train_time_leaky:.1f}s")


In [ ]:
plot_training_curves(history_leaky, title="Leaky ReLU MLP (alpha=0.1 + Dropout 0.3 + BatchNorm + Adam)")

Again we are looking at an overfitted model.

In [ ]:
metrics_leaky_val  = evaluate_keras_model(leaky_model, X_val_arr,  y_val_arr,  split_name="Val")
metrics_leaky_test = evaluate_keras_model(leaky_model, X_test_arr, y_test_arr, split_name="Test")

leaky relu helps a little but not enough to close the train-val gap. we move on to architecture sweeps.

Our "No Default" class increased its f1-score by 2 pp, which now sits at 50%, which is still bad. The "Default" class stayed the same and this model is very similar to the ReLu one.

### 5.1 ReLU vs Leaky ReLU, side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("ReLU vs Leaky ReLU — Training Dynamics", fontsize=13, fontweight='bold')

#find the AUC key dynamically (Keras may suffix duplicate metric names)
def _auc_key(h): return [k for k in h.history if 'auc' in k and 'val' not in k][0]

for history, label, color in [
    (history_base,  'ReLU',        'steelblue'),
    (history_leaky, 'Leaky ReLU',  'tomato'),
]:
    auc_k = _auc_key(history)
    axes[0].plot(history.history['val_loss'], label=label, color=color)
    axes[1].plot(history.history[f'val_{auc_k}'], label=label, color=color)

axes[0].set_title("Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend()

axes[1].set_title("Val AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.tight_layout()
save_plot(); plt.show()

#quick summary table
auc_relu   = roc_auc_score(y_val_arr, base_model.predict(X_val_arr, verbose=0).ravel())
auc_leaky  = roc_auc_score(y_val_arr, leaky_model.predict(X_val_arr, verbose=0).ravel())

comp = pd.DataFrame({
    "Model"       : ["ReLU", "Leaky ReLU (alpha=0.1)"],
    "Val AUC"     : [f"{auc_relu:.4f}", f"{auc_leaky:.4f}"],
    "Train time"  : [f"{train_time_base:.1f}s", f"{train_time_leaky:.1f}s"],
    "Dead neurons": ["Possible with large lr", "Prevented by alpha slope"],
})
print(comp.to_string(index=False))

Leaky ReLu registered a lower Val Loss than with the normal ReLu.

### Hyperparameter Experiments

Systematic sweep over four axes.
1. **Dropout rate.** 0.2 / 0.3 / 0.4 / 0.5
2. **BatchNorm.** On / Off
3. **Layer sizes.** [128, 64] / [256, 128, 64] / [64, 32]
4. **Learning rate.** 0.001 / 0.0001

Each config is trained with early stopping (patience=10) to ensure fair comparison.
We use the best Leaky ReLU architecture (alpha=0.1) as the sweep base since it avoids
dead neurons and performed comparably or better than plain ReLU.

> **Runtime note.** 12 configs × up to 100 epochs on a T4 GPU ≈ 8, 15 min total.

In [ ]:
sweep_configs = [
    #dropout, use_batchnorm, hidden_units, lr,label
    (0.2,  True,  (128, 64),       0.001,  "drop=0.2 | BN | [128,64]  | lr=0.001"),
    (0.3,  True,  (128, 64),       0.001,  "drop=0.3 | BN | [128,64]  | lr=0.001  ← base"),
    (0.4,  True,  (128, 64),       0.001,  "drop=0.4 | BN | [128,64]  | lr=0.001"),
    (0.5,  True,  (128, 64),       0.001,  "drop=0.5 | BN | [128,64]  | lr=0.001"),
    (0.3,  False, (128, 64),       0.001,  "drop=0.3 | no BN | [128,64] | lr=0.001"),
    (0.3,  True,  (256, 128, 64),  0.001,  "drop=0.3 | BN | [256,128,64] | lr=0.001"),
    (0.3,  True,  (64, 32),        0.001,  "drop=0.3 | BN | [64,32]   | lr=0.001"),
    (0.3,  True,  (128, 64),       0.0001, "drop=0.3 | BN | [128,64]  | lr=0.0001"),
    (0.2,  True,  (256, 128, 64),  0.001,  "drop=0.2 | BN | [256,128,64] | lr=0.001"),
    (0.4,  False, (128, 64),       0.001,  "drop=0.4 | no BN | [128,64] | lr=0.001"),
    (0.3,  True,  (256, 128, 64),  0.0001, "drop=0.3 | BN | [256,128,64] | lr=0.0001"),
    (0.2,  True,  (128, 64),       0.0001, "drop=0.2 | BN | [128,64]  | lr=0.0001"),
]

sweep_results = []

for dropout, use_bn, units, lr, label in sweep_configs:
    print(f"\n[CONFIG] {label}")

    model = build_mlp_leaky_relu(
        N_FEATURES, hidden_units=units, dropout_rate=dropout,
        use_batchnorm=use_bn, learning_rate=lr, alpha=0.1
    )

    es = EarlyStopping(monitor='val_auc', patience=10, mode='max',
                       restore_best_weights=True, verbose=0)

    t0 = time.time()
    hist = model.fit(
        X_train_arr, y_train_arr,
        validation_data=(X_val_arr, y_val_arr),
        epochs=100,
        batch_size=1024,
        callbacks=[es],
        verbose=0
    )
    elapsed = time.time() - t0

    auc_key = [k for k in hist.history if 'auc' in k and 'val' not in k][0]
    best_val_auc  = max(hist.history[f'val_{auc_key}'])
    best_val_loss = min(hist.history['val_loss'])
    n_params      = model.count_params()
    epochs_run    = len(hist.history['loss'])

    sweep_results.append({
        "Config"         : label,
        "Dropout"        : dropout,
        "BatchNorm"      : use_bn,
        "Hidden units"   : str(units),
        "LR"             : lr,
        "Best val AUC"   : round(best_val_auc, 4),
        "Best val loss"  : round(best_val_loss, 4),
        "Epochs run"     : epochs_run,
        "Params"         : n_params,
        "Train time (s)" : round(elapsed, 1),
    })
    print(f"  Best val AUC: {best_val_auc:.4f}  |  epochs: {epochs_run}  |  time: {elapsed:.1f}s")

sweep_df = pd.DataFrame(sweep_results).sort_values("Best val AUC", ascending=False).reset_index(drop=True)
print("\n=== Hyperparameter Sweep Results (sorted by val AUC) ===")
display(sweep_df)

In [ ]:
# visual summary of the sweep
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(sweep_df))]
bars = ax.barh(sweep_df["Config"], sweep_df["Best val AUC"], color=colors)
ax.set_xlabel("Best Validation AUC")
ax.set_title("Hyperparameter Sweep — Best Validation AUC per Config", fontweight='bold')

#annotate each bar with its AUC value
for bar, val in zip(bars, sweep_df["Best val AUC"]):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va='center', fontsize=8)

ax.set_xlim(sweep_df["Best val AUC"].min() - 0.005, sweep_df["Best val AUC"].max() + 0.01)
plt.tight_layout()
save_plot(); plt.show()

best_config = sweep_df.iloc[0]
print(f"\nBest config: {best_config['Config']}")
print(f"Val AUC: {best_config['Best val AUC']:.4f}")

### SMOTE vs Non-SMOTE Comparison

We take the **best MLP configuration** identified from the ablation study (all three
regularisers active) and train it on both.
- `X_train` / `y_train`, original imbalanced set (~8% default rate)
- `X_train_smote` / `y_train_smote`, SMOTE-resampled (50/50 balance)

The goal is to understand how resampling shifts the **precision/recall trade-off**.
- SMOTE inflates the minority class, which typically improves recall on defaulters at the cost
  of some precision.
- The original set retains the natural class distribution; the model may prefer predicting
  non-default since it is by far the majority class.

In [ ]:
def train_best_mlp(X_train_data, y_train_data, label="original"):
    """
    Train the best MLP config (all three regularisers) on a given training set.
    Returns (model, history, training_time).
    """
    model = build_mlp_leaky_relu(
        N_FEATURES,
        hidden_units=BEST_UNITS,
        dropout_rate=BEST_DROPOUT,
        use_batchnorm=True,
        learning_rate=BEST_LR,
        alpha=0.1
    )

    es = EarlyStopping(monitor='val_auc', patience=10, mode='max',
                       restore_best_weights=True, verbose=0)

    print(f"Training best MLP on {label} set (n={len(X_train_data):,})...")
    t0 = time.time()
    hist = model.fit(
        X_train_data, y_train_data,
        validation_data=(X_val_arr, y_val_arr),
        epochs=100,
        batch_size=1024,
        callbacks=[es],
        verbose=0
    )
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s")
    return model, hist, elapsed


best_model_orig,  hist_orig,  time_orig  = train_best_mlp(X_train_arr,       y_train_arr,       label="original")
best_model_smote, hist_smote, time_smote = train_best_mlp(X_train_smote_arr, y_train_smote_arr, label="SMOTE")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def smote_comparison_metrics(model, X, y, label, threshold=0.3):
    """Return a dict of metrics for the SMOTE comparison table."""
    y_prob = model.predict(X, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model"         : label,
        "Split"         : "Val" if X is X_val_arr else "Test",
        "ROC-AUC"       : round(roc_auc_score(y, y_prob), 4),
        "PR-AUC"        : round(average_precision_score(y, y_prob), 4),
        "Precision (1)" : round(precision_score(y, y_pred, pos_label=1, zero_division=0), 4),
        "Recall (1)"    : round(recall_score(y, y_pred, pos_label=1), 4),
        "F1 (1)"        : round(f1_score(y, y_pred, pos_label=1), 4),
    }

rows = []
for model, label in [(best_model_orig, "Original"), (best_model_smote, "SMOTE")]:
    for X, y, split in [(X_val_arr, y_val_arr, "Val"), (X_test_arr, y_test_arr, "Test")]:
        r = smote_comparison_metrics(model, X, y, label)
        r["Split"] = split
        rows.append(r)

smote_df = pd.DataFrame(rows)
print("=== SMOTE vs Original — Precision/Recall Impact ===")
display(smote_df)

SMOTE data created some models with high recall, this is because SMOTE is known to make models overestimate the minority class, this can be seen in the extremely low precision score.

In [ ]:
# PR curve comparison (val set)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SMOTE vs Original — ROC and Precision-Recall curves (Val set)",
             fontsize=13, fontweight='bold')

for model, label, color in [
    (best_model_orig,  "Original", 'steelblue'),
    (best_model_smote, "SMOTE",    'tomato'),
]:
    y_prob = model.predict(X_val_arr, verbose=0).ravel()

    fpr, tpr, _ = roc_curve(y_val_arr, y_prob)
    auc = roc_auc_score(y_val_arr, y_prob)
    axes[0].plot(fpr, tpr, label=f"{label} (AUC={auc:.4f})", color=color)

    prec, rec, _ = precision_recall_curve(y_val_arr, y_prob)
    ap = average_precision_score(y_val_arr, y_prob)
    axes[1].plot(rec, prec, label=f"{label} (AP={ap:.4f})", color=color)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve"); axes[0].legend()

baseline_rate = y_val_arr.mean()
axes[1].axhline(baseline_rate, color='k', linestyle='--', linewidth=0.8,
                label=f"Random baseline ({baseline_rate:.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()

plt.tight_layout()
save_plot(); plt.show()

### Final MLP, Default DNN Configuration

We train one final MLP following the **published default DNN configuration** (Géron, 2022,
*Hands-On Machine Learning*, Table 11-2). This configuration is explicitly designed for
self-normalising deep networks and differs substantially from the dropout-based approach above.

| Hyperparameter | Value | Rationale |
|---|---|---|
| Kernel initialiser | LeCun (Normal) | Ensures variance ≈ 1 at each layer, precondition for SELU self-normalisation |
| Activation | SELU | Scaled Exponential Linear Unit: pushes activations towards zero mean / unit variance automatically |
| Normalisation | None (self-normalisation) | SELU + LeCun init provides normalisation without BatchNorm |
| Regularisation | Early stopping | Complexity control via training duration |
| Optimizer | Nadam | Nesterov momentum + Adam adaptive rates; often converges faster than plain Adam |
| LR schedule | Performance (ReduceLROnPlateau) | Halves LR when val_loss stagnates, coarse-to-fine optimisation |

> **SELU caveat.** SELU requires the inputs to each neuron to be i.i.d. with zero mean and
> unit variance, the network weights to be initialised with LeCun Normal, and the network to
> be fully connected (no skip connections, no BatchNorm). All three conditions are met here.

In [ ]:
# builds the final default DNN configuration

def build_mlp_default_dnn(n_features, hidden_units=(128, 64), learning_rate=0.001):
    keras.backend.clear_session()
    tf.random.set_seed(SEED)

    lecun_init = keras.initializers.LecunNormal(seed=SEED)

    model = keras.Sequential(name="MLP_Default_DNN")
    model.add(layers.Input(shape=(n_features,)))

    for units in hidden_units:
        model.add(layers.Dense(
            units,
            activation='selu',
            kernel_initializer=lecun_init
        ))
        #no BatchNorm, SELU self-normalises
        #no Dropout, would need AlphaDropout to preserve SELU's statistics

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Nadam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model


dnn_model = build_mlp_default_dnn(N_FEATURES, hidden_units=BEST_UNITS, learning_rate=0.001)
dnn_model.summary()

In [ ]:
# we set up callbacks

#Early stopping: stops when val_auc has not improved for 10 epochs
early_stop_dnn = EarlyStopping(
    monitor='val_auc',
    patience=10,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

#Performance scheduling (ReduceLROnPlateau): halves LR when val_loss does not
#improve for 5 epochs, coarse optimisation first, then fine-tuning
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)


In [ ]:
# we train the model

print("Training Default DNN (SELU + LeCun init + Nadam + ReduceLROnPlateau + EarlyStopping)...")
t0 = time.time()

history_dnn = dnn_model.fit(
    X_train_arr, y_train_arr,
    validation_data=(X_val_arr, y_val_arr),
    epochs=100,
    batch_size=1024,
    callbacks=[early_stop_dnn, reduce_lr],
    verbose=1
)

train_time_dnn = time.time() - t0
print(f"\nTraining time: {train_time_dnn:.1f}s")


In [ ]:
plot_training_curves(history_dnn,
                     title="Default DNN Config (SELU + LeCun + Nadam + ReduceLROnPlateau)")

In [ ]:
metrics_dnn_val  = evaluate_keras_model(dnn_model, X_val_arr,  y_val_arr,  split_name="Val")
metrics_dnn_test = evaluate_keras_model(dnn_model, X_test_arr, y_test_arr, split_name="Test")

In [ ]:
plot_evaluation_keras(dnn_model, X_val_arr, y_val_arr, X_test_arr, y_test_arr,
                      title_prefix="Default DNN (SELU + LeCun + Nadam)")

Despite this model overfitting the data, we came up with a 95% f1-score for "No Default", which is one of the best from all our models. Of course, the "Default" f1-score is very low, with a even lower recall, which is unnaceptable as we are trying to capture all "Default" cases.

### Consolidated Model Comparison

All MLP variants evaluated on the **validation set** in one table, sorted by AUC.

In [ ]:
def quick_val_metrics(model, name):
    y_prob = model.predict(X_val_arr, verbose=0).ravel()
    y_pred = (y_prob >= 0.3).astype(int)
    return {
        "Model"         : name,
        "Val ROC-AUC"   : round(roc_auc_score(y_val_arr, y_prob), 4),
        "Val PR-AUC"    : round(average_precision_score(y_val_arr, y_prob), 4),
        "Precision (1)" : round(precision_score(y_val_arr, y_pred, pos_label=1, zero_division=0), 4),
        "Recall (1)"    : round(recall_score(y_val_arr, y_pred, pos_label=1), 4),
        "F1 (1)"        : round(f1_score(y_val_arr, y_pred, pos_label=1), 4),
    }

summary_rows = [
    quick_val_metrics(base_model,        "Base MLP (ReLU)"),
    quick_val_metrics(leaky_model,       "Base MLP (Leaky ReLU)"),
    quick_val_metrics(best_model_orig,   "Best config — Original data"),
    quick_val_metrics(best_model_smote,  "Best config — SMOTE data"),
    quick_val_metrics(dnn_model,         "Default DNN (SELU + LeCun + Nadam)"),
]

summary_df = pd.DataFrame(summary_rows).sort_values("Val ROC-AUC", ascending=False).reset_index(drop=True)
print("=== MLP Variant Summary (Val set, sorted by AUC) ===")
display(summary_df)

### Save Best Model and Results

In [ ]:
# identify and save the best model (highest val AUC)
best_model_name = summary_df.iloc[0]["Model"]
best_val_auc    = summary_df.iloc[0]["Val ROC-AUC"]
print(f"Best model: {best_model_name}  |  Val AUC: {best_val_auc:.4f}")

model_map = {
    "Base MLP (ReLU)"                    : base_model,
    "Base MLP (Leaky ReLU)"              : leaky_model,
    "Best config — Original data"        : best_model_orig,
    "Best config — SMOTE data"           : best_model_smote,
    "Default DNN (SELU + LeCun + Nadam)" : dnn_model,
}

best_mlp = model_map[best_model_name]

# save in Keras native format
best_mlp.save(f"{OUTPUT_DIR}/mlp_best.keras")
print(f"Saved best MLP to {OUTPUT_DIR}/mlp_best.keras")

# save the default DNN config model separately (distinct architecture)
dnn_model.save(f"{OUTPUT_DIR}/mlp_default_dnn.keras")
print(f"Saved default DNN config to {OUTPUT_DIR}/mlp_default_dnn.keras")

# save the ablation table and sweep table
ablation_df.to_csv(f"{OUTPUT_DIR}/mlp_ablation_results.csv", index=False)
sweep_df.to_csv(f"{OUTPUT_DIR}/mlp_sweep_results.csv",    index=False)
smote_df.to_csv(f"{OUTPUT_DIR}/mlp_smote_comparison.csv", index=False)
summary_df.to_csv(f"{OUTPUT_DIR}/mlp_summary.csv",        index=False)

print("\nAll result tables saved.")

### Final Test-Set Evaluation of Best MLP

The test set is touched **only once**. Do not re-tune any hyperparameters after seeing these results.

In [ ]:
print(f"=== Final Test-Set Evaluation: {best_model_name} ===")
metrics_final = evaluate_keras_model(best_mlp, X_test_arr, y_test_arr, split_name="Test")

plot_evaluation_keras(best_mlp, X_val_arr, y_val_arr, X_test_arr, y_test_arr,
                      title_prefix=f"Final MLP — {best_model_name}")

print(f"\n\u2713 Test ROC-AUC : {metrics_final['roc_auc']:.4f}")
print(f"\u2713 Test PR-AUC  : {metrics_final['pr_auc']:.4f}")

**Conclusion**

Our best model in terms of Val AUC is the one with Dropout = 0.5, BatchNorm, Early Stopping with 1 layer with 128 nodes, another with 64 and the output node.

Still this model has low recall, which will means that we are horrible at finding who is going to default. Unbalanced data is extremely hard to predict with MLP models.

### 8.2 PCA features

### PCA Data, Load and Prepare

We now load the PCA-reduced splits produced by `02_preprocessing_pca.ipynb`.  
These splits live in `03_processed_pca/` and their columns are principal components
(`PC1 … PCk`) rather than original features.

Key differences vs the original data.
- Features are orthogonal by construction, multicollinearity is eliminated.
- All PCs are already zero-mean and unit-variance (PCA standardises during fitting).
- The number of features is smaller (k components capturing 95% of variance vs the full
  original feature set), which reduces the input dimensionality of the MLP.
- The target arrays `y_*` are identical to those used in Sections 4, 12.

> **Path note.** The PCA data folder is `03_processed_pca` inside the same Drive root
> as your original data. Adjust `PCA_DATA_DIR` if your folder structure differs.

we re-use the PCA matrices already in memory from section 3.2 instead of reloading from disk. saves time on rerun and keeps the chain explicit.

In [ ]:
# we use the PCA matrices already produced in section 3.2 (no re-load from disk)
X_train_pca_arr       = X_train_pca.values.astype(np.float32)
X_train_smote_pca_arr = X_train_smote_pca.values.astype(np.float32)
X_val_pca_arr         = X_val_pca.values.astype(np.float32)
X_test_pca_arr        = X_test_pca.values.astype(np.float32)

N_FEATURES_PCA = X_train_pca_arr.shape[1]
print(f"PCA input dimensionality: {N_FEATURES_PCA} features (float32)")
print(f"dimensionality reduction: {N_FEATURES} → {N_FEATURES_PCA} ({N_FEATURES - N_FEATURES_PCA} removed)")


### PCA, Base MLP (ReLU)

We retrain the exact same base architecture from Section 4, only changing the input
dimensionality from `N_FEATURES` to `N_FEATURES_PCA`.  
Everything else, hidden units, dropout rate, BatchNorm, optimizer, class_weight,
early stopping patience, is kept identical so the comparison is fair.

If PCA improves AUC, it suggests the removed variance was noise that was hurting
generalisation.  If AUC drops, the discarded components contained discriminative
signal that the MLP was using.

In [ ]:
# we build base ReLU MLP on PCA features
base_model_pca = build_mlp_relu(
    N_FEATURES_PCA, hidden_units=(128, 64),
    dropout_rate=0.3, use_batchnorm=True, learning_rate=0.001
)
base_model_pca.summary()

In [ ]:
# we set up callbacks

early_stop_base_pca = EarlyStopping(
    monitor='val_auc', patience=10, mode='max',
    restore_best_weights=True, verbose=1
)


In [ ]:
# we train the model

print("Training Base MLP ReLU on PCA data...")
t0 = time.time()

history_base_pca = base_model_pca.fit(
    X_train_pca_arr, y_train_arr,
    validation_data=(X_val_pca_arr, y_val_arr),
    epochs=100,
    batch_size=1024,
    callbacks=[early_stop_base_pca],
    class_weight=CLASS_WEIGHT,
    verbose=1
)

train_time_base_pca = time.time() - t0
print(f"\nTraining time: {train_time_base_pca:.1f}s")


In [ ]:
plot_training_curves(history_base_pca,
                     title="PCA — Base MLP (ReLU + Dropout 0.3 + BatchNorm + Adam)")

Even with dimensionality reduction, we can still see overfitting as the gap between train and val loss is big.

In [ ]:
metrics_base_pca_val  = evaluate_keras_model(base_model_pca, X_val_pca_arr,  y_val_arr,  split_name="Val  (PCA)")
metrics_base_pca_test = evaluate_keras_model(base_model_pca, X_test_pca_arr, y_test_arr, split_name="Test (PCA)")

In [ ]:
plot_evaluation_keras(base_model_pca, X_val_pca_arr, y_val_arr, X_test_pca_arr, y_test_arr,
                      title_prefix="PCA — Base MLP (ReLU)")

Our model is predicting many defaults and that is why recall is big, but this is not ideal, as precision is terrible, making the f1-score low.

### PCA, Leaky ReLU MLP

Same as Section 5 (Leaky ReLU) but trained on PCA features.  
The dead-neuron motivation for Leaky ReLU applies equally here, PCA components
are real-valued and can be negative, so the same risk of zero-gradient neurons exists.

In [ ]:
# we set up callbacks

leaky_model_pca = build_mlp_leaky_relu(
    N_FEATURES_PCA, hidden_units=(128, 64),
    dropout_rate=0.3, use_batchnorm=True, learning_rate=0.001, alpha=0.1
)

early_stop_leaky_pca = EarlyStopping(
    monitor='val_auc', patience=10, mode='max',
    restore_best_weights=True, verbose=1
)


In [ ]:
# we train the model

print("Training Leaky ReLU MLP on PCA data...")
t0 = time.time()

history_leaky_pca = leaky_model_pca.fit(
    X_train_pca_arr, y_train_arr,
    validation_data=(X_val_pca_arr, y_val_arr),
    epochs=100,
    batch_size=1024,
    callbacks=[early_stop_leaky_pca],
    class_weight=CLASS_WEIGHT,
    verbose=1
)

train_time_leaky_pca = time.time() - t0
print(f"\nTraining time: {train_time_leaky_pca:.1f}s")


In [ ]:
plot_training_curves(history_leaky_pca,
                     title="PCA — Leaky ReLU MLP (alpha=0.1 + Dropout 0.3 + BatchNorm + Adam)")

Slighly less overfitted than the ReLu version, but still not ideal.

In [ ]:
metrics_leaky_pca_val  = evaluate_keras_model(leaky_model_pca, X_val_pca_arr,  y_val_arr,  split_name="Val  (PCA)")
metrics_leaky_pca_test = evaluate_keras_model(leaky_model_pca, X_test_pca_arr, y_test_arr, split_name="Test (PCA)")

Just a small improvement on the "No Default" f1-score (2 p.p.). But again we have a good precision in class 0 and good recall in class 1, so the model is good at predicting when it is not a "Default" and catching "No Default", but it is not suitable at predicting "No Default" and catching "Default".

In [ ]:
plot_evaluation_keras(leaky_model_pca, X_val_pca_arr, y_val_arr, X_test_pca_arr, y_test_arr,
                      title_prefix="PCA — Leaky ReLU MLP")

### PCA, Best Configuration (Original & SMOTE)

We retrain the best configuration identified in the hyperparameter sweep (Section 6)
on the PCA data, both on the original class-weighted training set and the SMOTE-resampled
set.

This mirrors Section 8 (SMOTE comparison on original features), giving us four
comparable training scenarios.

| Training data | Features | Section |
|---|---|---|
| Original + class_weight | Original | 8 |
| SMOTE | Original | 8 |
| Original + class_weight | PCA | 16 |
| SMOTE | PCA | 16 |

In [ ]:
def train_best_mlp_pca(X_train_data, y_train_data, cw=None, label=""):
    """Train the best sweep config on PCA data. cw = class_weight dict or None (for SMOTE)."""
    model = build_mlp_leaky_relu(
        N_FEATURES_PCA,
        hidden_units=BEST_UNITS,
        dropout_rate=BEST_DROPOUT,
        use_batchnorm=True,
        learning_rate=BEST_LR,
        alpha=0.1
    )
    es = EarlyStopping(monitor='val_auc', patience=10, mode='max',
                       restore_best_weights=True, verbose=0)
    print(f"Training best MLP (PCA) on {label} set (n={len(X_train_data):,})...")
    t0 = time.time()
    hist = model.fit(
        X_train_data, y_train_data,
        validation_data=(X_val_pca_arr, y_val_arr),
        epochs=100,
        batch_size=1024,
        callbacks=[es],
        class_weight=cw,
        verbose=0
    )
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s")
    return model, hist, elapsed


best_pca_orig,  hist_pca_orig,  time_pca_orig  = train_best_mlp_pca(
    X_train_pca_arr, y_train_arr,
    cw=CLASS_WEIGHT, label="Original (class_weight)"
)
best_pca_smote, hist_pca_smote, time_pca_smote = train_best_mlp_pca(
    X_train_smote_pca_arr, y_train_smote_arr,
    cw=None, label="SMOTE (balanced)"
)

on PCA features SMOTE works better than on the raw 113 features. with fewer dimensions the synthetic samples land in more sensible parts of the space.

In [ ]:
# SMOTE comparison table on PCA data
rows_pca = []
for model, label in [(best_pca_orig, "PCA — Original"), (best_pca_smote, "PCA — SMOTE")]:
    for X, y, split in [
        (X_val_pca_arr,  y_val_arr,  "Val"),
        (X_test_pca_arr, y_test_arr, "Test")
    ]:
        y_prob = model.predict(X, verbose=0).ravel()
        y_pred = (y_prob >= 0.3).astype(int)
        from sklearn.metrics import precision_score, recall_score, f1_score
        rows_pca.append({
            "Model"         : label,
            "Split"         : split,
            "ROC-AUC"       : round(roc_auc_score(y, y_prob), 4),
            "PR-AUC"        : round(average_precision_score(y, y_prob), 4),
            "Precision (1)" : round(precision_score(y, y_pred, pos_label=1, zero_division=0), 4),
            "Recall (1)"    : round(recall_score(y, y_pred, pos_label=1), 4),
            "F1 (1)"        : round(f1_score(y, y_pred, pos_label=1), 4),
        })

smote_pca_df = pd.DataFrame(rows_pca)
print("=== PCA — SMOTE vs Original comparison ===")
display(smote_pca_df)

On our PCA data, SMOTE worked better than in our past attempt. Still we are looking at ~20% f1-score, which is bad.

### PCA vs Original Features, Comprehensive Comparison

This section directly answers the key question, **does dimensionality reduction via PCA
improve, hurt, or leave unchanged the MLP's predictive performance?**

We compare equivalent model configurations trained on original vs PCA features
on the validation set. The test set is only shown for the final best model.

Potential outcomes and their interpretation.
- **PCA ≈ Original.** The removed variance components were noise; PCA is lossless
  for this task. Prefer PCA for faster training and lower memory.
- **PCA < Original.** The discarded components (bottom 5% variance) contained
  discriminative signal, the MLP was using fine-grained feature interactions that
  PCA collapsed. Stick with original features.
- **PCA > Original.** PCA removed noise that was causing overfitting, effectively
  acting as an additional regulariser. This would support using PCA in production.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def quick_metrics_pca(model, X_val, name, threshold=0.3):
    """Compute val-set metrics for the PCA comparison table."""
    y_prob = model.predict(X_val, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model"        : name,
        "Features"     : "PCA" if "PCA" in name else "Original",
        "Val ROC-AUC"  : round(roc_auc_score(y_val_arr, y_prob), 4),
        "Val PR-AUC"   : round(average_precision_score(y_val_arr, y_prob), 4),
        "Precision (1)": round(precision_score(y_val_arr, y_pred, pos_label=1, zero_division=0), 4),
        "Recall (1)"   : round(recall_score(y_val_arr, y_pred, pos_label=1), 4),
        "F1 (1)"       : round(f1_score(y_val_arr, y_pred, pos_label=1), 4),
    }

comparison_rows = [
    # Original feature models
    quick_metrics_pca(base_model,       X_val_arr,     "Base MLP ReLU — Original"),
    quick_metrics_pca(leaky_model,      X_val_arr,     "Leaky ReLU — Original"),
    quick_metrics_pca(best_model_orig,  X_val_arr,     "Best config (orig data) — Original"),
    quick_metrics_pca(best_model_smote, X_val_arr,     "Best config (SMOTE) — Original"),
    # PCA models
    quick_metrics_pca(base_model_pca,   X_val_pca_arr, "Base MLP ReLU — PCA"),
    quick_metrics_pca(leaky_model_pca,  X_val_pca_arr, "Leaky ReLU — PCA"),
    quick_metrics_pca(best_pca_orig,    X_val_pca_arr, "Best config (orig data) — PCA"),
    quick_metrics_pca(best_pca_smote,   X_val_pca_arr, "Best config (SMOTE) — PCA"),
]

comparison_df = pd.DataFrame(comparison_rows).sort_values("Val ROC-AUC", ascending=False).reset_index(drop=True)
print("=== PCA vs Original — Full Comparison (Val set, sorted by AUC) ===")
display(comparison_df)

In [ ]:
# grouped bar chart: AUC by model pair (Original vs PCA)
labels_pairs = [
    ("Base MLP ReLU",          base_model,       X_val_arr,     base_model_pca,   X_val_pca_arr),
    ("Leaky ReLU",             leaky_model,      X_val_arr,     leaky_model_pca,  X_val_pca_arr),
    ("Best config — orig",     best_model_orig,  X_val_arr,     best_pca_orig,    X_val_pca_arr),
    ("Best config — SMOTE",    best_model_smote, X_val_arr,     best_pca_smote,   X_val_pca_arr),
]

orig_aucs = [roc_auc_score(y_val_arr, m.predict(X, verbose=0).ravel())
             for _, m, X, _, _ in labels_pairs]
pca_aucs  = [roc_auc_score(y_val_arr, m.predict(X, verbose=0).ravel())
             for _, _, _, m, X in labels_pairs]
pair_names = [l for l, *_ in labels_pairs]

x = np.arange(len(pair_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, orig_aucs, width, label='Original features', color='steelblue')
bars2 = ax.bar(x + width/2, pca_aucs,  width, label='PCA features',      color='tomato')

for bar, val in zip(list(bars1) + list(bars2), orig_aucs + pca_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{val:.4f}", ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(pair_names, rotation=15, ha='right')
ax.set_ylabel("Val ROC-AUC")
ax.set_title("PCA vs Original Features — Val AUC by Model", fontweight='bold')
ax.legend()
ax.set_ylim(min(orig_aucs + pca_aucs) - 0.02, max(orig_aucs + pca_aucs) + 0.02)
plt.tight_layout()
save_plot(); plt.show()

In [ ]:
# ROC curve overlay, all 8 models on the validation set
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_title("ROC Curves — Original vs PCA Features (Val set)", fontsize=13, fontweight='bold')

styles = {
    "Original": {"linestyle": "-",  "alpha": 0.9},
    "PCA"     : {"linestyle": "--", "alpha": 0.9},
}
colors = ['steelblue', 'tomato', 'forestgreen', 'darkorange']

model_pairs = [
    ("Base MLP ReLU",       base_model,       X_val_arr,     base_model_pca,   X_val_pca_arr,   colors[0]),
    ("Leaky ReLU",          leaky_model,      X_val_arr,     leaky_model_pca,  X_val_pca_arr,   colors[1]),
    ("Best config — orig",  best_model_orig,  X_val_arr,     best_pca_orig,    X_val_pca_arr,   colors[2]),
    ("Best config — SMOTE", best_model_smote, X_val_arr,     best_pca_smote,   X_val_pca_arr,   colors[3]),
]

for name, m_orig, X_orig, m_pca, X_pca, color in model_pairs:
    for m, X, feat_label in [(m_orig, X_orig, "Original"), (m_pca, X_pca, "PCA")]:
        y_prob = m.predict(X, verbose=0).ravel()
        fpr, tpr, _ = roc_curve(y_val_arr, y_prob)
        auc = roc_auc_score(y_val_arr, y_prob)
        ax.plot(fpr, tpr, color=color,
                linestyle=styles[feat_label]["linestyle"],
                alpha=styles[feat_label]["alpha"],
                label=f"{name} ({feat_label}, AUC={auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random baseline')
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=7, loc='lower right')
plt.tight_layout()
save_plot(); plt.show()

In [ ]:
# we save PCA comparison results
comparison_df.to_csv(f"{OUTPUT_DIR}/mlp_pca_vs_original_comparison.csv", index=False)
smote_pca_df.to_csv(f"{OUTPUT_DIR}/mlp_pca_smote_comparison.csv",        index=False)

# Save best PCA model
best_pca_auc_orig  = roc_auc_score(y_val_arr, best_pca_orig.predict(X_val_pca_arr,  verbose=0).ravel())
best_pca_auc_smote = roc_auc_score(y_val_arr, best_pca_smote.predict(X_val_pca_arr, verbose=0).ravel())
best_pca_model = best_pca_orig if best_pca_auc_orig >= best_pca_auc_smote else best_pca_smote
best_pca_model.save(f"{OUTPUT_DIR}/mlp_pca_best.keras")

print("Saved PCA comparison CSVs and best PCA model.")
print(f"\nBest PCA model val AUC  : {max(best_pca_auc_orig, best_pca_auc_smote):.4f}")
print(f"Best Orig model val AUC : {summary_df.iloc[0]['Val ROC-AUC']:.4f}")
delta = max(best_pca_auc_orig, best_pca_auc_smote) - summary_df.iloc[0]['Val ROC-AUC']
direction = "higher" if delta > 0 else "lower"
print(f"\nPCA vs Original delta   : {delta:+.4f} ({direction} with PCA)")

Our best model had a val AUC of ~76%. On our imbalanced data, this is not a good result. This model used 0.3 Dropout, BatchNorm and Early Stopping. Interestingly, we achieved a 0,0017 gap between val loss and train loss, which was something that other models were not even close.

In [ ]:
# we save MLP test metrics for the cross-model comparison

mlp_test_metrics = metrics_final.copy()
print(mlp_test_metrics)


## **9. Regularization Ablation (RQ1)**

centrepiece for RQ1. we ask whether regularisation actually helps, and we answer it by training five MLP variants on the same architecture, varying only which regularisers are active. the variants are no regularisation, dropout only at rate 0.3, batch norm only, L2 only at 1e-4, and all three combined.

cross-model regularisation evidence from earlier sections, the elastic net sparsity-vs-AUC trade-off in 5.x, the RF cross-validation variance across configs in 6.x, and XGBoost's train-vs-val gap narrowing with regularisation in 7.x.

### Regularisation Ablation Study, RQ1 Centrepiece

This section directly addresses **RQ1**, *How do different regularisation strategies affect
predictive performance and model complexity?*

We train five variants of the same architecture (best config from Section 6), toggling
regularisation techniques on and off one at a time.

| Variant | Dropout | BatchNorm | Early stopping | Notes |
|---|---|---|---|---|
| None | no | no | no (100 epochs fixed) | Unregularised, expected to overfit |
| Dropout only | yes | no | no (100 epochs fixed) | L2-equivalent effect via stochastic masking |
| BatchNorm only | no | yes | no (100 epochs fixed) | Mild regulariser + stable training |
| Early stopping only | no | no | yes | Complexity control via training duration |
| All three | yes | yes | yes | Full regularisation, expected best val AUC |

In [ ]:
# we define the ablation configs

# we use the best architecture from the sweep
BEST_UNITS   = eval(best_config["Hidden units"])
BEST_DROPOUT = best_config["Dropout"]
BEST_LR      = best_config["LR"]

print(f"Ablation architecture: units={BEST_UNITS}, dropout={BEST_DROPOUT}, lr={BEST_LR}")

ablation_configs = [
    # label,                       dropout,       use_bn, use_early_stop
    ("No regularisation",          0.0,           False,  False),
    ("Dropout only",               BEST_DROPOUT,  False,  False),
    ("BatchNorm only",             0.0,           True,   False),
    ("Early stopping only",        0.0,           False,  True),
    ("All three (Dropout+BN+ES)",  BEST_DROPOUT,  True,   True),
]

ablation_results = []
ablation_histories = {}


In [ ]:
# we run the ablation loop

for label, dropout, use_bn, use_es in ablation_configs:
    print(f"\n[ABLATION] {label}")

    model = build_mlp_leaky_relu(
        N_FEATURES, hidden_units=BEST_UNITS,
        dropout_rate=dropout, use_batchnorm=use_bn,
        learning_rate=BEST_LR, alpha=0.1
    )

    callbacks = []
    if use_es:
        callbacks.append(EarlyStopping(
            monitor='val_auc', patience=10, mode='max',
            restore_best_weights=True, verbose=0
        ))

    t0 = time.time()
    hist = model.fit(
        X_train_arr, y_train_arr,
        validation_data=(X_val_arr, y_val_arr),
        epochs=100,
        batch_size=1024,
        callbacks=callbacks,
        verbose=0
    )
    elapsed = time.time() - t0

    auc_key     = [k for k in hist.history if 'auc' in k and 'val' not in k][0]
    train_auc   = max(hist.history[auc_key])
    val_auc     = max(hist.history[f'val_{auc_key}'])
    train_loss  = min(hist.history['loss'])
    val_loss    = min(hist.history['val_loss'])
    epochs_run  = len(hist.history['loss'])
    overfit_gap = round(train_auc - val_auc, 4)

    ablation_results.append({
        "Variant"           : label,
        "Dropout"           : dropout,
        "BatchNorm"         : use_bn,
        "Early stopping"    : use_es,
        "Best train AUC"    : round(train_auc, 4),
        "Best val AUC"      : round(val_auc, 4),
        "Train-Val gap"     : overfit_gap,
        "Best val loss"     : round(val_loss, 4),
        "Epochs run"        : epochs_run,
        "Train time (s)"    : round(elapsed, 1),
    })
    ablation_histories[label] = hist

    print(f"  Train AUC: {train_auc:.4f}  |  Val AUC: {val_auc:.4f}  "
          f"|  Gap: {overfit_gap:.4f}  |  Epochs: {epochs_run}")

ablation_df = pd.DataFrame(ablation_results)
print("\n=== Regularisation Ablation Study Results ===")
display(ablation_df)


In [ ]:
# training curves for all 5 ablation variants
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Regularisation Ablation — Validation Loss & AUC", fontsize=13, fontweight='bold')

palette = ['black', 'tomato', 'steelblue', 'orange', 'forestgreen']

for (label, _, _, _), color in zip(ablation_configs, palette):
    hist = ablation_histories[label]
    auc_key = [k for k in hist.history if 'auc' in k and 'val' not in k][0]

    axes[0].plot(hist.history['val_loss'],           label=label, color=color, alpha=0.85)
    axes[1].plot(hist.history[f'val_{auc_key}'],     label=label, color=color, alpha=0.85)

for ax, title, ylabel in [
    (axes[0], "Validation Loss", "Binary cross-entropy"),
    (axes[1], "Validation AUC",  "ROC-AUC"),
]:
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

plt.tight_layout()
save_plot(); plt.show()

From these graphs we can conclude that only the models "Dropout only" and "All three" are the ones where the loss is reducing after around 10 epochs. More importantly, the model with "All three" generated the best Val AUC, Val Loss and Train-Val gap, which means it is the one overfitting the least while still having good predictive power.

In [ ]:
# bar chart: best val AUC per variant
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Regularisation Ablation — Summary", fontsize=13, fontweight='bold')

palette_ablation = ['black', 'tomato', 'steelblue', 'orange', 'forestgreen']

#Val AUC
bars = axes[0].bar(ablation_df["Variant"], ablation_df["Best val AUC"],
                   color=palette_ablation, edgecolor='white')
axes[0].set_ylabel("Best Val AUC")
axes[0].set_title("Best Validation AUC per Variant")
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylim(ablation_df["Best val AUC"].min() - 0.01, ablation_df["Best val AUC"].max() + 0.01)
for bar, val in zip(bars, ablation_df["Best val AUC"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                 f"{val:.4f}", ha='center', fontsize=8)

#Train-Val gap (overfitting proxy)
bars2 = axes[1].bar(ablation_df["Variant"], ablation_df["Train-Val gap"],
                    color=palette_ablation, edgecolor='white')
axes[1].set_ylabel("Train AUC − Val AUC")
axes[1].set_title("Overfitting Gap (lower = better generalisation)")
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(0, color='k', linewidth=0.8, linestyle='--')
for bar, val in zip(bars2, ablation_df["Train-Val gap"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                 f"{val:.4f}", ha='center', fontsize=8)

plt.tight_layout()
save_plot(); plt.show()

## **10. Feature Importance Comparison (RQ2)**

does the regularised LR (elastic net) end up agreeing with the trees on which features matter. this section consolidates the cross-model comparison cells that were originally scattered across the elastic net, random forest, and XGBoost notebooks.

per-model importance plots stay inside the model sections above. only the cross-model rankings are aggregated here.

### 10.1 Elastic Net against LR Baseline (top-10 overlap, side-by-side)

### Coefficient Comparison, Elastic Net vs LR Baseline

Both models are linear, so their coefficients live on the same scale. The interesting
question is, **does regularisation change which features look important?**

We compare.
- **Spearman rank correlation** of `|coefficient|` across the full feature set
- **Top-10 overlap** between the two models
- **Side-by-side bar chart** of top-10 features by |coefficient|

In [ ]:
# --- Build aligned ranking tables ---
lr_rank = (
    lr_coefs.assign(abs_coef=lr_coefs["coefficient"].abs())
            .sort_values("abs_coef", ascending=False)
            .reset_index(drop=True)
            .reset_index().rename(columns={"index": "lr_rank"})
            [["feature", "lr_rank", "coefficient"]]
            .rename(columns={"coefficient": "lr_coef"})
)

enet_rank = (
    enet_coef_df.sort_values("abs_coef", ascending=False)
                .reset_index(drop=True)
                .reset_index().rename(columns={"index": "enet_rank"})
                [["feature", "enet_rank", "coefficient"]]
                .rename(columns={"coefficient": "enet_coef"})
)

ranks = lr_rank.merge(enet_rank, on="feature")
print(f"Features common to both rankings: {len(ranks)}\n")
print("Top 15 features by LR |coefficient| with corresponding Elastic Net rank/coef:")
ranks.sort_values("lr_rank").head(15).round(4)

In [ ]:
# --- Top-10 overlap ---
top10_lr   = set(lr_rank.head(10)["feature"])
top10_enet = set(enet_rank.head(10)["feature"])
overlap    = top10_lr & top10_enet

print(f"Top-10 overlap (LR baseline vs Elastic Net): {len(overlap)} / 10 features")
print(f"  Common   : {sorted(overlap)}")
print(f"  LR-only  : {sorted(top10_lr - top10_enet)}")
print(f"  ENet-only: {sorted(top10_enet - top10_lr)}")

# --- Spearman rank correlation across the full feature set ---
rho, pval = spearmanr(ranks["lr_rank"], ranks["enet_rank"])
print(f"\nSpearman rank correlation (LR vs Elastic Net): {rho:.3f} (p={pval:.2e})")

In [ ]:
# --- Side-by-side top-10 bar chart ---
top10_lr_df   = lr_rank.assign(abs_coef=lr_rank["lr_coef"].abs()).sort_values("abs_coef", ascending=False).head(10)
top10_enet_df = enet_rank.assign(abs_coef=enet_rank["enet_coef"].abs()).sort_values("abs_coef", ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Top 10 Features by |coefficient| — LR Baseline vs Elastic Net",
             fontsize=13, fontweight='bold')

# LR baseline
axes[0].barh(top10_lr_df["feature"], top10_lr_df["abs_coef"],
             color="seagreen", edgecolor="black", linewidth=0.6)
axes[0].set_title("LR Baseline (no regularisation)\n|coefficient|", fontsize=11)
axes[0].set_xlabel("|coefficient|")
axes[0].invert_yaxis()

# Elastic Net
axes[1].barh(top10_enet_df["feature"], top10_enet_df["abs_coef"],
             color="purple", edgecolor="black", linewidth=0.6)
axes[1].set_title(f"Elastic Net (l1_ratio={enet_best_params['l1_ratio']}, "
                  f"C={enet_best_params['C']})\n|coefficient|", fontsize=11)
axes[1].set_xlabel("|coefficient|")
axes[1].invert_yaxis()

plt.tight_layout()
save_plot(); plt.show()

### Takeaway, what does Elastic Net change vs the LR baseline?

The two linear models agree almost perfectly. **Spearman ρ = 0.917** (p ≈ 5.5×10⁻⁴⁶) and **8/10 top-10 overlap.** Eight features are common to both top-10 lists (`AMT_ANNUITY`, `AMT_CREDIT`, `AMT_GOODS_PRICE`, `CREDIT_TERM`, `EXT_SOURCE_2`, `EXT_SOURCE_2_MISSING`, `EXT_SOURCE_3`, `NAME_INCOME_TYPE_Pensioner`). Only two swap in/out at the boundary.

- **What ENet promotes.** `CODE_GENDER_M` (LR rank 12 to ENet rank 8) and `NAME_INCOME_TYPE_Unemployed` (LR rank 13 to ENet rank 5). The L2 component has redistributed coefficient magnitude, likely shrinking some correlated competitors and leaving these two with relatively more weight.
- **What ENet demotes.** `NAME_TYPE_SUITE_Missing` (LR rank 4 to ENet rank 10, magnitude shrunk from −0.439 to −0.297) and `OCCUPATION_TYPE_Drivers` (LR rank 8 to ENet rank 11).
- **`NAME_INCOME_TYPE_Pensioner` is the most striking re-ranking.** LR rank 7 (−0.376) but ENet rank 0 (−0.560). Elastic Net actually *amplified* this coefficient, most likely because L2 shrank its correlated competitors, leaving the pensioner signal to absorb more of the credit.

**For the report.** combined with the sparsity result above, the headline is clean, *Elastic Net achieves the same predictive performance as unregularised LR (Test ROC-AUC 0.7523) with 6.2% of coefficients pruned and a Spearman ρ = 0.92 agreement on feature ranking.* Regularisation is essentially "free" parsimony in the linear family. By contrast, the RF feature ranking had Spearman ρ = 0.13, 0.15 vs LR, so the **linear-vs-tree** gap remains the dominant axis of disagreement, not the **regularised-vs-unregularised** gap within the linear family.

### 10.2 Random Forest against LR Baseline (Spearman rank, top-20 overlap)

### Feature Importance Comparison, RF vs LR (RQ2)

For RQ2 ("which features are most predictive of default, and do the most important features
differ across model types?") we compare the RF rankings (Gini and permutation) against
the LR baseline coefficient magnitudes.

We use.
- **Top-20 overlap**, how many features appear in both top-20 lists.
- **Spearman rank correlation**, does the *ordering* agree across the full feature space?
- **Side-by-side bar chart**, visual comparison of the top features.

In [ ]:
# --- Build a unified ranking table ---
lr_rank = (
    lr_coefs.assign(abs_coef=lr_coefs["coefficient"].abs())
            .sort_values("abs_coef", ascending=False)
            .reset_index(drop=True)
            .reset_index().rename(columns={"index": "lr_rank"})[["feature", "lr_rank", "coefficient"]]
)

rf_gini_rank = gini_df.reset_index().rename(columns={"index": "gini_rank"})[
    ["feature", "gini_rank", "importance"]
].rename(columns={"importance": "gini_importance"})

rf_perm_rank = perm_df.reset_index().rename(columns={"index": "perm_rank"})[
    ["feature", "perm_rank", "importance"]
].rename(columns={"importance": "perm_importance"})

ranks = lr_rank.merge(rf_gini_rank, on="feature").merge(rf_perm_rank, on="feature")
print(f"Features common to all three rankings: {len(ranks)}\n")
print("Top 15 features by LR |coefficient| with corresponding RF ranks:")
ranks.sort_values("lr_rank").head(15).round(4)

In [ ]:
# --- Top-20 overlap counts ---
top20_lr        = set(lr_rank.head(20)["feature"])
top20_gini_set  = set(top20_gini["feature"])
top20_perm_set  = set(top20_perm["feature"])

overlap_lr_gini   = top20_lr & top20_gini_set
overlap_lr_perm   = top20_lr & top20_perm_set
overlap_gini_perm = top20_gini_set & top20_perm_set

print(f"Top-20 overlap (LR vs RF Gini)       : {len(overlap_lr_gini)} features")
print(f"Top-20 overlap (LR vs RF Permutation): {len(overlap_lr_perm)} features")
print(f"Top-20 overlap (RF Gini vs RF Perm)  : {len(overlap_gini_perm)} features")

# --- Spearman rank correlation across full feature set ---
rho_lr_gini, _   = spearmanr(ranks["lr_rank"], ranks["gini_rank"])
rho_lr_perm, _   = spearmanr(ranks["lr_rank"], ranks["perm_rank"])
rho_gini_perm, _ = spearmanr(ranks["gini_rank"], ranks["perm_rank"])

print(f"\nSpearman rank correlation (full feature set):")
print(f"  LR vs RF Gini       : {rho_lr_gini:.3f}")
print(f"  LR vs RF Permutation: {rho_lr_perm:.3f}")
print(f"  RF Gini vs Perm     : {rho_gini_perm:.3f}")

RF and LR disagree dramatically on which features matter, the spearman rank correlation sits below 0.2. trees split on EXT_SOURCE thresholds while the linear model leans on the engineered ratios.

### Takeaway, RQ2 headline finding

The rank-correlation numbers above are striking.

| Comparison | Spearman ρ | Top-20 overlap |
|---|---|---|
| LR vs RF Gini | **0.13** | 7 / 20 |
| LR vs RF Permutation | **0.15** | 7 / 20 |
| RF Gini vs RF Permutation | **0.69** | 14 / 20 |

The two **RF** importance measures broadly agree with each other (ρ = 0.69), but **neither agrees with LR** (ρ ≈ 0.13, 0.15). This is direct evidence that linear and non-linear models extract different signal from the same data.

- **LR's top features** are dominated by **categoricals and missingness indicators** (`NAME_TYPE_SUITE_Missing`, `EXT_SOURCE_2_MISSING`, `NAME_INCOME_TYPE_Pensioner`, `OCCUPATION_TYPE_Drivers`). These features have large coefficients because, conditional on everything else, their main effect on log-odds is large.
- **RF's top features** are dominated by **continuous variables and engineered ratios** (`EXT_SOURCE_3`, `EXT_SOURCE_2`, `CREDIT_TERM`, `CREDIT_GOODS_RATIO`, `DAYS_EMPLOYED`). Trees exploit these in non-linear splits and interactions that linear models can't represent.
- `EXT_SOURCE_2` is a good example, LR rank 6, RF Gini rank 2, RF Permutation rank 2. The signal is there for both, but the RF gets more out of it through interactions.

**Implication for the report.** the choice of model isn't just a question of accuracy, it changes *which features look important*, which has direct consequences for the explainability and fairness analysis in Phase 6. We can't pick a single "true" feature ranking.

In [ ]:
# --- Side-by-side top-20 bar chart for the report ---
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle("Top 20 Features — LR Baseline vs Random Forest (Gini & Permutation)",
             fontsize=13, fontweight='bold')

# Panel 1, LR top 20 by |coefficient|
top20_lr_df = lr_rank.head(20).copy()
top20_lr_df["abs_coef"] = top20_lr_df["coefficient"].abs()
axes[0].barh(top20_lr_df["feature"], top20_lr_df["abs_coef"],
             color="seagreen", edgecolor="black", linewidth=0.6)
axes[0].set_title("LR Baseline\n(|coefficient|)", fontsize=11)
axes[0].set_xlabel("|coefficient|")
axes[0].invert_yaxis()

# Panel 2, RF Gini top 20
axes[1].barh(top20_gini["feature"], top20_gini["importance"],
             color="steelblue", edgecolor="black", linewidth=0.6)
axes[1].set_title("Random Forest\n(Gini importance)", fontsize=11)
axes[1].set_xlabel("Mean decrease in impurity")
axes[1].invert_yaxis()

# Panel 3, RF Permutation top 20
axes[2].barh(top20_perm["feature"], top20_perm["importance"],
             color="darkorange", edgecolor="black", linewidth=0.6)
axes[2].set_title("Random Forest\n(Permutation importance)", fontsize=11)
axes[2].set_xlabel("Drop in ROC-AUC")
axes[2].invert_yaxis()

plt.tight_layout()
save_plot(); plt.show()

### 10.3 XGBoost Gain against Permutation Importance

### Feature Importance Comparison, Gain vs Permutation (RQ2)

Compare XGBoost's two importance measures (built-in gain and model-agnostic permutation)
to check whether they agree on which features drive default prediction.

In [ ]:
# --- Build ranking table ---
xgb_gain_rank = gain_df.reset_index().rename(columns={"index": "xgb_gain_rank"})[
    ["feature", "xgb_gain_rank", "importance"]
].rename(columns={"importance": "xgb_gain_importance"})

xgb_perm_rank = perm_df.reset_index().rename(columns={"index": "xgb_perm_rank"})[
    ["feature", "xgb_perm_rank", "importance"]
].rename(columns={"importance": "xgb_perm_importance"})

ranks = xgb_gain_rank.merge(xgb_perm_rank, on="feature")
print(f"Features in ranking table: {len(ranks)}\n")
print("Top 15 features by Gain with corresponding Permutation ranks:")
ranks.sort_values("xgb_gain_rank").head(15).round(4)

In [ ]:
# --- Top-20 overlap ---
top20_xgb_set  = set(top20_gain["feature"])
top20_xgb_perm = set(top20_perm["feature"])
print(f"Top-20 overlap (Gain vs Permutation): {len(top20_xgb_set & top20_xgb_perm)} features")

# --- Spearman rank correlation ---
rho, _ = spearmanr(ranks["xgb_gain_rank"], ranks["xgb_perm_rank"])
print(f"Spearman rank correlation (Gain vs Permutation): {rho:.3f}")

In [ ]:
# --- Side-by-side top-20 bar chart: XGB Gain vs XGB Perm ---
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Top 20 Features \u2014 XGBoost Gain vs Permutation Importance",
             fontsize=13, fontweight='bold')

axes[0].barh(top20_gain["feature"], top20_gain["importance"],
             color="mediumorchid", edgecolor="black", linewidth=0.6)
axes[0].set_title("XGBoost\n(Gain importance)", fontsize=11)
axes[0].set_xlabel("Mean gain per split")
axes[0].invert_yaxis()

axes[1].barh(top20_perm["feature"], top20_perm["importance"],
             color="darkorange", edgecolor="black", linewidth=0.6)
axes[1].set_title("XGBoost\n(Permutation importance)", fontsize=11)
axes[1].set_xlabel("Mean drop in ROC-AUC")
axes[1].invert_yaxis()

plt.tight_layout()
save_plot(); plt.show()

## **11. Fairness Analysis (RQ3)**

**outside the course material.** none of the original team notebooks contain fairness analysis code. **placeholder section.** future work would compute demographic parity (false positive rate gap), equal opportunity (true positive rate gap), and calibration gaps across age bands, gender, and family status, using the predictions from the best model in section 7 (XGBoost).

the EDA in section 2 already shows default-rate variation across these subgroups, so the descriptive groundwork exists. only the per-model fairness metrics are missing.

## **12. Final Results and Comparison**

cross-model performance summary moved here from the XGBoost notebook. it reads the per-model val and test ROC-AUC and PR-AUC numbers reported in each model section above and assembles them into a single table.

### Cross-Model Performance Comparison

Consolidate the best test ROC-AUC from every notebook for the report's model-comparison table.

In [ ]:
# we collect best test metrics from each model into one dataframe

all_models = pd.DataFrame([
    {"Model": "LR Baseline",   "Test ROC-AUC": lr_test_metrics["roc_auc"],
                               "Test PR-AUC":  lr_test_metrics["pr_auc"]},
    {"Model": "Elastic Net",   "Test ROC-AUC": enet_test_metrics["roc_auc"],
                               "Test PR-AUC":  enet_test_metrics["pr_auc"]},
    {"Model": "Random Forest", "Test ROC-AUC": rf_test_metrics["roc_auc"],
                               "Test PR-AUC":  rf_test_metrics["pr_auc"]},
    {"Model": "XGBoost",       "Test ROC-AUC": xgb_test_metrics["roc_auc"],
                               "Test PR-AUC":  xgb_test_metrics["pr_auc"]},
    {"Model": "MLP",           "Test ROC-AUC": mlp_test_metrics["roc_auc"],
                               "Test PR-AUC":  mlp_test_metrics["pr_auc"]},
])

all_models = all_models.sort_values("Test ROC-AUC", ascending=False).reset_index(drop=True)
print(all_models.to_string(index=False))


In [ ]:
# bar chart of test ROC-AUC by model

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(all_models["Model"], all_models["Test ROC-AUC"],
       color='steelblue', edgecolor='none')
ax.set_ylim(0.7, all_models["Test ROC-AUC"].max() * 1.02)
ax.set_ylabel("Test ROC-AUC")
ax.set_title("Cross-model comparison")
for i, v in enumerate(all_models["Test ROC-AUC"]):
    ax.text(i, v + 0.001, f"{v:.4f}", ha='center', fontsize=9)

plt.tight_layout()
save_plot('cross_model_comparison')
plt.show()
